# 700 — Program Signature Construction

## Objective

Construct and validate the frozen Phase 7 program-query representations and the outcome-blind preparatory objects required by the prespecified conditional-null and benchmark designs before notebook 701 connectivity inference begins.

Notebook 700 is a preparatory and computational-calibration notebook. It does not generate perturbagen rankings, observed program–perturbagen connectivity results, conditional-randomization p-values, multiplicity results, compound prioritization, or mechanism-of-action summaries.

## Analytical boundary

This notebook consumes the registered frozen Phase 4, Phase 3, and notebook-110 handoffs as authoritative inputs.

It may use the frozen LINCS Level 5 matrix only for prospectively authorized outcome-blind quantities, including gene-level perturbational dispersion and the namespace-3 computational benchmark. It must not apply the real frozen program queries to the observed perturbational matrix in a way that computes or exposes real program–perturbagen connectivity.

The notebook does not:

* alter frozen Phase 4 program membership, weights, sign, or identity;
* use Phase 5 or Phase 6 evidence for query construction, eligibility, prioritization, or rescue;
* inspect compound connectivity or rank perturbagens;
* execute the primary inference reserved for notebook 701; or
* select the final Monte Carlo `Bmax` using observed scientific results.

The resistance-like orientation used by Phase 7 is derived only from frozen upstream Phase 3/4 evidence under the governing Phase 7 contract. It is stored separately from the unchanged Phase 4 gene weights.


In [ ]:
# =============================================================================
# Imports
# =============================================================================

import h5py
import numpy as np
import pandas as pd

import shutil
import tempfile
from pathlib import Path
import time 
import threading
import ctypes

import json
import os
import platform
import subprocess
import sys
from importlib.metadata import version as package_version

from threadpoolctl import threadpool_info

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Resolve frozen Phase 7 preparation inputs
# =============================================================================

artifact_registry = load_artifact_registry()

INPUT_ARTIFACT_IDS = (
    "phase4.401.consensus_transcriptomic_program_catalog",
    "phase4.401.consensus_transcriptomic_gene_weights",
    "phase3.311.program_robustness_summary",
    "phase1.110.raw_input_inventory",
    "phase1.110.program_representation_fidelity",
    "phase1.110.bing_gene_handoff",
    "phase1.110.landmark_gene_handoff",
    "phase1.110.primary_signature_handoff",
    "phase1.110.primary_perturbagen_coverage",
)

input_paths = {
    artifact_id: resolve_artifact_path(
        artifact_registry,
        artifact_id,
    )
    for artifact_id in INPUT_ARTIFACT_IDS
}

for artifact_id, path in input_paths.items():
    print(
        f"{artifact_id}: "
        f"{project_relative_path(path)}"
    )


phase4.401.consensus_transcriptomic_program_catalog: data/processed/consensus_programs/401_consensus_transcriptomic_program_catalog.csv
phase4.401.consensus_transcriptomic_gene_weights: data/processed/consensus_programs/401_consensus_transcriptomic_gene_weights.csv
phase3.311.program_robustness_summary: data/processed/cellline_programs/311_program_robustness_summary.csv
phase1.110.raw_input_inventory: data/interim/perturbational/110_lincs_raw_input_inventory.csv
phase1.110.program_representation_fidelity: data/interim/perturbational/110_lincs_program_representation_fidelity.csv
phase1.110.bing_gene_handoff: data/interim/perturbational/110_lincs_bing_consensus_gene_handoff.csv
phase1.110.landmark_gene_handoff: data/interim/perturbational/110_lincs_landmark_consensus_gene_handoff.csv
phase1.110.primary_signature_handoff: data/interim/perturbational/110_lincs_primary_signature_handoff.parquet
phase1.110.primary_perturbagen_coverage: data/interim/perturbational/110_lincs_primary_perturbage

In [3]:
# =============================================================================
# Frozen Phase 7 program order
# =============================================================================

PROGRAM_IDS = (
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
)

In [4]:
# =============================================================================
# Load frozen Phase 7 preparation inputs
# =============================================================================

program_catalog = pd.read_csv(
    input_paths[
        "phase4.401.consensus_transcriptomic_program_catalog"
    ]
)

phase4_gene_weights = pd.read_csv(
    input_paths[
        "phase4.401.consensus_transcriptomic_gene_weights"
    ]
)

phase3_robustness = pd.read_csv(
    input_paths[
        "phase3.311.program_robustness_summary"
    ]
)

raw_input_inventory = pd.read_csv(
    input_paths[
        "phase1.110.raw_input_inventory"
    ]
)

representation_fidelity = pd.read_csv(
    input_paths[
        "phase1.110.program_representation_fidelity"
    ]
)

bing_gene_handoff = pd.read_csv(
    input_paths[
        "phase1.110.bing_gene_handoff"
    ]
)

landmark_gene_handoff = pd.read_csv(
    input_paths[
        "phase1.110.landmark_gene_handoff"
    ]
)

primary_signature_handoff = pd.read_parquet(
    input_paths[
        "phase1.110.primary_signature_handoff"
    ]
)

primary_perturbagen_coverage = pd.read_csv(
    input_paths[
        "phase1.110.primary_perturbagen_coverage"
    ]
)

print("Program catalog shape:", program_catalog.shape)
print("Phase 4 gene weights shape:", phase4_gene_weights.shape)
print("Phase 3 robustness shape:", phase3_robustness.shape)
print("Raw input inventory shape:", raw_input_inventory.shape)
print("Representation fidelity shape:", representation_fidelity.shape)
print("BING handoff shape:", bing_gene_handoff.shape)
print("Landmark handoff shape:", landmark_gene_handoff.shape)
print(
    "Primary signature handoff shape:",
    primary_signature_handoff.shape,
)
print(
    "Primary perturbagen coverage shape:",
    primary_perturbagen_coverage.shape,
)

Program catalog shape: (3, 21)
Phase 4 gene weights shape: (7167, 8)
Phase 3 robustness shape: (10, 33)
Raw input inventory shape: (8, 4)
Representation fidelity shape: (3, 19)
BING handoff shape: (4485, 14)
Landmark handoff shape: (270, 14)
Primary signature handoff shape: (362036, 20)
Primary perturbagen coverage shape: (7914, 4)


In [5]:
# =============================================================================
# Derive resistance-like orientation from frozen Phase 3/4 evidence
# =============================================================================

program_source_metadata = (
    program_catalog[
        [
            "consensus_program_id",
            "cell_line_program",
            "orientation_multiplier",
        ]
    ]
    .rename(
        columns={
            "consensus_program_id": "program_id",
            "cell_line_program": "source_cell_line_program",
            "orientation_multiplier": (
                "phase4_orientation_multiplier"
            ),
        }
    )
    .merge(
        phase3_robustness[
            [
                "program_id",
                "lineage_adjusted_rho",
                "robustness_category",
            ]
        ].rename(
            columns={
                "program_id": "source_cell_line_program",
                "lineage_adjusted_rho": (
                    "phase3_lineage_adjusted_rho"
                ),
                "robustness_category": (
                    "phase3_robustness_category"
                ),
            }
        ),
        on="source_cell_line_program",
        how="left",
        validate="one_to_one",
    )
    .sort_values("program_id")
    .reset_index(drop=True)
)

if program_source_metadata[
    [
        "phase4_orientation_multiplier",
        "phase3_lineage_adjusted_rho",
    ]
].isna().any().any():
    raise RuntimeError(
        "Incomplete frozen Phase 3/4 orientation metadata."
    )

program_source_metadata[
    "resistance_like_orientation_multiplier"
] = np.sign(
    program_source_metadata[
        "phase4_orientation_multiplier"
    ]
    * program_source_metadata[
        "phase3_lineage_adjusted_rho"
    ]
).astype("int8")


In [6]:
# =============================================================================
# Validate the prospectively frozen resistance-like orientation
# =============================================================================

# This exact map is a contract cross-check, not an alternate orientation
# method. The multiplier itself is derived only from frozen upstream evidence.
EXPECTED_RESISTANCE_LIKE_MULTIPLIERS = {
    "CONSENSUS_TX_01": -1,
    "CONSENSUS_TX_02": 1,
    "CONSENSUS_TX_03": 1,
}

observed_multipliers = (
    program_source_metadata
    .set_index("program_id")[
        "resistance_like_orientation_multiplier"
    ]
    .astype(int)
    .to_dict()
)

if observed_multipliers != EXPECTED_RESISTANCE_LIKE_MULTIPLIERS:
    raise RuntimeError(
        "Derived resistance-like orientation does not "
        "match the frozen Phase 7 contract."
    )

display(program_source_metadata)


,program_id,source_cell_line_program,phase4_orientation_multiplier,phase3_lineage_adjusted_rho,phase3_robustness_category,resistance_like_orientation_multiplier
0,CONSENSUS_TX_01,ICA_PROGRAM_09,1,-0.038606,CONTEXT_SENSITIVE_CANDIDATE,-1
1,CONSENSUS_TX_02,ICA_PROGRAM_29,-1,-0.097379,ROBUSTNESS_SUPPORTED_CANDIDATE,1
2,CONSENSUS_TX_03,ICA_PROGRAM_13,-1,-0.136528,ROBUSTNESS_SUPPORTED_CANDIDATE,1


In [7]:
# =============================================================================
# Materialize deterministic continuous BING query definitions
# =============================================================================

program_query_definitions = (
    bing_gene_handoff[
        [
            "consensus_program_id",
            "gene_id",
            "gene_symbol",
            "feature_space",
            "gctx_gene_index",
            "consensus_weight",
        ]
    ]
    .rename(
        columns={
            "consensus_program_id": "program_id",
            "consensus_weight": "frozen_weight",
        }
    )
    .merge(
        program_source_metadata,
        on="program_id",
        how="left",
        validate="many_to_one",
    )
    .assign(
        gene_id=lambda data: pd.to_numeric(
            data["gene_id"],
            errors="raise",
        ).astype("int64"),
        gctx_gene_index=lambda data: pd.to_numeric(
            data["gctx_gene_index"],
            errors="raise",
        ).astype("int64"),
        primary_bing_member=True,
        landmark_sensitivity_member=lambda data: (
            data["feature_space"].eq("landmark")
        ),
    )
    .sort_values(
        [
            "program_id",
            "gene_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

program_query_definitions["query_gene_order"] = (
    program_query_definitions
    .groupby(
        "program_id",
        sort=False,
    )
    .cumcount()
    .astype("int64")
)


In [8]:
# =============================================================================
# Validate continuous BING query definitions
# =============================================================================

frozen_weight_values = (
    program_query_definitions["frozen_weight"]
    .to_numpy(dtype=np.float64)
)

if not np.isfinite(frozen_weight_values).all():
    raise RuntimeError(
        "Program query contains a non-finite frozen weight."
    )

query_norms = (
    program_query_definitions
    .groupby(
        "program_id",
        sort=False,
    )["frozen_weight"]
    .apply(
        lambda values: np.linalg.norm(
            values.to_numpy(dtype=np.float64)
        )
    )
    .reindex(PROGRAM_IDS)
)

if not query_norms.gt(0).all():
    raise RuntimeError(
        f"Program query has zero norm: "
        f"{query_norms.to_dict()}"
    )

if len(program_query_definitions) != 4485:
    raise RuntimeError(
        "Unexpected total number of BING "
        "query-definition rows."
    )

if program_query_definitions[
    [
        "program_id",
        "gene_id",
    ]
].duplicated().any():
    raise RuntimeError(
        "Duplicate program × gene rows in query definitions."
    )

query_summary = (
    program_query_definitions
    .groupby(
        "program_id",
        as_index=False,
    )
    .agg(
        bing_genes=("gene_id", "nunique"),
        landmark_genes=(
            "landmark_sensitivity_member",
            "sum",
        ),
        positive_weights=(
            "frozen_weight",
            lambda values: int((values > 0).sum()),
        ),
        negative_weights=(
            "frozen_weight",
            lambda values: int((values < 0).sum()),
        ),
    )
)

query_summary["l2_norm"] = (
    query_summary["program_id"].map(query_norms)
)

display(query_summary)


,program_id,bing_genes,landmark_genes,positive_weights,negative_weights,l2_norm
0,CONSENSUS_TX_01,1495,90,510,985,0.850741
1,CONSENSUS_TX_02,1495,90,864,631,0.801657
2,CONSENSUS_TX_03,1495,90,712,783,0.813026


In [9]:
# =============================================================================
# Select fixed CMap-style 150 / 150 query membership
# =============================================================================

program_query_definitions["cmap_style_arm"] = pd.Series(
    pd.NA,
    index=program_query_definitions.index,
    dtype="string",
)

for program_id in PROGRAM_IDS:
    program_mask = (
        program_query_definitions[
            "program_id"
        ].eq(program_id)
    )

    positive_candidates = (
        program_query_definitions.loc[
            program_mask
            & program_query_definitions[
                "frozen_weight"
            ].gt(0)
        ]
        .sort_values(
            [
                "frozen_weight",
                "gene_id",
            ],
            ascending=[False, True],
            kind="mergesort",
        )
    )

    negative_candidates = (
        program_query_definitions.loc[
            program_mask
            & program_query_definitions[
                "frozen_weight"
            ].lt(0)
        ]
        .sort_values(
            [
                "frozen_weight",
                "gene_id",
            ],
            ascending=[True, True],
            kind="mergesort",
        )
    )

    if len(positive_candidates) < 150:
        raise RuntimeError(
            f"{program_id} has fewer than "
            "150 positive BING weights."
        )

    if len(negative_candidates) < 150:
        raise RuntimeError(
            f"{program_id} has fewer than "
            "150 negative BING weights."
        )

    program_query_definitions.loc[
        positive_candidates.head(150).index,
        "cmap_style_arm",
    ] = "up"

    program_query_definitions.loc[
        negative_candidates.head(150).index,
        "cmap_style_arm",
    ] = "down"

program_query_definitions["cmap_style_member"] = (
    program_query_definitions[
        "cmap_style_arm"
    ].notna()
)


In [10]:
# =============================================================================
# Validate fixed CMap-style query membership
# =============================================================================

cmap_membership_summary = (
    program_query_definitions.loc[
        program_query_definitions[
            "cmap_style_member"
        ]
    ]
    .groupby(
        [
            "program_id",
            "cmap_style_arm",
        ],
        as_index=False,
    )
    .agg(
        genes=("gene_id", "nunique"),
    )
    .sort_values(
        [
            "program_id",
            "cmap_style_arm",
        ]
    )
    .reset_index(drop=True)
)

cmap_count_matrix = (
    cmap_membership_summary
    .pivot(
        index="program_id",
        columns="cmap_style_arm",
        values="genes",
    )
    .reindex(PROGRAM_IDS)
)

if not (
    cmap_count_matrix["up"].eq(150).all()
    and cmap_count_matrix["down"].eq(150).all()
):
    raise RuntimeError(
        "Fixed CMap-style query membership is not "
        "150 / 150 for every frozen program."
    )

display(cmap_membership_summary)


,program_id,cmap_style_arm,genes
0,CONSENSUS_TX_01,down,150
1,CONSENSUS_TX_01,up,150
2,CONSENSUS_TX_02,down,150
3,CONSENSUS_TX_02,up,150
4,CONSENSUS_TX_03,down,150
5,CONSENSUS_TX_03,up,150


In [11]:
# =============================================================================
# Resolve raw LINCS resources required for MAD stratification
# =============================================================================

raw_resource_paths = (
    raw_input_inventory
    .set_index("resource")["relative_path"]
)

LINCS_GENEINFO_PATH = (
    Paths.root
    / raw_resource_paths.loc["geneinfo"]
)

LINCS_LEVEL5_TRT_CP_PATH = (
    Paths.root
    / raw_resource_paths.loc["level5_trt_cp"]
)

for resource_name, path in {
    "geneinfo": LINCS_GENEINFO_PATH,
    "level5_trt_cp": LINCS_LEVEL5_TRT_CP_PATH,
}.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Required LINCS resource not found: {path}"
        )

    print(
        f"{resource_name}: "
        f"{project_relative_path(path)}"
    )

geneinfo: data/raw/lincs/geneinfo_beta.txt
level5_trt_cp: data/raw/lincs/level5_beta_trt_cp_n720216x12328.gctx


In [12]:
# =============================================================================
# Construct the complete BING gene universe for MAD stratification
# =============================================================================

lincs_geneinfo = pd.read_csv(
    LINCS_GENEINFO_PATH,
    sep="\t",
    low_memory=False,
)

bing_gene_universe = (
    lincs_geneinfo.loc[
        lincs_geneinfo["feature_space"].isin(
            [
                "landmark",
                "best inferred",
            ]
        ),
        [
            "gene_id",
            "gene_symbol",
            "feature_space",
        ],
    ]
    .assign(
        gene_id=lambda data: pd.to_numeric(
            data["gene_id"],
            errors="raise",
        ).astype("int64")
    )
)

with h5py.File(
    LINCS_LEVEL5_TRT_CP_PATH,
    mode="r",
) as gctx:
    gctx_gene_ids = [
        (
            value.decode("utf-8")
            if isinstance(value, bytes)
            else str(value)
        )
        for value in gctx["0/META/ROW/id"][:]
    ]

gctx_gene_index = pd.DataFrame(
    {
        "gene_id": pd.to_numeric(
            pd.Series(gctx_gene_ids),
            errors="raise",
        ).astype("int64"),
        "gctx_gene_index": np.arange(
            len(gctx_gene_ids),
            dtype=np.int64,
        ),
    }
)

bing_gene_universe = (
    bing_gene_universe
    .merge(
        gctx_gene_index,
        on="gene_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "gene_id",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if bing_gene_universe["gctx_gene_index"].isna().any():
    raise RuntimeError(
        "At least one BING gene is missing a GCTX gene index."
    )

if len(bing_gene_universe) != 10174:
    raise RuntimeError(
        "Unexpected complete BING gene-universe size: "
        f"{len(bing_gene_universe):,}"
    )

print(
    "Complete BING gene universe:",
    f"{len(bing_gene_universe):,}",
)

display(
    bing_gene_universe
    .groupby(
        "feature_space",
        as_index=False,
    )
    .agg(
        genes=("gene_id", "size"),
    )
)

Complete BING gene universe: 10,174


,feature_space,genes
0,best inferred,9196
1,landmark,978


In [13]:
# =============================================================================
# Prepare frozen primary-signature GCTX indices
# =============================================================================

primary_signature_indices = (
    pd.to_numeric(
        primary_signature_handoff["gctx_signature_index"],
        errors="raise",
    )
    .astype("int64")
    .to_numpy()
)

if len(primary_signature_indices) != 362036:
    raise RuntimeError(
        "Unexpected primary-signature count: "
        f"{len(primary_signature_indices):,}"
    )

if len(np.unique(primary_signature_indices)) != len(
    primary_signature_indices
):
    raise RuntimeError(
        "Primary-signature GCTX indices are not unique."
    )

primary_signature_indices = np.sort(
    primary_signature_indices,
    kind="stable",
)

print(
    "Frozen primary GCTX signature indices:",
    f"{len(primary_signature_indices):,}",
)
print(
    "Index range:",
    f"{primary_signature_indices[0]:,}"
    " to "
    f"{primary_signature_indices[-1]:,}",
)

Frozen primary GCTX signature indices: 362,036
Index range: 0 to 719,985


In [14]:
# =============================================================================
# Inspect GCTX storage layout for exact MAD computation
# =============================================================================

with h5py.File(
    LINCS_LEVEL5_TRT_CP_PATH,
    mode="r",
) as gctx:
    matrix = gctx["0/DATA/0/matrix"]

    print("Matrix shape:", matrix.shape)
    print("Matrix dtype:", matrix.dtype)
    print("HDF5 chunks:", matrix.chunks)
    print("Compression:", matrix.compression)
    print(
        "Primary signatures:",
        f"{len(primary_signature_indices):,}",
    )

Matrix shape: (720216, 12328)
Matrix dtype: float32
HDF5 chunks: None
Compression: None
Primary signatures: 362,036


In [15]:
# =============================================================================
# Check temporary-storage requirements for exact BING MAD computation
# =============================================================================

TEMP_DIR = Path(tempfile.gettempdir())

n_primary_signatures = len(primary_signature_indices)
n_bing_genes = len(bing_gene_universe)

bing_matrix_bytes = (
    n_primary_signatures
    * n_bing_genes
    * np.dtype(np.float32).itemsize
)

disk_usage = shutil.disk_usage(TEMP_DIR)

print("Temporary directory:", TEMP_DIR)
print(
    "Projected BING matrix size:",
    f"{bing_matrix_bytes / 1024**3:.2f} GiB",
)
print(
    "Free temporary-storage space:",
    f"{disk_usage.free / 1024**3:.2f} GiB",
)

Temporary directory: C:\Users\paula\AppData\Local\Temp
Projected BING matrix size: 13.72 GiB
Free temporary-storage space: 294.21 GiB


In [16]:
# =============================================================================
# Materialize primary-signature BING matrix for exact MAD computation
# =============================================================================

TEMP_BING_MATRIX_PATH = (
    TEMP_DIR
    / "phase7_700_primary_bing_matrix.npy"
)

READ_BLOCK_ROWS = 4096

bing_gctx_indices = (
    bing_gene_universe["gctx_gene_index"]
    .to_numpy(dtype=np.int64)
)

primary_bing_matrix = np.lib.format.open_memmap(
    TEMP_BING_MATRIX_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(
        n_bing_genes,
        n_primary_signatures,
    ),
)

with h5py.File(
    LINCS_LEVEL5_TRT_CP_PATH,
    mode="r",
) as gctx:
    matrix = gctx["0/DATA/0/matrix"]

    for block_start in range(
        0,
        matrix.shape[0],
        READ_BLOCK_ROWS,
    ):
        block_stop = min(
            block_start + READ_BLOCK_ROWS,
            matrix.shape[0],
        )

        left = np.searchsorted(
            primary_signature_indices,
            block_start,
            side="left",
        )
        right = np.searchsorted(
            primary_signature_indices,
            block_stop,
            side="left",
        )

        if left == right:
            continue

        block = matrix[
            block_start:block_stop,
            :,
        ]

        local_primary_rows = (
            primary_signature_indices[left:right]
            - block_start
        )

        primary_bing_matrix[
            :,
            left:right,
        ] = block[
            local_primary_rows
        ][
            :,
            bing_gctx_indices
        ].T

primary_bing_matrix.flush()

print(
    "Temporary primary BING matrix:",
    project_relative_path(TEMP_BING_MATRIX_PATH)
    if TEMP_BING_MATRIX_PATH.is_relative_to(Paths.root)
    else TEMP_BING_MATRIX_PATH,
)
print(
    "Shape:",
    primary_bing_matrix.shape,
)
print(
    "Dtype:",
    primary_bing_matrix.dtype,
)
print(
    "Size:",
    f"{TEMP_BING_MATRIX_PATH.stat().st_size / 1024**3:.2f} GiB",
)

Temporary primary BING matrix: C:\Users\paula\AppData\Local\Temp\phase7_700_primary_bing_matrix.npy
Shape: (10174, 362036)
Dtype: float32
Size: 13.72 GiB


In [17]:
# =============================================================================
# Compute exact gene-level perturbational MAD across primary signatures
# =============================================================================

MAD_GENE_BATCH_SIZE = 32

gene_medians = np.empty(
    n_bing_genes,
    dtype=np.float64,
)

gene_mads = np.empty(
    n_bing_genes,
    dtype=np.float64,
)

for batch_start in range(
    0,
    n_bing_genes,
    MAD_GENE_BATCH_SIZE,
):
    batch_stop = min(
        batch_start + MAD_GENE_BATCH_SIZE,
        n_bing_genes,
    )

    values = np.asarray(
        primary_bing_matrix[
            batch_start:batch_stop,
            :
        ],
        dtype=np.float64,
    )

    if not np.isfinite(values).all():
        raise RuntimeError(
            "Non-finite Level 5 values encountered "
            f"for BING genes {batch_start}:{batch_stop}."
        )

    batch_medians = np.median(
        values,
        axis=1,
    )

    batch_mads = np.median(
        np.abs(
            values
            - batch_medians[:, np.newaxis]
        ),
        axis=1,
    )

    gene_medians[
        batch_start:batch_stop
    ] = batch_medians

    gene_mads[
        batch_start:batch_stop
    ] = batch_mads

bing_gene_dispersion = (
    bing_gene_universe
    .copy()
    .assign(
        perturbational_median=gene_medians,
        perturbational_mad=gene_mads,
    )
)

if not np.isfinite(
    bing_gene_dispersion[
        "perturbational_mad"
    ].to_numpy()
).all():
    raise RuntimeError(
        "At least one BING gene has a non-finite MAD."
    )

print(
    "BING genes with exact MAD:",
    f"{len(bing_gene_dispersion):,}",
)

display(
    bing_gene_dispersion[
        "perturbational_mad"
    ].describe()
)

BING genes with exact MAD: 10,174


count    10174.000000
mean         0.499179
std          0.019002
min          0.000000
25%          0.489956
50%          0.494616
75%          0.501806
max          0.612916
Name: perturbational_mad, dtype: float64

In [18]:
# =============================================================================
# Assign deterministic feature-space × MAD-decile strata
# =============================================================================

stratified_gene_blocks = []

for feature_space in (
    "best inferred",
    "landmark",
):
    feature_genes = (
        bing_gene_dispersion.loc[
            bing_gene_dispersion[
                "feature_space"
            ].eq(feature_space)
        ]
        .sort_values(
            [
                "perturbational_mad",
                "gene_id",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    n_genes = len(feature_genes)
    base_size, remainder = divmod(
        n_genes,
        10,
    )

    decile_sizes = np.array(
        [
            base_size + (decile < remainder)
            for decile in range(10)
        ],
        dtype=np.int64,
    )

    feature_genes["mad_decile"] = np.repeat(
        np.arange(
            1,
            11,
            dtype=np.int8,
        ),
        decile_sizes,
    )

    stratified_gene_blocks.append(
        feature_genes
    )

null_stratification_genes = (
    pd.concat(
        stratified_gene_blocks,
        ignore_index=True,
    )
)

stratum_order = (
    null_stratification_genes[
        [
            "feature_space",
            "mad_decile",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "feature_space",
            "mad_decile",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

stratum_order["stratum_id"] = np.arange(
    len(stratum_order),
    dtype=np.int64,
)

null_stratification_genes = (
    null_stratification_genes
    .merge(
        stratum_order,
        on=[
            "feature_space",
            "mad_decile",
        ],
        how="left",
        validate="many_to_one",
    )
)

stratum_summary = (
    null_stratification_genes
    .groupby(
        [
            "stratum_id",
            "feature_space",
            "mad_decile",
        ],
        as_index=False,
    )
    .agg(
        genes=("gene_id", "size"),
        min_mad=("perturbational_mad", "min"),
        max_mad=("perturbational_mad", "max"),
    )
)

print(
    "Null strata:",
    len(stratum_summary),
)
print(
    "Genes with MAD == 0:",
    int(
        null_stratification_genes[
            "perturbational_mad"
        ].eq(0).sum()
    ),
)

display(stratum_summary)

Null strata: 20
Genes with MAD == 0: 1


,stratum_id,feature_space,mad_decile,genes,min_mad,max_mad
0,0,best inferred,1,920,0.118600,0.486450
1,1,best inferred,2,920,0.486450,0.488641
2,2,best inferred,3,920,0.488645,0.490355
3,3,best inferred,4,920,0.490355,0.491942
4,4,best inferred,5,920,0.491942,0.493686
5,5,best inferred,6,920,0.493692,0.495577
6,6,best inferred,7,919,0.495578,0.497707
7,7,best inferred,8,919,0.497711,0.500791
8,8,best inferred,9,919,0.500794,0.505192
9,9,best inferred,10,919,0.505197,0.530183


In [19]:
# =============================================================================
# Map frozen program queries to the null-stratification strata
# =============================================================================

query_stratification = (
    program_query_definitions
    .merge(
        null_stratification_genes[
            [
                "gene_id",
                "feature_space",
                "perturbational_mad",
                "mad_decile",
                "stratum_id",
            ]
        ],
        on=[
            "gene_id",
            "feature_space",
        ],
        how="left",
        validate="many_to_one",
    )
)

if query_stratification[
    [
        "perturbational_mad",
        "mad_decile",
        "stratum_id",
    ]
].isna().any().any():
    raise RuntimeError(
        "At least one frozen query gene could not be assigned "
        "to a null stratum."
    )

query_counts = (
    query_stratification
    .groupby(
        [
            "program_id",
            "stratum_id",
        ],
        as_index=False,
    )
    .agg(
        query_genes=("gene_id", "size"),
    )
)

program_stratum_grid = (
    pd.DataFrame(
        {
            "program_id": PROGRAM_IDS,
        }
    )
    .merge(
        stratum_summary[
            [
                "stratum_id",
                "feature_space",
                "mad_decile",
                "genes",
            ]
        ],
        how="cross",
    )
    .rename(
        columns={
            "genes": "total_bing_genes",
        }
    )
)

query_stratum_summary = (
    program_stratum_grid
    .merge(
        query_counts,
        on=[
            "program_id",
            "stratum_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .fillna(
        {
            "query_genes": 0,
        }
    )
)

query_stratum_summary["query_genes"] = (
    query_stratum_summary["query_genes"]
    .astype("int64")
)

query_stratum_summary[
    "effectively_permutable_query_genes"
] = np.where(
    query_stratum_summary["query_genes"] > 1,
    query_stratum_summary["query_genes"],
    0,
)

query_stratum_summary[
    "degenerate_one_gene_stratum"
] = query_stratum_summary[
    "query_genes"
].eq(1)

print(
    "Program × stratum rows:",
    len(query_stratum_summary),
)
print(
    "Degenerate one-gene strata:",
    int(
        query_stratum_summary[
            "degenerate_one_gene_stratum"
        ].sum()
    ),
)
print(
    "Empty query strata:",
    int(
        query_stratum_summary[
            "query_genes"
        ].eq(0).sum()
    ),
)

display(query_stratum_summary)

Program × stratum rows: 60
Degenerate one-gene strata: 0
Empty query strata: 0


,program_id,stratum_id,feature_space,mad_decile,total_bing_genes,query_genes,effectively_permutable_query_genes,degenerate_one_gene_stratum
0,CONSENSUS_TX_01,0,best inferred,1,920,236,236,False
1,CONSENSUS_TX_01,1,best inferred,2,920,229,229,False
2,CONSENSUS_TX_01,2,best inferred,3,920,154,154,False
3,CONSENSUS_TX_01,3,best inferred,4,920,138,138,False
4,CONSENSUS_TX_01,4,best inferred,5,920,114,114,False
5,CONSENSUS_TX_01,5,best inferred,6,920,117,117,False
6,CONSENSUS_TX_01,6,best inferred,7,919,107,107,False
7,CONSENSUS_TX_01,7,best inferred,8,919,106,106,False
8,CONSENSUS_TX_01,8,best inferred,9,919,114,114,False
9,CONSENSUS_TX_01,9,best inferred,10,919,90,90,False


In [20]:
# =============================================================================
# Construct coupled three-program weight vectors for the primary null
# =============================================================================

coupled_query_weights = (
    query_stratification[
        [
            "gene_id",
            "gene_symbol",
            "feature_space",
            "perturbational_mad",
            "mad_decile",
            "stratum_id",
            "program_id",
            "frozen_weight",
        ]
    ]
    .pivot(
        index=[
            "gene_id",
            "gene_symbol",
            "feature_space",
            "perturbational_mad",
            "mad_decile",
            "stratum_id",
        ],
        columns="program_id",
        values="frozen_weight",
    )
    .reset_index()
)

coupled_query_weights.columns.name = None

if coupled_query_weights[
    list(PROGRAM_IDS)
].isna().any().any():
    raise RuntimeError(
        "At least one query gene lacks a complete "
        "three-program signed-weight vector."
    )

if len(coupled_query_weights) != 1495:
    raise RuntimeError(
        "Unexpected coupled-query gene count: "
        f"{len(coupled_query_weights):,}"
    )

coupled_query_weights = (
    coupled_query_weights
    .sort_values(
        [
            "stratum_id",
            "gene_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

coupled_query_weights["permutation_position"] = (
    coupled_query_weights
    .groupby(
        "stratum_id",
        sort=False,
    )
    .cumcount()
    .astype("int64")
)

positions_per_stratum = (
    coupled_query_weights
    .groupby(
        "stratum_id",
        as_index=False,
    )
    .agg(
        distinct_permutation_positions=(
            "permutation_position",
            "size",
        )
    )
)

query_stratum_summary = (
    query_stratum_summary
    .merge(
        positions_per_stratum,
        on="stratum_id",
        how="left",
        validate="many_to_one",
    )
)

print(
    "Coupled query genes:",
    f"{len(coupled_query_weights):,}",
)
print(
    "Complete three-program weight vectors:",
    coupled_query_weights[
        list(PROGRAM_IDS)
    ].notna().all(axis=1).all(),
)

display(
    query_stratum_summary[
        [
            "program_id",
            "stratum_id",
            "feature_space",
            "mad_decile",
            "total_bing_genes",
            "query_genes",
            "effectively_permutable_query_genes",
            "degenerate_one_gene_stratum",
            "distinct_permutation_positions",
        ]
    ]
)

Coupled query genes: 1,495
Complete three-program weight vectors: True


,program_id,stratum_id,feature_space,mad_decile,total_bing_genes,query_genes,effectively_permutable_query_genes,degenerate_one_gene_stratum,distinct_permutation_positions
0,CONSENSUS_TX_01,0,best inferred,1,920,236,236,False,236
1,CONSENSUS_TX_01,1,best inferred,2,920,229,229,False,229
2,CONSENSUS_TX_01,2,best inferred,3,920,154,154,False,154
3,CONSENSUS_TX_01,3,best inferred,4,920,138,138,False,138
4,CONSENSUS_TX_01,4,best inferred,5,920,114,114,False,114
5,CONSENSUS_TX_01,5,best inferred,6,920,117,117,False,117
6,CONSENSUS_TX_01,6,best inferred,7,919,107,107,False,107
7,CONSENSUS_TX_01,7,best inferred,8,919,106,106,False,106
8,CONSENSUS_TX_01,8,best inferred,9,919,114,114,False,114
9,CONSENSUS_TX_01,9,best inferred,10,919,90,90,False,90


In [21]:
# =============================================================================
# Freeze deterministic Phase 7 RNG and permutation mapping
# =============================================================================

RNG_ENTROPY = 701

RNG_NAMESPACES = {
    "primary_conditional_null": 1,
    "competitive_null_sensitivity": 2,
    "outcome_blind_benchmark": 3,
}

FEATURE_SPACE_ORDER = (
    "best inferred",
    "landmark",
)

STRATUM_IDS = tuple(
    stratum_order["stratum_id"]
    .astype(int)
    .tolist()
)


def make_phase7_rng(
    namespace,
    replicate_id,
    stratum_id,
):
    seed_sequence = np.random.SeedSequence(
        entropy=RNG_ENTROPY,
        spawn_key=(
            int(namespace),
            int(replicate_id),
            int(stratum_id),
        ),
    )

    return np.random.Generator(
        np.random.PCG64DXSM(seed_sequence)
    )


def phase7_uniform_permutation(
    n_positions,
    namespace,
    replicate_id,
    stratum_id,
):
    rng = make_phase7_rng(
        namespace=namespace,
        replicate_id=replicate_id,
        stratum_id=stratum_id,
    )

    return rng.permutation(
        int(n_positions)
    ).astype(
        np.int64,
        copy=False,
    )


print("RNG entropy:", RNG_ENTROPY)
print("RNG namespaces:", RNG_NAMESPACES)
print("Program order:", PROGRAM_IDS)
print("Feature-space order:", FEATURE_SPACE_ORDER)
print("Stratum order:", STRATUM_IDS)
print(
    "Permutation mapping:",
    "NumPy Generator.permutation on independently seeded "
    "namespace × replicate × stratum streams",
)

RNG entropy: 701
RNG namespaces: {'primary_conditional_null': 1, 'competitive_null_sensitivity': 2, 'outcome_blind_benchmark': 3}
Program order: ('CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03')
Feature-space order: ('best inferred', 'landmark')
Stratum order: (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19)
Permutation mapping: NumPy Generator.permutation on independently seeded namespace × replicate × stratum streams


In [22]:
# =============================================================================
# Validate deterministic Phase 7 permutation streams
# =============================================================================

TEST_STRATUM_ID = 0
TEST_N_POSITIONS = int(
    positions_per_stratum.loc[
        positions_per_stratum[
            "stratum_id"
        ].eq(TEST_STRATUM_ID),
        "distinct_permutation_positions",
    ].iloc[0]
)

permutation_a = phase7_uniform_permutation(
    n_positions=TEST_N_POSITIONS,
    namespace=RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
    replicate_id=0,
    stratum_id=TEST_STRATUM_ID,
)

permutation_b = phase7_uniform_permutation(
    n_positions=TEST_N_POSITIONS,
    namespace=RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
    replicate_id=0,
    stratum_id=TEST_STRATUM_ID,
)

permutation_next_replicate = (
    phase7_uniform_permutation(
        n_positions=TEST_N_POSITIONS,
        namespace=RNG_NAMESPACES[
            "outcome_blind_benchmark"
        ],
        replicate_id=1,
        stratum_id=TEST_STRATUM_ID,
    )
)

permutation_next_stratum = (
    phase7_uniform_permutation(
        n_positions=TEST_N_POSITIONS,
        namespace=RNG_NAMESPACES[
            "outcome_blind_benchmark"
        ],
        replicate_id=0,
        stratum_id=1,
    )
)

expected_positions = np.arange(
    TEST_N_POSITIONS,
    dtype=np.int64,
)

if not np.array_equal(
    permutation_a,
    permutation_b,
):
    raise RuntimeError(
        "Identical Phase 7 RNG coordinates are not reproducible."
    )

if not np.array_equal(
    np.sort(permutation_a),
    expected_positions,
):
    raise RuntimeError(
        "Generated index vector is not a valid permutation."
    )

if np.array_equal(
    permutation_a,
    permutation_next_replicate,
):
    raise RuntimeError(
        "Adjacent replicate streams unexpectedly coincide."
    )

if np.array_equal(
    permutation_a,
    permutation_next_stratum,
):
    raise RuntimeError(
        "Adjacent stratum streams unexpectedly coincide."
    )

print("Deterministic RNG reproducibility: PASS")
print("Permutation validity: PASS")
print("Replicate-stream separation: PASS")
print("Stratum-stream separation: PASS")
print(
    "Test permutation positions:",
    TEST_N_POSITIONS,
)

Deterministic RNG reproducibility: PASS
Permutation validity: PASS
Replicate-stream separation: PASS
Stratum-stream separation: PASS
Test permutation positions: 236


In [23]:
# =============================================================================
# Materialize the stable null-stratification manifest in memory
# =============================================================================

query_permutation_map = (
    coupled_query_weights[
        [
            "gene_id",
            "stratum_id",
            "permutation_position",
        ]
    ]
    .rename(
        columns={
            "permutation_position": (
                "query_permutation_position"
            ),
        }
    )
)

null_stratification_manifest = (
    null_stratification_genes
    .sort_values(
        [
            "stratum_id",
            "perturbational_mad",
            "gene_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

null_stratification_manifest[
    "stratum_gene_order"
] = (
    null_stratification_manifest
    .groupby(
        "stratum_id",
        sort=False,
    )
    .cumcount()
    .astype("int64")
)

null_stratification_manifest = (
    null_stratification_manifest
    .merge(
        query_permutation_map,
        on=[
            "gene_id",
            "stratum_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

null_stratification_manifest[
    "primary_query_member"
] = null_stratification_manifest[
    "query_permutation_position"
].notna()

null_stratification_manifest[
    "query_permutation_position"
] = (
    null_stratification_manifest[
        "query_permutation_position"
    ]
    .astype("Int64")
)

if len(null_stratification_manifest) != 10174:
    raise RuntimeError(
        "Unexpected null-stratification manifest size."
    )

if not null_stratification_manifest[
    "gene_id"
].is_unique:
    raise RuntimeError(
        "Gene IDs are not unique in the "
        "null-stratification manifest."
    )

if (
    null_stratification_manifest[
        "stratum_id"
    ].nunique()
    != 20
):
    raise RuntimeError(
        "Unexpected number of null strata."
    )

if (
    null_stratification_manifest[
        "primary_query_member"
    ].sum()
    != 1495
):
    raise RuntimeError(
        "Unexpected primary-query membership "
        "in the null-stratification manifest."
    )

print(
    "Null-stratification manifest rows:",
    f"{len(null_stratification_manifest):,}",
)
print(
    "Frozen null strata:",
    null_stratification_manifest[
        "stratum_id"
    ].nunique(),
)
print(
    "Primary-query genes:",
    int(
        null_stratification_manifest[
            "primary_query_member"
        ].sum()
    ),
)

display(
    null_stratification_manifest.head()
)

Null-stratification manifest rows: 10,174
Frozen null strata: 20
Primary-query genes: 1495


,gene_id,gene_symbol,feature_space,gctx_gene_index,perturbational_median,perturbational_mad,mad_decile,stratum_id,stratum_gene_order,query_permutation_position,primary_query_member
0,7431,VIM,best inferred,9631,0.000000,0.118600,1,0,0,<NA>,False
1,3945,LDHB,best inferred,4878,0.000000,0.282002,1,0,1,<NA>,False
2,6277,S100A6,best inferred,8401,0.000000,0.348789,1,0,2,119,True
3,2171,FABP5,best inferred,2112,0.000000,0.356900,1,0,3,37,True
4,6171,RPL41,best inferred,8329,0.003999,0.405973,1,0,4,<NA>,False


In [24]:
# =============================================================================
# Validate the frozen null-stratification manifest
# =============================================================================

manifest_stratum_summary = (
    null_stratification_manifest
    .groupby(
        [
            "stratum_id",
            "feature_space",
            "mad_decile",
        ],
        as_index=False,
    )
    .agg(
        total_bing_genes=("gene_id", "size"),
        query_genes=("primary_query_member", "sum"),
    )
)

manifest_stratum_summary[
    "query_genes"
] = manifest_stratum_summary[
    "query_genes"
].astype("int64")

for stratum_id, stratum_data in (
    null_stratification_manifest.loc[
        null_stratification_manifest[
            "primary_query_member"
        ]
    ]
    .groupby(
        "stratum_id",
        sort=True,
    )
):
    observed_positions = (
        stratum_data[
            "query_permutation_position"
        ]
        .astype("int64")
        .sort_values()
        .to_numpy()
    )

    expected_positions = np.arange(
        len(stratum_data),
        dtype=np.int64,
    )

    if not np.array_equal(
        observed_positions,
        expected_positions,
    ):
        raise RuntimeError(
            "Non-contiguous query permutation positions "
            f"in stratum {stratum_id}."
        )

manifest_program_summary = (
    pd.DataFrame(
        {
            "program_id": PROGRAM_IDS,
        }
    )
    .merge(
        manifest_stratum_summary,
        how="cross",
    )
    .assign(
        effectively_permutable_query_genes=lambda data: (
            np.where(
                data["query_genes"] > 1,
                data["query_genes"],
                0,
            )
        ),
        degenerate_one_gene_stratum=lambda data: (
            data["query_genes"].eq(1)
        ),
        distinct_permutation_positions=lambda data: (
            data["query_genes"]
        ),
    )
    .sort_values(
        [
            "program_id",
            "stratum_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

comparison_columns = [
    "program_id",
    "stratum_id",
    "total_bing_genes",
    "query_genes",
    "effectively_permutable_query_genes",
    "degenerate_one_gene_stratum",
    "distinct_permutation_positions",
]

expected_summary = (
    query_stratum_summary[
        comparison_columns
    ]
    .sort_values(
        [
            "program_id",
            "stratum_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not manifest_program_summary[
    comparison_columns
].equals(expected_summary):
    raise RuntimeError(
        "Null-stratification manifest does not reproduce "
        "the frozen program × stratum summary."
    )

print("Null-stratification manifest validation: PASS")
print(
    "Program × stratum combinations:",
    len(manifest_program_summary),
)
print(
    "Degenerate one-gene strata:",
    int(
        manifest_program_summary[
            "degenerate_one_gene_stratum"
        ].sum()
    ),
)
print(
    "Minimum query genes per stratum:",
    int(
        manifest_program_summary[
            "query_genes"
        ].min()
    ),
)
print(
    "Maximum query genes per stratum:",
    int(
        manifest_program_summary[
            "query_genes"
        ].max()
    ),
)

Null-stratification manifest validation: PASS
Program × stratum combinations: 60
Degenerate one-gene strata: 0
Minimum query genes per stratum: 5
Maximum query genes per stratum: 236


In [25]:
# =============================================================================
# Inspect execution host before freezing the benchmark resource envelope
# =============================================================================

powershell_script = r"""
$cpu = Get-CimInstance Win32_Processor | Select-Object -First 1
$system = Get-CimInstance Win32_ComputerSystem

[PSCustomObject]@{
    cpu_model = $cpu.Name
    physical_cores = $cpu.NumberOfCores
    logical_cores = $cpu.NumberOfLogicalProcessors
    physical_ram_bytes = [int64]$system.TotalPhysicalMemory
} | ConvertTo-Json -Compress
"""

host_result = subprocess.run(
    [
        "powershell",
        "-NoProfile",
        "-Command",
        powershell_script,
    ],
    check=True,
    capture_output=True,
    text=True,
)

windows_hardware = json.loads(
    host_result.stdout.strip()
)

threading_info = threadpool_info()

execution_host_info = {
    "host_identifier": platform.node(),
    "operating_system": platform.platform(),
    "cpu_model": windows_hardware["cpu_model"].strip(),
    "physical_cores": int(
        windows_hardware["physical_cores"]
    ),
    "logical_cores": int(
        windows_hardware["logical_cores"]
    ),
    "physical_ram_gib": (
        int(windows_hardware["physical_ram_bytes"])
        / 1024**3
    ),
    "gpu_used": False,
    "gpu_model": None,
    "usable_gpu_vram_gib": None,
    "python_version": sys.version.split()[0],
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "h5py_version": h5py.__version__,
    "threadpoolctl_version": package_version(
        "threadpoolctl"
    ),
    "thread_environment": {
        variable: os.environ.get(variable)
        for variable in (
            "OMP_NUM_THREADS",
            "OPENBLAS_NUM_THREADS",
            "MKL_NUM_THREADS",
            "NUMEXPR_NUM_THREADS",
        )
    },
    "threadpool_info": threading_info,
}

print(
    "Host identifier:",
    execution_host_info["host_identifier"],
)
print(
    "CPU:",
    execution_host_info["cpu_model"],
)
print(
    "Physical / logical cores:",
    execution_host_info["physical_cores"],
    "/",
    execution_host_info["logical_cores"],
)
print(
    "Physical RAM:",
    f'{execution_host_info["physical_ram_gib"]:.2f} GiB',
)
print(
    "GPU used:",
    execution_host_info["gpu_used"],
)
print(
    "Python:",
    execution_host_info["python_version"],
)
print(
    "NumPy:",
    execution_host_info["numpy_version"],
)
print(
    "pandas:",
    execution_host_info["pandas_version"],
)
print(
    "h5py:",
    execution_host_info["h5py_version"],
)
print(
    "threadpoolctl:",
    execution_host_info["threadpoolctl_version"],
)

print("\nThread environment:")
display(
    pd.DataFrame(
        execution_host_info[
            "thread_environment"
        ].items(),
        columns=[
            "variable",
            "value",
        ],
    )
)

print("Loaded numerical thread pools:")
display(
    pd.DataFrame(threading_info)
)

Host identifier: CompuPaulaCasa
CPU: Intel(R) Core(TM) i7-1065G7 CPU @ 1.30GHz
Physical / logical cores: 4 / 8
Physical RAM: 19.80 GiB
GPU used: False
Python: 3.11.8
NumPy: 2.4.6
pandas: 3.0.3
h5py: 3.16.0
threadpoolctl: 3.6.0

Thread environment:


,variable,value
0,OMP_NUM_THREADS,None
1,OPENBLAS_NUM_THREADS,None
2,MKL_NUM_THREADS,None
3,NUMEXPR_NUM_THREADS,None


Loaded numerical thread pools:


,user_api,internal_api,num_threads,prefix,filepath,version,threading_layer,architecture
0,blas,openblas,8,libscipy_openblas,C:\Users\paula\OneDrive\Documentos\Proyectos\p...,0.3.31.188.0,pthreads,SkylakeX


In [26]:
# =============================================================================
# Freeze and persist the pre-benchmark resource envelope
# =============================================================================

pip_freeze = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "freeze",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.splitlines()

BENCHMARK_RESOURCE_ENVELOPE_PATH = (
    Paths.perturbational_hypotheses
    / "700_benchmark_resource_envelope.json"
)

BENCHMARK_RESOURCE_ENVELOPE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

current_free_temp_gib = (
    shutil.disk_usage(TEMP_DIR).free
    / 1024**3
)

benchmark_resource_envelope = {
    "execution_host": {
        "host_identifier": execution_host_info[
            "host_identifier"
        ],
        "operating_system": execution_host_info[
            "operating_system"
        ],
        "cpu_model": execution_host_info[
            "cpu_model"
        ],
        "physical_cores": execution_host_info[
            "physical_cores"
        ],
        "logical_cores": execution_host_info[
            "logical_cores"
        ],
        "physical_ram_gib": round(
            execution_host_info[
                "physical_ram_gib"
            ],
            6,
        ),
        "gpu_used": False,
        "gpu_model": None,
        "usable_gpu_vram_gib": None,
    },
    "resource_limits": {
        "maximum_ram_gib": 12.0,
        "maximum_projected_primary_wall_clock_hours": 24.0,
        "maximum_total_benchmark_wall_clock_hours": 2.0,
        "maximum_temporary_storage_gib": 64.0,
        "free_temporary_storage_gib_at_freeze": round(
            current_free_temp_gib,
            6,
        ),
    },
    "threading": {
        "environment": execution_host_info[
            "thread_environment"
        ],
        "threadpool_info": execution_host_info[
            "threadpool_info"
        ],
    },
    "checkpoint_restart_policy": {
        "checkpoint_interval_replicates": 1000,
        "checkpoint_at_case_completion": True,
        "checkpoint_write_mode": (
            "atomic temporary-file write followed by replace"
        ),
        "restart_rule": (
            "resume from the highest fully committed replicate "
            "boundary; discard any incomplete replicate"
        ),
        "rng_state_restoration_required": False,
        "rng_restart_basis": (
            "namespace × replicate_id × stratum_id"
        ),
    },
    "benchmark_design_freeze": {
        "rng_namespace": RNG_NAMESPACES[
            "outcome_blind_benchmark"
        ],
        "candidate_bmax_ladder": [
            500000,
            250000,
            100000,
        ],
        "primary_hypothesis_count": 23742,
        "active_set_cases": [
            23742,
            11871,
            2375,
        ],
        "active_set_case_labels": [
            "maximum_100_percent",
            "reduced_50_percent",
            "reduced_10_percent_ceiling",
        ],
        "selection_rule": (
            "select the largest candidate Bmax whose "
            "conservative projected execution satisfies "
            "all frozen resource limits"
        ),
    },
    "software_environment": {
        "python": execution_host_info[
            "python_version"
        ],
        "numpy": execution_host_info[
            "numpy_version"
        ],
        "pandas": execution_host_info[
            "pandas_version"
        ],
        "h5py": execution_host_info[
            "h5py_version"
        ],
        "threadpoolctl": execution_host_info[
            "threadpoolctl_version"
        ],
        "pip_freeze": pip_freeze,
    },
}


def normalize_resource_envelope_for_comparison(
    envelope,
):
    normalized = json.loads(
        json.dumps(envelope)
    )

    # This value records free disk space at the original freeze.
    # It is historical provenance and is not expected to remain
    # identical across later executions.
    normalized[
        "resource_limits"
    ].pop(
        "free_temporary_storage_gib_at_freeze",
        None,
    )

    software = normalized[
        "software_environment"
    ]

    pip_entries = software.get(
        "pip_freeze",
        [],
    )

    # The editable project entry changes SHA whenever the checked-out
    # repository commit changes. Git provenance tracks that separately;
    # it is not an external dependency change.
    software["pip_freeze"] = [
        entry
        for entry in pip_entries
        if not (
            entry.startswith("-e git+")
            and (
                "pancancer-epigenetics.git@"
                in entry
            )
            and (
                "#egg=pancancer_epigenetics"
                in entry
            )
        )
    ]

    # Legacy pre-benchmark envelope: threadpoolctl was already frozen
    # inside pip_freeze but was not yet duplicated as an explicit field.
    if "threadpoolctl" not in software:
        threadpool_entries = [
            entry
            for entry in pip_entries
            if entry.lower().startswith(
                "threadpoolctl=="
            )
        ]

        if len(threadpool_entries) != 1:
            raise RuntimeError(
                "Cannot unambiguously recover the frozen "
                "threadpoolctl version."
            )

        software["threadpoolctl"] = (
            threadpool_entries[0]
            .split("==", 1)[1]
        )

    return normalized


required_temp_gib = (
    benchmark_resource_envelope[
        "resource_limits"
    ][
        "maximum_temporary_storage_gib"
    ]
)

if current_free_temp_gib < required_temp_gib:
    raise RuntimeError(
        "Current temporary-storage availability is below "
        "the prospectively frozen requirement: "
        f"{current_free_temp_gib:.2f} GiB available vs "
        f"{required_temp_gib:.2f} GiB required."
    )


if BENCHMARK_RESOURCE_ENVELOPE_PATH.exists():
    with BENCHMARK_RESOURCE_ENVELOPE_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:
        persisted_resource_envelope = json.load(
            handle
        )

    normalized_persisted = (
        normalize_resource_envelope_for_comparison(
            persisted_resource_envelope
        )
    )

    normalized_current = (
        normalize_resource_envelope_for_comparison(
            benchmark_resource_envelope
        )
    )

    if normalized_persisted != normalized_current:
        raise RuntimeError(
            "The frozen benchmark resource envelope "
            "differs from the current execution environment "
            "after applying only the prespecified "
            "non-substantive normalization rules. "
            "Refusing to proceed before benchmark timing."
        )

    if (
        "threadpoolctl"
        not in persisted_resource_envelope[
            "software_environment"
        ]
    ):
        frozen_threadpool_entries = [
            entry
            for entry in persisted_resource_envelope[
                "software_environment"
            ]["pip_freeze"]
            if entry.lower().startswith(
                "threadpoolctl=="
            )
        ]

        if len(frozen_threadpool_entries) != 1:
            raise RuntimeError(
                "Frozen pip environment does not contain "
                "exactly one threadpoolctl version."
            )

        frozen_threadpool_version = (
            frozen_threadpool_entries[0]
            .split("==", 1)[1]
        )

        if (
            frozen_threadpool_version
            != execution_host_info[
                "threadpoolctl_version"
            ]
        ):
            raise RuntimeError(
                "Current threadpoolctl version differs "
                "from the version already frozen in "
                "pip_freeze."
            )

        persisted_resource_envelope[
            "software_environment"
        ]["threadpoolctl"] = (
            frozen_threadpool_version
        )

        with BENCHMARK_RESOURCE_ENVELOPE_PATH.open(
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                persisted_resource_envelope,
                handle,
                indent=2,
                sort_keys=True,
            )
            handle.write("\n")

        resource_envelope_action = (
            "AUGMENTED_PRE_TIMING_PROVENANCE"
        )

    else:
        resource_envelope_action = (
            "REUSED_VALIDATED_FREEZE"
        )

else:
    with BENCHMARK_RESOURCE_ENVELOPE_PATH.open(
        "x",
        encoding="utf-8",
    ) as handle:
        json.dump(
            benchmark_resource_envelope,
            handle,
            indent=2,
            sort_keys=True,
        )
        handle.write("\n")

    resource_envelope_action = "CREATED"


with BENCHMARK_RESOURCE_ENVELOPE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    persisted_resource_envelope = json.load(
        handle
    )

if (
    normalize_resource_envelope_for_comparison(
        persisted_resource_envelope
    )
    !=
    normalize_resource_envelope_for_comparison(
        benchmark_resource_envelope
    )
):
    raise RuntimeError(
        "Persisted benchmark resource envelope failed "
        "post-validation."
    )

print(
    "Pre-benchmark resource envelope: FROZEN"
)
print(
    "Freeze action:",
    resource_envelope_action,
)
print(
    "Path:",
    project_relative_path(
        BENCHMARK_RESOURCE_ENVELOPE_PATH
    ),
)
print(
    "Maximum RAM:",
    benchmark_resource_envelope[
        "resource_limits"
    ]["maximum_ram_gib"],
    "GiB",
)
print(
    "Maximum projected primary runtime:",
    benchmark_resource_envelope[
        "resource_limits"
    ][
        "maximum_projected_primary_wall_clock_hours"
    ],
    "hours",
)
print(
    "Temporary-storage allowance:",
    benchmark_resource_envelope[
        "resource_limits"
    ]["maximum_temporary_storage_gib"],
    "GiB",
)
print(
    "Current free temporary storage:",
    f"{current_free_temp_gib:.2f} GiB",
)
print(
    "Checkpoint interval:",
    benchmark_resource_envelope[
        "checkpoint_restart_policy"
    ]["checkpoint_interval_replicates"],
    "replicates",
)
print(
    "Benchmark active-set cases:",
    benchmark_resource_envelope[
        "benchmark_design_freeze"
    ]["active_set_cases"],
)

Pre-benchmark resource envelope: FROZEN
Freeze action: REUSED_VALIDATED_FREEZE
Path: data/processed/perturbational_hypotheses/700_benchmark_resource_envelope.json
Maximum RAM: 12.0 GiB
Maximum projected primary runtime: 24.0 hours
Temporary-storage allowance: 64.0 GiB
Current free temporary storage: 299.52 GiB
Checkpoint interval: 1000 replicates
Benchmark active-set cases: [23742, 11871, 2375]


In [27]:
# =============================================================================
# Diagnose differences against the frozen benchmark resource envelope
# =============================================================================

def flatten_mapping(mapping, prefix=""):
    flattened = {}

    for key, value in mapping.items():
        path = f"{prefix}.{key}" if prefix else str(key)

        if isinstance(value, dict):
            flattened.update(
                flatten_mapping(
                    value,
                    prefix=path,
                )
            )
        else:
            flattened[path] = value

    return flattened


with BENCHMARK_RESOURCE_ENVELOPE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    frozen_envelope_for_diagnosis = json.load(handle)

frozen_flat = flatten_mapping(
    frozen_envelope_for_diagnosis
)

current_flat = flatten_mapping(
    benchmark_resource_envelope
)

all_paths = sorted(
    set(frozen_flat)
    | set(current_flat)
)

envelope_differences = []

for path in all_paths:
    frozen_value = frozen_flat.get(
        path,
        "<MISSING>",
    )
    current_value = current_flat.get(
        path,
        "<MISSING>",
    )

    if frozen_value != current_value:
        envelope_differences.append(
            {
                "field": path,
                "frozen_value": frozen_value,
                "current_value": current_value,
            }
        )

difference_frame = pd.DataFrame(
    envelope_differences
)

print(
    "Frozen-envelope differences:",
    len(difference_frame),
)

display(difference_frame)

Frozen-envelope differences: 2


,field,frozen_value,current_value
0,resource_limits.free_temporary_storage_gib_at_...,309.231491,299.524273
1,software_environment.pip_freeze,"[anyio==4.13.0, argon2-cffi==25.1.0, argon2-cf...","[anyio==4.13.0, argon2-cffi==25.1.0, argon2-cf..."


In [28]:
# =============================================================================
# Diagnose exact pip-freeze differences
# =============================================================================

frozen_pip_freeze = set(
    frozen_envelope_for_diagnosis[
        "software_environment"
    ]["pip_freeze"]
)

current_pip_freeze = set(
    benchmark_resource_envelope[
        "software_environment"
    ]["pip_freeze"]
)

only_in_frozen = sorted(
    frozen_pip_freeze
    - current_pip_freeze
)

only_in_current = sorted(
    current_pip_freeze
    - frozen_pip_freeze
)

print(
    "Only in frozen environment:",
    len(only_in_frozen),
)
for item in only_in_frozen:
    print("  -", item)

print(
    "\nOnly in current environment:",
    len(only_in_current),
)
for item in only_in_current:
    print("  +", item)

print(
    "\nFrozen threadpoolctl entry:",
    [
        item
        for item in frozen_pip_freeze
        if item.lower().startswith(
            "threadpoolctl=="
        )
    ],
)

Only in frozen environment: 1
  - -e git+https://github.com/paulairazoqui/pancancer-epigenetics.git@01ed216c780114d87c5ed6c426ab59f3fee434aa#egg=pancancer_epigenetics

Only in current environment: 1
  + -e git+https://github.com/paulairazoqui/pancancer-epigenetics.git@e9d5ca555853632dc2ea44e418d86d6a061fffa8#egg=pancancer_epigenetics

Frozen threadpoolctl entry: ['threadpoolctl==3.6.0']


In [29]:
# =============================================================================
# Align primary-signature metadata and define the frozen aggregation hierarchy
# =============================================================================

primary_signature_metadata = (
    primary_signature_handoff
    .sort_values(
        "gctx_signature_index",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

aligned_signature_indices = (
    primary_signature_metadata[
        "gctx_signature_index"
    ]
    .to_numpy(dtype=np.int64)
)

if not np.array_equal(
    aligned_signature_indices,
    primary_signature_indices,
):
    raise RuntimeError(
        "Primary-signature metadata are not aligned "
        "with the temporary BING matrix."
    )

aggregation_columns = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
    "pert_dose",
    "pert_dose_unit",
    "pert_time",
    "pert_time_unit",
]

if primary_signature_metadata[
    aggregation_columns
].isna().any().any():
    raise RuntimeError(
        "Frozen aggregation metadata contain missing values."
    )

condition_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
    "pert_dose",
    "pert_dose_unit",
    "pert_time",
    "pert_time_unit",
]

timepoint_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
    "pert_time",
    "pert_time_unit",
]

cell_line_keys = [
    "pert_id",
    "cell_iname",
    "cell_lineage",
]

lineage_keys = [
    "pert_id",
    "cell_lineage",
]

hierarchy_counts = {
    "signatures": len(primary_signature_metadata),
    "conditions": (
        primary_signature_metadata[
            condition_keys
        ]
        .drop_duplicates()
        .shape[0]
    ),
    "timepoints_within_cell": (
        primary_signature_metadata[
            timepoint_keys
        ]
        .drop_duplicates()
        .shape[0]
    ),
    "perturbagen_cell_lines": (
        primary_signature_metadata[
            cell_line_keys
        ]
        .drop_duplicates()
        .shape[0]
    ),
    "perturbagen_lineages": (
        primary_signature_metadata[
            lineage_keys
        ]
        .drop_duplicates()
        .shape[0]
    ),
    "perturbagens": (
        primary_signature_metadata[
            "pert_id"
        ]
        .nunique()
    ),
}

if hierarchy_counts["perturbagens"] != 7914:
    raise RuntimeError(
        "Unexpected perturbagen count in the "
        "frozen aggregation hierarchy."
    )

print("Primary-signature metadata alignment: PASS")
print(
    "Frozen aggregation hierarchy:"
)
for level, count in hierarchy_counts.items():
    print(
        f"  {level}: {count:,}"
    )

Primary-signature metadata alignment: PASS
Frozen aggregation hierarchy:
  signatures: 362,036
  conditions: 325,190
  timepoints_within_cell: 113,097
  perturbagen_cell_lines: 90,673
  perturbagen_lineages: 55,827
  perturbagens: 7,914


In [30]:
# =============================================================================
# Precompute deterministic aggregation-group mappings
# =============================================================================

def factorize_group_keys(frame, columns):
    keys = pd.MultiIndex.from_frame(
        frame[columns]
    )

    codes, uniques = pd.factorize(
        keys,
        sort=True,
    )

    return (
        codes.astype(np.int32),
        uniques,
    )


condition_id, condition_index = factorize_group_keys(
    primary_signature_metadata,
    condition_keys,
)

timepoint_id, timepoint_index = factorize_group_keys(
    primary_signature_metadata,
    timepoint_keys,
)

cell_line_id, cell_line_index = factorize_group_keys(
    primary_signature_metadata,
    cell_line_keys,
)

lineage_id, lineage_index = factorize_group_keys(
    primary_signature_metadata,
    lineage_keys,
)

perturbagen_id, perturbagen_index = factorize_group_keys(
    primary_signature_metadata,
    ["pert_id"],
)

signature_hierarchy_index = pd.DataFrame(
    {
        "condition_id": condition_id,
        "timepoint_id": timepoint_id,
        "cell_line_id": cell_line_id,
        "lineage_id": lineage_id,
        "perturbagen_id": perturbagen_id,
    }
)


def build_parent_map(
    child_ids,
    parent_ids,
    n_children,
    label,
):
    mapping = (
        pd.DataFrame(
            {
                "child_id": child_ids,
                "parent_id": parent_ids,
            }
        )
        .drop_duplicates()
    )

    if mapping["child_id"].duplicated().any():
        raise RuntimeError(
            f"{label} does not map uniquely "
            "to its parent level."
        )

    mapping = (
        mapping
        .sort_values(
            "child_id",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    if not np.array_equal(
        mapping["child_id"].to_numpy(),
        np.arange(
            n_children,
            dtype=np.int64,
        ),
    ):
        raise RuntimeError(
            f"{label} IDs are not complete and contiguous."
        )

    return mapping[
        "parent_id"
    ].to_numpy(dtype=np.int32)


condition_to_timepoint = build_parent_map(
    condition_id,
    timepoint_id,
    len(condition_index),
    "Condition",
)

timepoint_to_cell_line = build_parent_map(
    timepoint_id,
    cell_line_id,
    len(timepoint_index),
    "Timepoint",
)

cell_line_to_lineage = build_parent_map(
    cell_line_id,
    lineage_id,
    len(cell_line_index),
    "Cell line",
)

lineage_to_perturbagen = build_parent_map(
    lineage_id,
    perturbagen_id,
    len(lineage_index),
    "Lineage",
)

observed_group_counts = {
    "conditions": len(condition_index),
    "timepoints_within_cell": len(timepoint_index),
    "perturbagen_cell_lines": len(cell_line_index),
    "perturbagen_lineages": len(lineage_index),
    "perturbagens": len(perturbagen_index),
}

for level, expected_count in hierarchy_counts.items():
    if level == "signatures":
        continue

    if observed_group_counts[level] != expected_count:
        raise RuntimeError(
            f"Hierarchy count mismatch for {level}: "
            f"{observed_group_counts[level]:,} != "
            f"{expected_count:,}"
        )

print("Deterministic hierarchy mappings: PASS")
print(
    "Signature → condition:",
    f"{len(condition_id):,} → {len(condition_index):,}",
)
print(
    "Condition → timepoint:",
    f"{len(condition_to_timepoint):,} → "
    f"{len(timepoint_index):,}",
)
print(
    "Timepoint → cell line:",
    f"{len(timepoint_to_cell_line):,} → "
    f"{len(cell_line_index):,}",
)
print(
    "Cell line → lineage:",
    f"{len(cell_line_to_lineage):,} → "
    f"{len(lineage_index):,}",
)
print(
    "Lineage → perturbagen:",
    f"{len(lineage_to_perturbagen):,} → "
    f"{len(perturbagen_index):,}",
)

Deterministic hierarchy mappings: PASS
Signature → condition: 362,036 → 325,190
Condition → timepoint: 325,190 → 113,097
Timepoint → cell line: 113,097 → 90,673
Cell line → lineage: 90,673 → 55,827
Lineage → perturbagen: 55,827 → 7,914


In [31]:
# =============================================================================
# Build and validate reusable hierarchical median reductions
# =============================================================================

NUMERICAL_RTOL = 1e-10
NUMERICAL_ATOL = 1e-12


def build_median_reduction_plan(
    parent_ids,
):
    parent_ids = np.asarray(
        parent_ids,
        dtype=np.int64,
    )

    order = np.argsort(
        parent_ids,
        kind="stable",
    )

    sorted_parent_ids = parent_ids[order]

    unique_parents, starts, counts = np.unique(
        sorted_parent_ids,
        return_index=True,
        return_counts=True,
    )

    expected_parents = np.arange(
        len(unique_parents),
        dtype=np.int64,
    )

    if not np.array_equal(
        unique_parents,
        expected_parents,
    ):
        raise RuntimeError(
            "Parent IDs must be complete and contiguous."
        )

    plan = []

    for group_size in np.unique(counts):
        size_mask = counts == group_size

        parent_group_ids = unique_parents[
            size_mask
        ]

        group_starts = starts[
            size_mask
        ]

        child_indices = order[
            group_starts[:, np.newaxis]
            + np.arange(
                group_size,
                dtype=np.int64,
            )
        ]

        plan.append(
            {
                "group_size": int(group_size),
                "parent_ids": (
                    parent_group_ids.astype(
                        np.int32
                    )
                ),
                "child_indices": (
                    child_indices.astype(
                        np.int32
                    )
                ),
            }
        )

    return plan, len(unique_parents)


def apply_median_reduction(
    values,
    plan,
    n_parents,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    input_was_1d = values.ndim == 1

    if input_was_1d:
        values = values[:, np.newaxis]

    if values.ndim != 2:
        raise ValueError(
            "Median reduction expects a 1D or 2D array."
        )

    reduced = np.empty(
        (
            n_parents,
            values.shape[1],
        ),
        dtype=np.float64,
    )

    for block in plan:
        child_values = values[
            block["child_indices"],
            :,
        ]

        reduced[
            block["parent_ids"],
            :,
        ] = np.median(
            child_values,
            axis=1,
        )

    if input_was_1d:
        return reduced[:, 0]

    return reduced


aggregation_plans = {}

aggregation_plans[
    "signature_to_condition"
] = build_median_reduction_plan(
    condition_id
)

aggregation_plans[
    "condition_to_timepoint"
] = build_median_reduction_plan(
    condition_to_timepoint
)

aggregation_plans[
    "timepoint_to_cell_line"
] = build_median_reduction_plan(
    timepoint_to_cell_line
)

aggregation_plans[
    "cell_line_to_lineage"
] = build_median_reduction_plan(
    cell_line_to_lineage
)

aggregation_plans[
    "lineage_to_perturbagen"
] = build_median_reduction_plan(
    lineage_to_perturbagen
)


# Independent small float64 reference validation.
toy_scores = np.array(
    [
        [0.8, -0.3, 0.1],
        [0.2, -0.7, 0.4],
        [-0.4, 0.1, 0.3],
        [0.7, 0.2, -0.5],
        [0.1, 0.6, -0.1],
        [-0.2, 0.4, 0.9],
        [0.5, -0.8, 0.2],
        [0.3, -0.2, 0.6],
        [-0.1, 0.5, 0.7],
    ],
    dtype=np.float64,
)

toy_signature_to_condition = np.array(
    [0, 0, 1, 2, 2, 3, 4, 4, 4],
    dtype=np.int64,
)

toy_condition_to_timepoint = np.array(
    [0, 0, 1, 2, 2],
    dtype=np.int64,
)

toy_timepoint_to_cell = np.array(
    [0, 1, 1],
    dtype=np.int64,
)

toy_cell_to_lineage = np.array(
    [0, 1],
    dtype=np.int64,
)

toy_lineage_to_perturbagen = np.array(
    [0, 0],
    dtype=np.int64,
)


def reference_grouped_median(
    values,
    parent_ids,
):
    value_frame = pd.DataFrame(
        np.asarray(
            values,
            dtype=np.float64,
        )
    )

    value_frame["_parent_id"] = (
        np.asarray(
            parent_ids,
            dtype=np.int64,
        )
    )

    return (
        value_frame
        .groupby(
            "_parent_id",
            sort=True,
        )
        .median()
        .to_numpy(dtype=np.float64)
    )


toy_fast = toy_scores
toy_reference = toy_scores

for toy_parent_ids in (
    toy_signature_to_condition,
    toy_condition_to_timepoint,
    toy_timepoint_to_cell,
    toy_cell_to_lineage,
    toy_lineage_to_perturbagen,
):
    toy_plan, toy_n_parents = (
        build_median_reduction_plan(
            toy_parent_ids
        )
    )

    toy_fast = apply_median_reduction(
        toy_fast,
        toy_plan,
        toy_n_parents,
    )

    toy_reference = reference_grouped_median(
        toy_reference,
        toy_parent_ids,
    )

    np.testing.assert_allclose(
        toy_fast,
        toy_reference,
        rtol=NUMERICAL_RTOL,
        atol=NUMERICAL_ATOL,
    )

print(
    "Hierarchical median reference validation: PASS"
)

for level, (
    plan,
    n_parents,
) in aggregation_plans.items():
    print(
        f"{level}: "
        f"{n_parents:,} groups; "
        f"{len(plan)} distinct group sizes; "
        f"max size = "
        f"{max(block['group_size'] for block in plan):,}"
    )

Hierarchical median reference validation: PASS
signature_to_condition: 325,190 groups; 74 distinct group sizes; max size = 395
condition_to_timepoint: 113,097 groups; 36 distinct group sizes; max size = 41
timepoint_to_cell_line: 90,673 groups; 5 distinct group sizes; max size = 5
cell_line_to_lineage: 55,827 groups; 18 distinct group sizes; max size = 19
lineage_to_perturbagen: 7,914 groups; 17 distinct group sizes; max size = 20


In [32]:
# =============================================================================
# Define and validate the complete frozen aggregation path
# =============================================================================

def aggregate_signature_scores(
    signature_scores,
):
    signature_scores = np.asarray(
        signature_scores,
        dtype=np.float64,
    )

    if signature_scores.ndim == 1:
        signature_scores = (
            signature_scores[:, np.newaxis]
        )

    if (
        signature_scores.ndim != 2
        or signature_scores.shape[0]
        != hierarchy_counts["signatures"]
    ):
        raise ValueError(
            "Signature scores do not match the frozen "
            "primary signature universe."
        )

    condition_scores = apply_median_reduction(
        signature_scores,
        *aggregation_plans[
            "signature_to_condition"
        ],
    )

    timepoint_scores = apply_median_reduction(
        condition_scores,
        *aggregation_plans[
            "condition_to_timepoint"
        ],
    )

    cell_line_scores = apply_median_reduction(
        timepoint_scores,
        *aggregation_plans[
            "timepoint_to_cell_line"
        ],
    )

    lineage_scores = apply_median_reduction(
        cell_line_scores,
        *aggregation_plans[
            "cell_line_to_lineage"
        ],
    )

    perturbagen_scores = apply_median_reduction(
        lineage_scores,
        *aggregation_plans[
            "lineage_to_perturbagen"
        ],
    )

    return {
        "condition": condition_scores,
        "timepoint": timepoint_scores,
        "cell_line": cell_line_scores,
        "lineage": lineage_scores,
        "perturbagen": perturbagen_scores,
    }


# Outcome-blind synthetic scores for structural validation only.
synthetic_signature_scores = np.column_stack(
    [
        np.sin(
            np.arange(
                hierarchy_counts["signatures"],
                dtype=np.float64,
            )
            / scale
        )
        for scale in (
            101.0,
            211.0,
            307.0,
        )
    ]
)

synthetic_aggregation = (
    aggregate_signature_scores(
        synthetic_signature_scores
    )
)

expected_shapes = {
    "condition": (
        hierarchy_counts["conditions"],
        3,
    ),
    "timepoint": (
        hierarchy_counts[
            "timepoints_within_cell"
        ],
        3,
    ),
    "cell_line": (
        hierarchy_counts[
            "perturbagen_cell_lines"
        ],
        3,
    ),
    "lineage": (
        hierarchy_counts[
            "perturbagen_lineages"
        ],
        3,
    ),
    "perturbagen": (
        hierarchy_counts["perturbagens"],
        3,
    ),
}

for level, expected_shape in (
    expected_shapes.items()
):
    observed = synthetic_aggregation[level]

    if observed.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected {level} score shape: "
            f"{observed.shape} != "
            f"{expected_shape}"
        )

    if not np.isfinite(observed).all():
        raise RuntimeError(
            f"Non-finite synthetic {level} scores."
        )

print(
    "Complete frozen aggregation path: PASS"
)

for level in (
    "condition",
    "timepoint",
    "cell_line",
    "lineage",
    "perturbagen",
):
    print(
        f"{level}: "
        f"{synthetic_aggregation[level].shape}"
    )

Complete frozen aggregation path: PASS
condition: (325190, 3)
timepoint: (113097, 3)
cell_line: (90673, 3)
lineage: (55827, 3)
perturbagen: (7914, 3)


In [33]:
# =============================================================================
# Prepare frozen query arrays for pseudo-program cosine scoring
# =============================================================================

bing_row_lookup = (
    bing_gene_universe
    .reset_index(
        names="bing_matrix_row"
    )
    .set_index("gene_id")[
        "bing_matrix_row"
    ]
)

query_gene_ids = (
    coupled_query_weights[
        "gene_id"
    ]
    .to_numpy(dtype=np.int64)
)

query_bing_row_indices = (
    coupled_query_weights[
        "gene_id"
    ]
    .map(bing_row_lookup)
)

if query_bing_row_indices.isna().any():
    raise RuntimeError(
        "At least one frozen query gene is missing "
        "from the primary BING matrix."
    )

query_bing_row_indices = (
    query_bing_row_indices
    .to_numpy(dtype=np.int64)
)

frozen_weight_matrix = (
    coupled_query_weights[
        list(PROGRAM_IDS)
    ]
    .to_numpy(dtype=np.float64)
)

resistance_like_multiplier_vector = (
    program_source_metadata
    .set_index("program_id")
    .loc[
        list(PROGRAM_IDS),
        "resistance_like_orientation_multiplier",
    ]
    .to_numpy(dtype=np.float64)
)

frozen_weight_norms = np.linalg.norm(
    frozen_weight_matrix,
    axis=0,
)

if (
    len(np.unique(query_gene_ids))
    != 1495
):
    raise RuntimeError(
        "Frozen query support is not 1,495 unique genes."
    )

if not np.isfinite(
    frozen_weight_matrix
).all():
    raise RuntimeError(
        "Frozen query weights contain non-finite values."
    )

if not np.all(
    frozen_weight_norms > 0
):
    raise RuntimeError(
        "At least one frozen query has zero weight norm."
    )

expected_weight_norms = (
    query_norms
    .reindex(PROGRAM_IDS)
    .to_numpy(dtype=np.float64)
)

np.testing.assert_allclose(
    frozen_weight_norms,
    expected_weight_norms,
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)

print(
    "Shared frozen query support:",
    f"{len(query_gene_ids):,}",
    "genes",
)
print(
    "Weight matrix shape:",
    frozen_weight_matrix.shape,
)
print(
    "Primary BING matrix row range:",
    f"{query_bing_row_indices.min():,}"
    " to "
    f"{query_bing_row_indices.max():,}",
)
print(
    "Frozen weight norms:",
    dict(
        zip(
            PROGRAM_IDS,
            frozen_weight_norms,
        )
    ),
)
print(
    "Resistance-like multipliers:",
    dict(
        zip(
            PROGRAM_IDS,
            resistance_like_multiplier_vector.astype(int),
        )
    ),
)

Shared frozen query support: 1,495 genes
Weight matrix shape: (1495, 3)
Primary BING matrix row range: 2 to 10,149
Frozen weight norms: {'CONSENSUS_TX_01': np.float64(0.8507412531758485), 'CONSENSUS_TX_02': np.float64(0.8016572205527611), 'CONSENSUS_TX_03': np.float64(0.8130261461852406)}
Resistance-like multipliers: {'CONSENSUS_TX_01': np.int64(-1), 'CONSENSUS_TX_02': np.int64(1), 'CONSENSUS_TX_03': np.int64(1)}


In [34]:
# =============================================================================
# Generate coupled outcome-blind benchmark pseudo-program weights
# =============================================================================

query_stratum_row_indices = {
    int(stratum_id): (
        np.flatnonzero(
            coupled_query_weights[
                "stratum_id"
            ].to_numpy(dtype=np.int64)
            == int(stratum_id)
        )
    )
    for stratum_id in STRATUM_IDS
}


def generate_coupled_pseudo_weights(
    replicate_id,
    namespace=RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
):
    pseudo_weights = (
        frozen_weight_matrix.copy()
    )

    for stratum_id in STRATUM_IDS:
        row_indices = (
            query_stratum_row_indices[
                stratum_id
            ]
        )

        permutation = (
            phase7_uniform_permutation(
                n_positions=len(row_indices),
                namespace=namespace,
                replicate_id=replicate_id,
                stratum_id=stratum_id,
            )
        )

        pseudo_weights[
            row_indices,
            :,
        ] = frozen_weight_matrix[
            row_indices[permutation],
            :,
        ]

    return pseudo_weights


benchmark_pseudo_weights_0 = (
    generate_coupled_pseudo_weights(
        replicate_id=0
    )
)

benchmark_pseudo_weights_0_repeat = (
    generate_coupled_pseudo_weights(
        replicate_id=0
    )
)

if not np.array_equal(
    benchmark_pseudo_weights_0,
    benchmark_pseudo_weights_0_repeat,
):
    raise RuntimeError(
        "Benchmark pseudo-program replicate "
        "is not reproducible."
    )

for program_index, program_id in enumerate(
    PROGRAM_IDS
):
    np.testing.assert_array_equal(
        np.sort(
            benchmark_pseudo_weights_0[
                :,
                program_index,
            ]
        ),
        np.sort(
            frozen_weight_matrix[
                :,
                program_index,
            ]
        ),
    )

pseudo_weight_norms = np.linalg.norm(
    benchmark_pseudo_weights_0,
    axis=0,
)

np.testing.assert_allclose(
    pseudo_weight_norms,
    frozen_weight_norms,
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)

if np.array_equal(
    benchmark_pseudo_weights_0,
    frozen_weight_matrix,
):
    raise RuntimeError(
        "Benchmark replicate 0 unexpectedly produced "
        "the global identity assignment."
    )

print(
    "Benchmark pseudo-program generation: PASS"
)
print(
    "Namespace:",
    RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
)
print("Replicate:", 0)
print(
    "Coupled pseudo-weight matrix shape:",
    benchmark_pseudo_weights_0.shape,
)
print(
    "Weight norms preserved:",
    dict(
        zip(
            PROGRAM_IDS,
            pseudo_weight_norms,
        )
    ),
)

Benchmark pseudo-program generation: PASS
Namespace: 3
Replicate: 0
Coupled pseudo-weight matrix shape: (1495, 3)
Weight norms preserved: {'CONSENSUS_TX_01': np.float64(0.8507412531758487), 'CONSENSUS_TX_02': np.float64(0.8016572205527613), 'CONSENSUS_TX_03': np.float64(0.8130261461852405)}


In [35]:
# =============================================================================
# Define and validate the float64 pseudo-program cosine kernel
# =============================================================================

def score_pseudo_program_block(
    signature_block,
    weight_matrix,
):
    signature_block = np.asarray(
        signature_block,
        dtype=np.float64,
    )

    weight_matrix = np.asarray(
        weight_matrix,
        dtype=np.float64,
    )

    if signature_block.ndim != 2:
        raise ValueError(
            "Signature block must be genes × signatures."
        )

    if weight_matrix.shape != (
        signature_block.shape[0],
        len(PROGRAM_IDS),
    ):
        raise ValueError(
            "Weight matrix and signature block "
            "have incompatible shapes."
        )

    if not np.isfinite(signature_block).all():
        raise ValueError(
            "Signature block contains non-finite values."
        )

    signature_norms = np.linalg.norm(
        signature_block,
        axis=0,
    )

    weight_norms = np.linalg.norm(
        weight_matrix,
        axis=0,
    )

    numerators = (
        weight_matrix.T
        @ signature_block
    )

    denominators = (
        weight_norms[:, np.newaxis]
        * signature_norms[np.newaxis, :]
    )

    scores = np.full(
        numerators.shape,
        np.nan,
        dtype=np.float64,
    )

    evaluable = signature_norms > 0

    scores[
        :,
        evaluable,
    ] = (
        numerators[:, evaluable]
        / denominators[:, evaluable]
    )

    return scores.T, evaluable


# Independent small reference calculation.
toy_signature_block = np.array(
    [
        [0.5, -0.2, 0.7, 0.0],
        [-0.3, 0.4, 0.1, 0.0],
        [0.8, 0.6, -0.5, 0.0],
        [0.2, -0.7, 0.3, 0.0],
    ],
    dtype=np.float64,
)

toy_weight_matrix = np.array(
    [
        [0.4, -0.2, 0.1],
        [-0.1, 0.5, 0.3],
        [0.7, 0.2, -0.4],
        [-0.3, 0.1, 0.6],
    ],
    dtype=np.float64,
)

toy_scores, toy_evaluable = (
    score_pseudo_program_block(
        toy_signature_block,
        toy_weight_matrix,
    )
)

toy_reference = np.full(
    toy_scores.shape,
    np.nan,
    dtype=np.float64,
)

for signature_index in range(
    toy_signature_block.shape[1]
):
    z = toy_signature_block[
        :,
        signature_index,
    ]

    z_norm = np.sqrt(
        np.sum(z * z)
    )

    if z_norm == 0:
        continue

    for program_index in range(
        toy_weight_matrix.shape[1]
    ):
        w = toy_weight_matrix[
            :,
            program_index,
        ]

        toy_reference[
            signature_index,
            program_index,
        ] = (
            np.sum(w * z)
            / (
                np.sqrt(np.sum(w * w))
                * z_norm
            )
        )

np.testing.assert_allclose(
    toy_scores[
        toy_evaluable,
        :,
    ],
    toy_reference[
        toy_evaluable,
        :,
    ],
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)

if not np.array_equal(
    toy_evaluable,
    np.array(
        [True, True, True, False]
    ),
):
    raise RuntimeError(
        "Zero-norm signature evaluability "
        "was not handled as specified."
    )

if np.nanmax(
    np.abs(toy_scores)
) > 1 + NUMERICAL_ATOL:
    raise RuntimeError(
        "Cosine score exceeded its theoretical range."
    )

print(
    "Float64 cosine kernel reference validation: PASS"
)
print(
    "Toy evaluable signatures:",
    int(toy_evaluable.sum()),
    "/",
    len(toy_evaluable),
)
print(
    "Zero-norm signatures marked non-evaluable:",
    int((~toy_evaluable).sum()),
)

Float64 cosine kernel reference validation: PASS
Toy evaluable signatures: 3 / 4
Zero-norm signatures marked non-evaluable: 1


In [36]:
# =============================================================================
# Define the outcome-blind benchmark scoring path
# =============================================================================

BENCHMARK_SIGNATURE_BLOCK_SIZE = 4096


def score_benchmark_replicate(
    replicate_id,
):
    pseudo_weights = (
        generate_coupled_pseudo_weights(
            replicate_id=replicate_id,
            namespace=RNG_NAMESPACES[
                "outcome_blind_benchmark"
            ],
        )
    )

    n_signatures = hierarchy_counts[
        "signatures"
    ]

    signature_scores = np.empty(
        (
            n_signatures,
            len(PROGRAM_IDS),
        ),
        dtype=np.float64,
    )

    signature_evaluable = np.empty(
        n_signatures,
        dtype=bool,
    )

    for block_start in range(
        0,
        n_signatures,
        BENCHMARK_SIGNATURE_BLOCK_SIZE,
    ):
        block_stop = min(
            block_start
            + BENCHMARK_SIGNATURE_BLOCK_SIZE,
            n_signatures,
        )

        signature_block = np.asarray(
            primary_bing_matrix[
                query_bing_row_indices,
                block_start:block_stop,
            ],
            dtype=np.float64,
        )

        block_scores, block_evaluable = (
            score_pseudo_program_block(
                signature_block,
                pseudo_weights,
            )
        )

        signature_scores[
            block_start:block_stop,
            :,
        ] = block_scores

        signature_evaluable[
            block_start:block_stop
        ] = block_evaluable

    if not signature_evaluable.all():
        raise RuntimeError(
            "Outcome-blind benchmark encountered "
            "non-evaluable primary signatures."
        )

    raw_aggregation = aggregate_signature_scores(
        signature_scores
    )

    resistance_like_aggregation = {
        level: (
            scores
            * resistance_like_multiplier_vector[
                np.newaxis,
                :
            ]
        )
        for level, scores
        in raw_aggregation.items()
    }

    return {
        "replicate_id": int(replicate_id),
        "raw": raw_aggregation,
        "resistance_like": (
            resistance_like_aggregation
        ),
    }


print(
    "Outcome-blind benchmark scoring path: DEFINED"
)
print(
    "Signature block size:",
    BENCHMARK_SIGNATURE_BLOCK_SIZE,
)
print(
    "Namespace:",
    RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
)
print(
    "Full primary signatures per replicate:",
    f"{hierarchy_counts['signatures']:,}",
)
print(
    "No full benchmark replicate executed: YES"
)

Outcome-blind benchmark scoring path: DEFINED
Signature block size: 4096
Namespace: 3
Full primary signatures per replicate: 362,036
No full benchmark replicate executed: YES


In [37]:
# =============================================================================
# Smoke-test pseudo-program scoring on small primary-signature blocks
# =============================================================================

SMOKE_TEST_BLOCK_STARTS = (
    0,
    hierarchy_counts["signatures"]
    - BENCHMARK_SIGNATURE_BLOCK_SIZE,
)

smoke_pseudo_weights = (
    generate_coupled_pseudo_weights(
        replicate_id=0,
        namespace=RNG_NAMESPACES[
            "outcome_blind_benchmark"
        ],
    )
)

smoke_test_records = []

for block_start in SMOKE_TEST_BLOCK_STARTS:
    block_stop = min(
        block_start
        + BENCHMARK_SIGNATURE_BLOCK_SIZE,
        hierarchy_counts["signatures"],
    )

    signature_block = np.asarray(
        primary_bing_matrix[
            query_bing_row_indices,
            block_start:block_stop,
        ],
        dtype=np.float64,
    )

    block_scores, block_evaluable = (
        score_pseudo_program_block(
            signature_block,
            smoke_pseudo_weights,
        )
    )

    if not block_evaluable.all():
        raise RuntimeError(
            "Smoke test encountered a non-evaluable "
            "primary signature."
        )

    if not np.isfinite(block_scores).all():
        raise RuntimeError(
            "Smoke test produced non-finite cosine scores."
        )

    if (
        np.max(np.abs(block_scores))
        > 1 + NUMERICAL_ATOL
    ):
        raise RuntimeError(
            "Smoke-test cosine score exceeded "
            "the theoretical range."
        )

    smoke_test_records.append(
        {
            "block_start": int(block_start),
            "block_stop": int(block_stop),
            "signatures": int(
                block_stop - block_start
            ),
            "score_min": float(
                block_scores.min()
            ),
            "score_max": float(
                block_scores.max()
            ),
        }
    )

smoke_test_summary = pd.DataFrame(
    smoke_test_records
)

print("Pseudo-program scoring smoke test: PASS")
print(
    "Signatures tested:",
    f"{smoke_test_summary['signatures'].sum():,}",
)
print(
    "Full benchmark replicate executed: NO"
)

display(smoke_test_summary)

Pseudo-program scoring smoke test: PASS
Signatures tested: 8,192
Full benchmark replicate executed: NO


,block_start,block_stop,signatures,score_min,score_max
0,0,4096,4096,-0.116330,0.125419
1,357940,362036,4096,-0.119762,0.131841


In [38]:
# =============================================================================
# Validate process-memory measurement for benchmark
# =============================================================================

try:
    import psutil

    _benchmark_process = psutil.Process(
        os.getpid()
    )

    def current_working_set_bytes():
        return int(
            _benchmark_process.memory_info().rss
        )

    MEMORY_MEASUREMENT_BACKEND = "psutil"

except ImportError:
    from ctypes import wintypes

    class PROCESS_MEMORY_COUNTERS(
        ctypes.Structure
    ):
        _fields_ = [
            ("cb", wintypes.DWORD),
            ("PageFaultCount", wintypes.DWORD),
            (
                "PeakWorkingSetSize",
                ctypes.c_size_t,
            ),
            (
                "WorkingSetSize",
                ctypes.c_size_t,
            ),
            (
                "QuotaPeakPagedPoolUsage",
                ctypes.c_size_t,
            ),
            (
                "QuotaPagedPoolUsage",
                ctypes.c_size_t,
            ),
            (
                "QuotaPeakNonPagedPoolUsage",
                ctypes.c_size_t,
            ),
            (
                "QuotaNonPagedPoolUsage",
                ctypes.c_size_t,
            ),
            (
                "PagefileUsage",
                ctypes.c_size_t,
            ),
            (
                "PeakPagefileUsage",
                ctypes.c_size_t,
            ),
        ]

    _kernel32 = ctypes.WinDLL(
        "kernel32",
        use_last_error=True,
    )

    _psapi = ctypes.WinDLL(
        "psapi",
        use_last_error=True,
    )

    _kernel32.GetCurrentProcess.restype = (
        wintypes.HANDLE
    )

    _psapi.GetProcessMemoryInfo.argtypes = [
        wintypes.HANDLE,
        ctypes.POINTER(
            PROCESS_MEMORY_COUNTERS
        ),
        wintypes.DWORD,
    ]

    _psapi.GetProcessMemoryInfo.restype = (
        wintypes.BOOL
    )

    def current_working_set_bytes():
        counters = PROCESS_MEMORY_COUNTERS()
        counters.cb = ctypes.sizeof(
            counters
        )

        success = (
            _psapi.GetProcessMemoryInfo(
                _kernel32.GetCurrentProcess(),
                ctypes.byref(counters),
                counters.cb,
            )
        )

        if not success:
            error_code = (
                ctypes.get_last_error()
            )

            raise OSError(
                error_code,
                "GetProcessMemoryInfo failed",
            )

        return int(
            counters.WorkingSetSize
        )

    MEMORY_MEASUREMENT_BACKEND = (
        "Windows PSAPI"
    )


test_working_set_bytes = (
    current_working_set_bytes()
)

if test_working_set_bytes <= 0:
    raise RuntimeError(
        "Invalid process-memory measurement."
    )

print(
    "Process-memory measurement: PASS"
)
print(
    "Backend:",
    MEMORY_MEASUREMENT_BACKEND,
)
print(
    "Current working set:",
    f"{test_working_set_bytes / 1024**3:.2f} GiB",
)

Process-memory measurement: PASS
Backend: psutil
Current working set: 10.64 GiB


In [ ]:
# =============================================================================
# Benchmark one complete namespace-3 pseudo-program replicate
# =============================================================================

BENCHMARK_REPLICATE_ID = 0
MEMORY_SAMPLE_INTERVAL_SECONDS = 0.05

memory_samples = []
memory_stop_event = threading.Event()


def sample_benchmark_memory():
    while not memory_stop_event.is_set():
        memory_samples.append(
            current_working_set_bytes()
        )

        memory_stop_event.wait(
            MEMORY_SAMPLE_INTERVAL_SECONDS
        )


baseline_memory_bytes = (
    current_working_set_bytes()
)

memory_samples.append(
    baseline_memory_bytes
)

memory_thread = threading.Thread(
    target=sample_benchmark_memory,
    daemon=True,
)

benchmark_start = time.perf_counter()

memory_thread.start()

try:
    benchmark_replicate_result = (
        score_benchmark_replicate(
            replicate_id=BENCHMARK_REPLICATE_ID
        )
    )
finally:
    memory_stop_event.set()
    memory_thread.join()

benchmark_elapsed_seconds = (
    time.perf_counter()
    - benchmark_start
)

final_memory_bytes = (
    current_working_set_bytes()
)

memory_samples.append(
    final_memory_bytes
)

benchmark_peak_working_set_bytes = max(
    memory_samples
)

benchmark_peak_increment_bytes = max(
    0,
    benchmark_peak_working_set_bytes
    - baseline_memory_bytes,
)

benchmark_perturbagen_scores = (
    benchmark_replicate_result[
        "resistance_like"
    ]["perturbagen"]
)

if benchmark_perturbagen_scores.shape != (
    7914,
    len(PROGRAM_IDS),
):
    raise RuntimeError(
        "Unexpected perturbagen-score shape "
        "for the complete benchmark replicate."
    )

if not np.isfinite(
    benchmark_perturbagen_scores
).all():
    raise RuntimeError(
        "Complete benchmark replicate produced "
        "non-finite perturbagen scores."
    )

if (
    benchmark_peak_working_set_bytes
    / 1024**3
    >
    benchmark_resource_envelope[
        "resource_limits"
    ]["maximum_ram_gib"]
):
    raise RuntimeError(
        "Complete benchmark replicate exceeded "
        "the frozen RAM envelope."
    )

print(
    "FIRST COMPLETE BENCHMARK REPLICATE: PASS"
)
print(
    "Memory backend:",
    MEMORY_MEASUREMENT_BACKEND,
)
print(
    "Namespace:",
    RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
)
print(
    "Replicate:",
    BENCHMARK_REPLICATE_ID,
)
print(
    "Primary signatures processed:",
    f"{hierarchy_counts['signatures']:,}",
)
print(
    "Perturbagens produced:",
    f"{benchmark_perturbagen_scores.shape[0]:,}",
)
print(
    "Elapsed:",
    f"{benchmark_elapsed_seconds:.2f} seconds",
)
print(
    "Baseline working set:",
    f"{baseline_memory_bytes / 1024**3:.2f} GiB",
)
print(
    "Peak working set:",
    f"{benchmark_peak_working_set_bytes / 1024**3:.2f} GiB",
)
print(
    "Peak increment:",
    f"{benchmark_peak_increment_bytes / 1024**3:.2f} GiB",
)
print(
    "Observed real-program connectivity computed:",
    "NO",
)

FIRST COMPLETE BENCHMARK REPLICATE: PASS
Memory backend: psutil
Namespace: 3
Replicate: 0
Primary signatures processed: 362,036
Perturbagens produced: 7,914
Elapsed: 25.42 seconds
Baseline working set: 10.64 GiB
Peak working set: 10.91 GiB
Peak increment: 0.27 GiB
Observed real-program connectivity computed: NO


In [40]:
# =============================================================================
# Prepare perturbagen-to-signature membership for active-set benchmarking
# =============================================================================

perturbagen_labels = np.asarray(
    perturbagen_index,
    dtype=object,
)

if len(perturbagen_labels) != 7914:
    raise RuntimeError(
        "Unexpected perturbagen-index size."
    )

signature_indices_by_perturbagen = {
    int(perturbagen_id_value): np.flatnonzero(
        perturbagen_id == perturbagen_id_value
    ).astype(np.int64)
    for perturbagen_id_value in range(
        len(perturbagen_labels)
    )
}

signature_counts_by_perturbagen = np.array(
    [
        len(
            signature_indices_by_perturbagen[
                perturbagen_id_value
            ]
        )
        for perturbagen_id_value in range(
            len(perturbagen_labels)
        )
    ],
    dtype=np.int64,
)

if signature_counts_by_perturbagen.sum() != (
    hierarchy_counts["signatures"]
):
    raise RuntimeError(
        "Perturbagen-to-signature mapping does not "
        "cover the complete primary signature universe."
    )

if np.any(
    signature_counts_by_perturbagen <= 0
):
    raise RuntimeError(
        "At least one perturbagen has no primary signatures."
    )

print(
    "Perturbagen → signature membership: PASS"
)
print(
    "Perturbagens:",
    f"{len(perturbagen_labels):,}",
)
print(
    "Primary signatures covered:",
    f"{signature_counts_by_perturbagen.sum():,}",
)
print(
    "Signatures per perturbagen:",
)
print(
    "  min:",
    int(signature_counts_by_perturbagen.min()),
)
print(
    "  median:",
    float(
        np.median(
            signature_counts_by_perturbagen
        )
    ),
)
print(
    "  max:",
    int(signature_counts_by_perturbagen.max()),
)

Perturbagen → signature membership: PASS
Perturbagens: 7,914
Primary signatures covered: 362,036
Signatures per perturbagen:
  min: 5
  median: 15.0
  max: 2361


In [41]:
# =============================================================================
# Freeze deterministic active-hypothesis benchmark cases
# =============================================================================

perturbagen_key_table = (
    pd.DataFrame(
        {
            "perturbagen_id": perturbagen_id,
            "pert_id": (
                primary_signature_metadata[
                    "pert_id"
                ]
                .astype(str)
                .to_numpy()
            ),
        }
    )
    .drop_duplicates()
    .sort_values(
        "perturbagen_id",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(perturbagen_key_table) != 7914:
    raise RuntimeError(
        "Unexpected perturbagen-key table size."
    )

if not perturbagen_key_table[
    "perturbagen_id"
].is_unique:
    raise RuntimeError(
        "Perturbagen integer IDs are not unique."
    )

if not perturbagen_key_table[
    "pert_id"
].is_unique:
    raise RuntimeError(
        "Frozen pert_id values are not unique."
    )

perturbagen_key_table[
    "signature_count"
] = (
    signature_counts_by_perturbagen[
        perturbagen_key_table[
            "perturbagen_id"
        ].to_numpy(dtype=np.int64)
    ]
)

program_order_table = pd.DataFrame(
    {
        "program_id": PROGRAM_IDS,
        "program_index": np.arange(
            len(PROGRAM_IDS),
            dtype=np.int64,
        ),
    }
)

benchmark_hypothesis_catalog = (
    perturbagen_key_table
    .merge(
        program_order_table,
        how="cross",
    )
    .sort_values(
        [
            "signature_count",
            "pert_id",
            "program_index",
        ],
        ascending=[
            False,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

benchmark_hypothesis_catalog[
    "computational_burden_rank"
] = np.arange(
    len(benchmark_hypothesis_catalog),
    dtype=np.int64,
)

if len(benchmark_hypothesis_catalog) != 23742:
    raise RuntimeError(
        "Unexpected benchmark hypothesis count."
    )

benchmark_case_labels = (
    benchmark_resource_envelope[
        "benchmark_design_freeze"
    ]["active_set_case_labels"]
)

benchmark_case_sizes = (
    benchmark_resource_envelope[
        "benchmark_design_freeze"
    ]["active_set_cases"]
)

benchmark_case_blocks = []

for case_label, active_count in zip(
    benchmark_case_labels,
    benchmark_case_sizes,
):
    case_block = (
        benchmark_hypothesis_catalog
        .head(int(active_count))
        .copy()
    )

    case_block[
        "case_label"
    ] = case_label

    case_block[
        "active_hypothesis_rank"
    ] = np.arange(
        len(case_block),
        dtype=np.int64,
    )

    benchmark_case_blocks.append(
        case_block
    )

benchmark_active_set_manifest = (
    pd.concat(
        benchmark_case_blocks,
        ignore_index=True,
    )[
        [
            "case_label",
            "active_hypothesis_rank",
            "program_id",
            "program_index",
            "perturbagen_id",
            "pert_id",
            "signature_count",
            "computational_burden_rank",
        ]
    ]
)

BENCHMARK_ACTIVE_SET_MANIFEST_PATH = (
    Paths.perturbational_hypotheses
    / "700_benchmark_active_set_manifest.csv"
)

if BENCHMARK_ACTIVE_SET_MANIFEST_PATH.exists():
    persisted_active_set_manifest = (
        pd.read_csv(
            BENCHMARK_ACTIVE_SET_MANIFEST_PATH
        )
    )

    pd.testing.assert_frame_equal(
        persisted_active_set_manifest,
        benchmark_active_set_manifest,
        check_dtype=False,
        check_exact=True,
    )

    active_set_manifest_action = (
        "REUSED_IDENTICAL"
    )

else:
    benchmark_active_set_manifest.to_csv(
        BENCHMARK_ACTIVE_SET_MANIFEST_PATH,
        index=False,
    )

    active_set_manifest_action = "CREATED"


benchmark_case_summary = (
    benchmark_active_set_manifest
    .groupby(
        "case_label",
        sort=False,
        as_index=False,
    )
    .agg(
        active_hypotheses=(
            "perturbagen_id",
            "size",
        ),
        active_perturbagens=(
            "perturbagen_id",
            "nunique",
        ),
        signature_hypothesis_load=(
            "signature_count",
            "sum",
        ),
        max_signatures_per_hypothesis=(
            "signature_count",
            "max",
        ),
        median_signatures_per_hypothesis=(
            "signature_count",
            "median",
        ),
    )
)

program_case_summary = (
    benchmark_active_set_manifest
    .groupby(
        [
            "case_label",
            "program_id",
        ],
        sort=False,
        as_index=False,
    )
    .agg(
        active_hypotheses=(
            "perturbagen_id",
            "size",
        ),
        signature_load=(
            "signature_count",
            "sum",
        ),
    )
)

print(
    "Deterministic active-set benchmark design: FROZEN"
)
print(
    "Manifest action:",
    active_set_manifest_action,
)
print(
    "Selection rule:",
    "descending signature burden; "
    "pert_id ascending; program order ascending",
)
print(
    "Manifest path:",
    project_relative_path(
        BENCHMARK_ACTIVE_SET_MANIFEST_PATH
    ),
)

display(benchmark_case_summary)
display(program_case_summary)

Deterministic active-set benchmark design: FROZEN
Manifest action: REUSED_IDENTICAL
Selection rule: descending signature burden; pert_id ascending; program order ascending
Manifest path: data/processed/perturbational_hypotheses/700_benchmark_active_set_manifest.csv


,case_label,active_hypotheses,active_perturbagens,signature_hypothesis_load,max_signatures_per_hypothesis,median_signatures_per_hypothesis
0,maximum_100_percent,23742,7914,1086108,2361,15.0
1,reduced_50_percent,11871,3957,972312,2361,65.0
2,reduced_10_percent_ceiling,2375,792,478493,2361,149.0


,case_label,program_id,active_hypotheses,signature_load
0,maximum_100_percent,CONSENSUS_TX_01,7914,362036
1,maximum_100_percent,CONSENSUS_TX_02,7914,362036
2,maximum_100_percent,CONSENSUS_TX_03,7914,362036
3,reduced_50_percent,CONSENSUS_TX_01,3957,324104
4,reduced_50_percent,CONSENSUS_TX_02,3957,324104
5,reduced_50_percent,CONSENSUS_TX_03,3957,324104
6,reduced_10_percent_ceiling,CONSENSUS_TX_01,792,159534
7,reduced_10_percent_ceiling,CONSENSUS_TX_02,792,159534
8,reduced_10_percent_ceiling,CONSENSUS_TX_03,791,159425


In [42]:
# =============================================================================
# Prepare and validate active-set-aware hierarchical aggregation
# =============================================================================

# Direct perturbagen membership for every aggregation level.
condition_to_perturbagen = build_parent_map(
    condition_id,
    perturbagen_id,
    len(condition_index),
    "Condition -> perturbagen",
)

timepoint_to_perturbagen = build_parent_map(
    timepoint_id,
    perturbagen_id,
    len(timepoint_index),
    "Timepoint -> perturbagen",
)

cell_line_to_perturbagen_direct = build_parent_map(
    cell_line_id,
    perturbagen_id,
    len(cell_line_index),
    "Cell line -> perturbagen",
)

level_to_perturbagen = {
    "condition": condition_to_perturbagen,
    "timepoint": timepoint_to_perturbagen,
    "cell_line": cell_line_to_perturbagen_direct,
    "lineage": lineage_to_perturbagen,
    "perturbagen": np.arange(
        hierarchy_counts["perturbagens"],
        dtype=np.int32,
    ),
}


# -------------------------------------------------------------------------
# Materialize exact active hypothesis masks and signature unions
# -------------------------------------------------------------------------

benchmark_case_execution_plans = {}

for case_label in benchmark_case_labels:
    case_manifest = (
        benchmark_active_set_manifest.loc[
            benchmark_active_set_manifest[
                "case_label"
            ].eq(case_label)
        ]
    )

    active_hypothesis_mask = np.zeros(
        (
            hierarchy_counts["perturbagens"],
            len(PROGRAM_IDS),
        ),
        dtype=bool,
    )

    active_hypothesis_mask[
        case_manifest[
            "perturbagen_id"
        ].to_numpy(dtype=np.int64),
        case_manifest[
            "program_index"
        ].to_numpy(dtype=np.int64),
    ] = True

    active_perturbagen_union = (
        active_hypothesis_mask.any(axis=1)
    )

    active_signature_union = (
        active_perturbagen_union[
            perturbagen_id
        ]
    )

    active_signature_indices = (
        np.flatnonzero(
            active_signature_union
        ).astype(np.int64)
    )

    benchmark_case_execution_plans[
        case_label
    ] = {
        "active_hypothesis_mask": (
            active_hypothesis_mask
        ),
        "active_perturbagen_union": (
            active_perturbagen_union
        ),
        "active_signature_indices": (
            active_signature_indices
        ),
    }


# -------------------------------------------------------------------------
# Active-aware median reduction
# -------------------------------------------------------------------------

def apply_active_median_reduction(
    values,
    plan,
    n_parents,
    parent_to_perturbagen,
    active_hypothesis_mask,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    if values.ndim != 2:
        raise ValueError(
            "Active median reduction expects a 2D array."
        )

    reduced = np.full(
        (
            n_parents,
            values.shape[1],
        ),
        np.nan,
        dtype=np.float64,
    )

    for block in plan:
        parent_ids = block[
            "parent_ids"
        ].astype(
            np.int64,
            copy=False,
        )

        child_indices = block[
            "child_indices"
        ].astype(
            np.int64,
            copy=False,
        )

        parent_perturbagens = (
            parent_to_perturbagen[
                parent_ids
            ]
        )

        for program_index in range(
            len(PROGRAM_IDS)
        ):
            active_parent_mask = (
                active_hypothesis_mask[
                    parent_perturbagens,
                    program_index,
                ]
            )

            if not active_parent_mask.any():
                continue

            selected_parent_ids = (
                parent_ids[
                    active_parent_mask
                ]
            )

            selected_child_indices = (
                child_indices[
                    active_parent_mask,
                    :
                ]
            )

            selected_values = values[
                selected_child_indices,
                program_index,
            ]

            if not np.isfinite(
                selected_values
            ).all():
                raise RuntimeError(
                    "Active aggregation encountered "
                    "non-finite child values."
                )

            reduced[
                selected_parent_ids,
                program_index,
            ] = np.median(
                selected_values,
                axis=1,
            )

    return reduced


def aggregate_active_signature_scores(
    signature_scores,
    signature_indices,
    active_hypothesis_mask,
):
    signature_scores = np.asarray(
        signature_scores,
        dtype=np.float64,
    )

    signature_indices = np.asarray(
        signature_indices,
        dtype=np.int64,
    )

    if signature_scores.shape != (
        len(signature_indices),
        len(PROGRAM_IDS),
    ):
        raise ValueError(
            "Active signature scores and indices "
            "have incompatible shapes."
        )

    full_signature_scores = np.full(
        (
            hierarchy_counts["signatures"],
            len(PROGRAM_IDS),
        ),
        np.nan,
        dtype=np.float64,
    )

    full_signature_scores[
        signature_indices,
        :,
    ] = signature_scores

    selected_signature_perturbagens = (
        perturbagen_id[
            signature_indices
        ]
    )

    for program_index in range(
        len(PROGRAM_IDS)
    ):
        inactive_selected = ~(
            active_hypothesis_mask[
                selected_signature_perturbagens,
                program_index,
            ]
        )

        full_signature_scores[
            signature_indices[
                inactive_selected
            ],
            program_index,
        ] = np.nan

    condition_scores = (
        apply_active_median_reduction(
            full_signature_scores,
            *aggregation_plans[
                "signature_to_condition"
            ],
            parent_to_perturbagen=(
                condition_to_perturbagen
            ),
            active_hypothesis_mask=(
                active_hypothesis_mask
            ),
        )
    )

    timepoint_scores = (
        apply_active_median_reduction(
            condition_scores,
            *aggregation_plans[
                "condition_to_timepoint"
            ],
            parent_to_perturbagen=(
                timepoint_to_perturbagen
            ),
            active_hypothesis_mask=(
                active_hypothesis_mask
            ),
        )
    )

    cell_line_scores = (
        apply_active_median_reduction(
            timepoint_scores,
            *aggregation_plans[
                "timepoint_to_cell_line"
            ],
            parent_to_perturbagen=(
                cell_line_to_perturbagen_direct
            ),
            active_hypothesis_mask=(
                active_hypothesis_mask
            ),
        )
    )

    lineage_scores = (
        apply_active_median_reduction(
            cell_line_scores,
            *aggregation_plans[
                "cell_line_to_lineage"
            ],
            parent_to_perturbagen=(
                lineage_to_perturbagen
            ),
            active_hypothesis_mask=(
                active_hypothesis_mask
            ),
        )
    )

    perturbagen_scores = (
        apply_active_median_reduction(
            lineage_scores,
            *aggregation_plans[
                "lineage_to_perturbagen"
            ],
            parent_to_perturbagen=np.arange(
                hierarchy_counts[
                    "perturbagens"
                ],
                dtype=np.int32,
            ),
            active_hypothesis_mask=(
                active_hypothesis_mask
            ),
        )
    )

    return {
        "condition": condition_scores,
        "timepoint": timepoint_scores,
        "cell_line": cell_line_scores,
        "lineage": lineage_scores,
        "perturbagen": perturbagen_scores,
    }


# -------------------------------------------------------------------------
# Validate active-set plumbing against the already validated full hierarchy
# -------------------------------------------------------------------------

VALIDATION_CASE_LABEL = (
    "reduced_10_percent_ceiling"
)

validation_plan = (
    benchmark_case_execution_plans[
        VALIDATION_CASE_LABEL
    ]
)

validation_signature_indices = (
    validation_plan[
        "active_signature_indices"
    ]
)

validation_active_aggregation = (
    aggregate_active_signature_scores(
        synthetic_signature_scores[
            validation_signature_indices,
            :
        ],
        validation_signature_indices,
        validation_plan[
            "active_hypothesis_mask"
        ],
    )
)

for level in (
    "condition",
    "timepoint",
    "cell_line",
    "lineage",
    "perturbagen",
):
    level_perturbagens = (
        level_to_perturbagen[level]
    )

    observed = (
        validation_active_aggregation[
            level
        ]
    )

    reference = (
        synthetic_aggregation[
            level
        ]
    )

    for program_index in range(
        len(PROGRAM_IDS)
    ):
        active_rows = (
            validation_plan[
                "active_hypothesis_mask"
            ][
                level_perturbagens,
                program_index,
            ]
        )

        np.testing.assert_allclose(
            observed[
                active_rows,
                program_index,
            ],
            reference[
                active_rows,
                program_index,
            ],
            rtol=NUMERICAL_RTOL,
            atol=NUMERICAL_ATOL,
        )

        if not np.isnan(
            observed[
                ~active_rows,
                program_index,
            ]
        ).all():
            raise RuntimeError(
                "Inactive aggregation rows "
                "were not retained as NaN."
            )


case_execution_summary = []

for case_label, plan in (
    benchmark_case_execution_plans.items()
):
    case_execution_summary.append(
        {
            "case_label": case_label,
            "active_hypotheses": int(
                plan[
                    "active_hypothesis_mask"
                ].sum()
            ),
            "active_perturbagens_union": int(
                plan[
                    "active_perturbagen_union"
                ].sum()
            ),
            "signatures_to_score": int(
                len(
                    plan[
                        "active_signature_indices"
                    ]
                )
            ),
            "signature_fraction": (
                len(
                    plan[
                        "active_signature_indices"
                    ]
                )
                / hierarchy_counts[
                    "signatures"
                ]
            ),
        }
    )

case_execution_summary = pd.DataFrame(
    case_execution_summary
)

print(
    "Active-set-aware aggregation validation: PASS"
)
print(
    "Synthetic reference case:",
    VALIDATION_CASE_LABEL,
)

display(case_execution_summary)

Active-set-aware aggregation validation: PASS
Synthetic reference case: reduced_10_percent_ceiling


,case_label,active_hypotheses,active_perturbagens_union,signatures_to_score,signature_fraction
0,maximum_100_percent,23742,7914,362036,1.000000
1,reduced_50_percent,11871,3957,324104,0.895226
2,reduced_10_percent_ceiling,2375,792,159534,0.440658


In [43]:
# =============================================================================
# Define active-set-aware benchmark scoring path
# =============================================================================

def score_active_benchmark_replicate(
    replicate_id,
    case_label,
):
    if case_label not in (
        benchmark_case_execution_plans
    ):
        raise KeyError(
            f"Unknown benchmark case: {case_label}"
        )

    case_plan = (
        benchmark_case_execution_plans[
            case_label
        ]
    )

    active_hypothesis_mask = (
        case_plan[
            "active_hypothesis_mask"
        ]
    )

    active_signature_indices = (
        case_plan[
            "active_signature_indices"
        ]
    )

    pseudo_weights = (
        generate_coupled_pseudo_weights(
            replicate_id=replicate_id,
            namespace=RNG_NAMESPACES[
                "outcome_blind_benchmark"
            ],
        )
    )

    n_active_signatures = len(
        active_signature_indices
    )

    active_signature_scores = np.empty(
        (
            n_active_signatures,
            len(PROGRAM_IDS),
        ),
        dtype=np.float64,
    )

    active_signature_evaluable = np.empty(
        n_active_signatures,
        dtype=bool,
    )

    for local_start in range(
        0,
        n_active_signatures,
        BENCHMARK_SIGNATURE_BLOCK_SIZE,
    ):
        local_stop = min(
            local_start
            + BENCHMARK_SIGNATURE_BLOCK_SIZE,
            n_active_signatures,
        )

        global_signature_indices = (
            active_signature_indices[
                local_start:local_stop
            ]
        )

        signature_block = np.asarray(
            primary_bing_matrix[
                np.ix_(
                    query_bing_row_indices,
                    global_signature_indices,
                )
            ],
            dtype=np.float64,
        )

        block_scores, block_evaluable = (
            score_pseudo_program_block(
                signature_block,
                pseudo_weights,
            )
        )

        active_signature_scores[
            local_start:local_stop,
            :,
        ] = block_scores

        active_signature_evaluable[
            local_start:local_stop
        ] = block_evaluable

    if not active_signature_evaluable.all():
        raise RuntimeError(
            "Active benchmark encountered "
            "non-evaluable signatures."
        )

    raw_aggregation = (
        aggregate_active_signature_scores(
            active_signature_scores,
            active_signature_indices,
            active_hypothesis_mask,
        )
    )

    resistance_like_aggregation = {
        level: (
            scores
            * resistance_like_multiplier_vector[
                np.newaxis,
                :
            ]
        )
        for level, scores
        in raw_aggregation.items()
    }

    return {
        "replicate_id": int(
            replicate_id
        ),
        "case_label": case_label,
        "active_hypothesis_mask": (
            active_hypothesis_mask
        ),
        "active_signature_count": int(
            n_active_signatures
        ),
        "raw": raw_aggregation,
        "resistance_like": (
            resistance_like_aggregation
        ),
    }


print(
    "Active-set-aware benchmark scoring path: DEFINED"
)

for case_label in benchmark_case_labels:
    case_plan = (
        benchmark_case_execution_plans[
            case_label
        ]
    )

    print(
        f"{case_label}: "
        f"{case_plan['active_hypothesis_mask'].sum():,} "
        "active hypotheses; "
        f"{len(case_plan['active_signature_indices']):,} "
        "signatures"
    )

print(
    "Block extraction:",
    "direct gene × signature indexed subset",
)
print(
    "No reduced benchmark case executed: YES"
)

Active-set-aware benchmark scoring path: DEFINED
maximum_100_percent: 23,742 active hypotheses; 362,036 signatures
reduced_50_percent: 11,871 active hypotheses; 324,104 signatures
reduced_10_percent_ceiling: 2,375 active hypotheses; 159,534 signatures
Block extraction: direct gene × signature indexed subset
No reduced benchmark case executed: YES


In [44]:
# =============================================================================
# Benchmark and validate reduced 10-percent active set
# =============================================================================

REDUCED_CASE_LABEL = (
    "reduced_10_percent_ceiling"
)

REDUCED_BENCHMARK_REPLICATE_ID = 0
MEMORY_SAMPLE_INTERVAL_SECONDS = 0.05

reduced_memory_samples = []
reduced_memory_stop_event = threading.Event()


def sample_reduced_benchmark_memory():
    while not reduced_memory_stop_event.is_set():
        reduced_memory_samples.append(
            current_working_set_bytes()
        )

        reduced_memory_stop_event.wait(
            MEMORY_SAMPLE_INTERVAL_SECONDS
        )


reduced_baseline_memory_bytes = (
    current_working_set_bytes()
)

reduced_memory_samples.append(
    reduced_baseline_memory_bytes
)

reduced_memory_thread = threading.Thread(
    target=sample_reduced_benchmark_memory,
    daemon=True,
)

reduced_benchmark_start = time.perf_counter()

reduced_memory_thread.start()

try:
    reduced_benchmark_result = (
        score_active_benchmark_replicate(
            replicate_id=(
                REDUCED_BENCHMARK_REPLICATE_ID
            ),
            case_label=REDUCED_CASE_LABEL,
        )
    )
finally:
    reduced_memory_stop_event.set()
    reduced_memory_thread.join()

reduced_benchmark_elapsed_seconds = (
    time.perf_counter()
    - reduced_benchmark_start
)

reduced_final_memory_bytes = (
    current_working_set_bytes()
)

reduced_memory_samples.append(
    reduced_final_memory_bytes
)

reduced_peak_working_set_bytes = max(
    reduced_memory_samples
)

reduced_peak_increment_bytes = max(
    0,
    reduced_peak_working_set_bytes
    - reduced_baseline_memory_bytes,
)


# -------------------------------------------------------------------------
# Exact validation against the already computed full replicate 0
# -------------------------------------------------------------------------

reduced_scores = (
    reduced_benchmark_result[
        "resistance_like"
    ]["perturbagen"]
)

full_reference_scores = (
    benchmark_replicate_result[
        "resistance_like"
    ]["perturbagen"]
)

active_hypothesis_mask = (
    reduced_benchmark_result[
        "active_hypothesis_mask"
    ]
)

np.testing.assert_allclose(
    reduced_scores[
        active_hypothesis_mask
    ],
    full_reference_scores[
        active_hypothesis_mask
    ],
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)

if not np.isnan(
    reduced_scores[
        ~active_hypothesis_mask
    ]
).all():
    raise RuntimeError(
        "Inactive perturbagen × program hypotheses "
        "were not retained as NaN."
    )

if (
    reduced_peak_working_set_bytes
    / 1024**3
    >
    benchmark_resource_envelope[
        "resource_limits"
    ]["maximum_ram_gib"]
):
    raise RuntimeError(
        "Reduced benchmark exceeded "
        "the frozen RAM envelope."
    )


print(
    "REDUCED 10% ACTIVE-SET BENCHMARK: PASS"
)
print(
    "Case:",
    REDUCED_CASE_LABEL,
)
print(
    "Namespace:",
    RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
)
print(
    "Replicate:",
    REDUCED_BENCHMARK_REPLICATE_ID,
)
print(
    "Active hypotheses:",
    f"{active_hypothesis_mask.sum():,}",
)
print(
    "Signatures scored:",
    f"{reduced_benchmark_result['active_signature_count']:,}",
)
print(
    "Active scores match full replicate:",
    "PASS",
)
print(
    "Elapsed:",
    f"{reduced_benchmark_elapsed_seconds:.2f} seconds",
)
print(
    "Baseline working set:",
    f"{reduced_baseline_memory_bytes / 1024**3:.2f} GiB",
)
print(
    "Peak working set:",
    f"{reduced_peak_working_set_bytes / 1024**3:.2f} GiB",
)
print(
    "Peak increment:",
    f"{reduced_peak_increment_bytes / 1024**3:.2f} GiB",
)

REDUCED 10% ACTIVE-SET BENCHMARK: PASS
Case: reduced_10_percent_ceiling
Namespace: 3
Replicate: 0
Active hypotheses: 2,375
Signatures scored: 159,534
Active scores match full replicate: PASS
Elapsed: 6.84 seconds
Baseline working set: 10.75 GiB
Peak working set: 10.80 GiB
Peak increment: 0.05 GiB


In [45]:
# =============================================================================
# Benchmark and validate reduced 50-percent active set
# =============================================================================

REDUCED_50_CASE_LABEL = (
    "reduced_50_percent"
)

REDUCED_50_BENCHMARK_REPLICATE_ID = 0
MEMORY_SAMPLE_INTERVAL_SECONDS = 0.05

reduced_50_memory_samples = []
reduced_50_memory_stop_event = threading.Event()


def sample_reduced_50_benchmark_memory():
    while not reduced_50_memory_stop_event.is_set():
        reduced_50_memory_samples.append(
            current_working_set_bytes()
        )

        reduced_50_memory_stop_event.wait(
            MEMORY_SAMPLE_INTERVAL_SECONDS
        )


reduced_50_baseline_memory_bytes = (
    current_working_set_bytes()
)

reduced_50_memory_samples.append(
    reduced_50_baseline_memory_bytes
)

reduced_50_memory_thread = threading.Thread(
    target=sample_reduced_50_benchmark_memory,
    daemon=True,
)

reduced_50_benchmark_start = time.perf_counter()

reduced_50_memory_thread.start()

try:
    reduced_50_benchmark_result = (
        score_active_benchmark_replicate(
            replicate_id=(
                REDUCED_50_BENCHMARK_REPLICATE_ID
            ),
            case_label=REDUCED_50_CASE_LABEL,
        )
    )
finally:
    reduced_50_memory_stop_event.set()
    reduced_50_memory_thread.join()

reduced_50_benchmark_elapsed_seconds = (
    time.perf_counter()
    - reduced_50_benchmark_start
)

reduced_50_final_memory_bytes = (
    current_working_set_bytes()
)

reduced_50_memory_samples.append(
    reduced_50_final_memory_bytes
)

reduced_50_peak_working_set_bytes = max(
    reduced_50_memory_samples
)

reduced_50_peak_increment_bytes = max(
    0,
    reduced_50_peak_working_set_bytes
    - reduced_50_baseline_memory_bytes,
)


# -------------------------------------------------------------------------
# Exact validation against the already computed full replicate 0
# -------------------------------------------------------------------------

reduced_50_scores = (
    reduced_50_benchmark_result[
        "resistance_like"
    ]["perturbagen"]
)

active_50_hypothesis_mask = (
    reduced_50_benchmark_result[
        "active_hypothesis_mask"
    ]
)

np.testing.assert_allclose(
    reduced_50_scores[
        active_50_hypothesis_mask
    ],
    full_reference_scores[
        active_50_hypothesis_mask
    ],
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)

if not np.isnan(
    reduced_50_scores[
        ~active_50_hypothesis_mask
    ]
).all():
    raise RuntimeError(
        "Inactive perturbagen × program hypotheses "
        "were not retained as NaN."
    )

if (
    reduced_50_peak_working_set_bytes
    / 1024**3
    >
    benchmark_resource_envelope[
        "resource_limits"
    ]["maximum_ram_gib"]
):
    raise RuntimeError(
        "Reduced 50% benchmark exceeded "
        "the frozen RAM envelope."
    )


print(
    "REDUCED 50% ACTIVE-SET BENCHMARK: PASS"
)
print(
    "Case:",
    REDUCED_50_CASE_LABEL,
)
print(
    "Namespace:",
    RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
)
print(
    "Replicate:",
    REDUCED_50_BENCHMARK_REPLICATE_ID,
)
print(
    "Active hypotheses:",
    f"{active_50_hypothesis_mask.sum():,}",
)
print(
    "Signatures scored:",
    f"{reduced_50_benchmark_result['active_signature_count']:,}",
)
print(
    "Active scores match full replicate:",
    "PASS",
)
print(
    "Elapsed:",
    f"{reduced_50_benchmark_elapsed_seconds:.2f} seconds",
)
print(
    "Baseline working set:",
    f"{reduced_50_baseline_memory_bytes / 1024**3:.2f} GiB",
)
print(
    "Peak working set:",
    f"{reduced_50_peak_working_set_bytes / 1024**3:.2f} GiB",
)
print(
    "Peak increment:",
    f"{reduced_50_peak_increment_bytes / 1024**3:.2f} GiB",
)

REDUCED 50% ACTIVE-SET BENCHMARK: PASS
Case: reduced_50_percent
Namespace: 3
Replicate: 0
Active hypotheses: 11,871
Signatures scored: 324,104
Active scores match full replicate: PASS
Elapsed: 13.63 seconds
Baseline working set: 10.71 GiB
Peak working set: 10.82 GiB
Peak increment: 0.12 GiB


In [46]:
# =============================================================================
# Validate atomic checkpoint and deterministic restart
# =============================================================================

CHECKPOINT_SCHEMA_VERSION = (
    "phase7_mc_checkpoint_v1"
)

CHECKPOINT_TEST_PATH = (
    Paths.perturbational_hypotheses
    / "700_restart_validation_checkpoint.npz"
)

CHECKPOINT_TEST_TEMP_PATH = (
    CHECKPOINT_TEST_PATH.with_suffix(
        ".npz.tmp"
    )
)


def write_phase7_checkpoint(
    path,
    next_replicate_id,
    mc_g,
    mc_L,
    active_mask,
):
    path = Path(path)

    temp_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temp_path.open("wb") as handle:
        np.savez(
            handle,
            schema_version=np.array(
                [CHECKPOINT_SCHEMA_VERSION]
            ),
            next_replicate_id=np.array(
                [next_replicate_id],
                dtype=np.int64,
            ),
            mc_g=np.asarray(
                mc_g,
                dtype=np.int32,
            ),
            mc_L=np.asarray(
                mc_L,
                dtype=np.int32,
            ),
            active_mask=np.asarray(
                active_mask,
                dtype=bool,
            ),
        )

        handle.flush()
        os.fsync(
            handle.fileno()
        )

    os.replace(
        temp_path,
        path,
    )


def read_phase7_checkpoint(
    path,
):
    with np.load(
        path,
        allow_pickle=False,
    ) as checkpoint:
        schema_version = str(
            checkpoint[
                "schema_version"
            ][0]
        )

        if (
            schema_version
            != CHECKPOINT_SCHEMA_VERSION
        ):
            raise RuntimeError(
                "Unexpected checkpoint schema."
            )

        return {
            "next_replicate_id": int(
                checkpoint[
                    "next_replicate_id"
                ][0]
            ),
            "mc_g": checkpoint[
                "mc_g"
            ].copy(),
            "mc_L": checkpoint[
                "mc_L"
            ].copy(),
            "active_mask": checkpoint[
                "active_mask"
            ].copy(),
        }


# -------------------------------------------------------------------------
# Outcome-blind synthetic sequential state
# -------------------------------------------------------------------------

checkpoint_test_active_mask = (
    benchmark_case_execution_plans[
        "reduced_10_percent_ceiling"
    ][
        "active_hypothesis_mask"
    ]
    .reshape(-1)
    .copy()
)

n_checkpoint_hypotheses = len(
    checkpoint_test_active_mask
)

CHECKPOINT_TEST_NEXT_REPLICATE_ID = 1000

checkpoint_test_g = np.where(
    checkpoint_test_active_mask,
    np.arange(
        n_checkpoint_hypotheses,
        dtype=np.int32,
    )
    % 20,
    20,
).astype(np.int32)

checkpoint_test_L = np.where(
    checkpoint_test_active_mask,
    CHECKPOINT_TEST_NEXT_REPLICATE_ID,
    (
        20
        + (
            np.arange(
                n_checkpoint_hypotheses,
                dtype=np.int32,
            )
            % 980
        )
    ),
).astype(np.int32)


# RNG reconstruction expected after restart.
next_pseudo_weights_before_restart = (
    generate_coupled_pseudo_weights(
        replicate_id=(
            CHECKPOINT_TEST_NEXT_REPLICATE_ID
        ),
        namespace=RNG_NAMESPACES[
            "outcome_blind_benchmark"
        ],
    )
)


# -------------------------------------------------------------------------
# Atomic checkpoint write
# -------------------------------------------------------------------------

for candidate_path in (
    CHECKPOINT_TEST_PATH,
    CHECKPOINT_TEST_TEMP_PATH,
):
    if candidate_path.exists():
        candidate_path.unlink()

checkpoint_write_start = (
    time.perf_counter()
)

write_phase7_checkpoint(
    path=CHECKPOINT_TEST_PATH,
    next_replicate_id=(
        CHECKPOINT_TEST_NEXT_REPLICATE_ID
    ),
    mc_g=checkpoint_test_g,
    mc_L=checkpoint_test_L,
    active_mask=checkpoint_test_active_mask,
)

checkpoint_write_seconds = (
    time.perf_counter()
    - checkpoint_write_start
)


# -------------------------------------------------------------------------
# Simulated restart
# -------------------------------------------------------------------------

checkpoint_read_start = (
    time.perf_counter()
)

restored_checkpoint = (
    read_phase7_checkpoint(
        CHECKPOINT_TEST_PATH
    )
)

checkpoint_read_seconds = (
    time.perf_counter()
    - checkpoint_read_start
)

if (
    restored_checkpoint[
        "next_replicate_id"
    ]
    != CHECKPOINT_TEST_NEXT_REPLICATE_ID
):
    raise RuntimeError(
        "Restart replicate boundary "
        "was not preserved."
    )

np.testing.assert_array_equal(
    restored_checkpoint["mc_g"],
    checkpoint_test_g,
)

np.testing.assert_array_equal(
    restored_checkpoint["mc_L"],
    checkpoint_test_L,
)

np.testing.assert_array_equal(
    restored_checkpoint[
        "active_mask"
    ],
    checkpoint_test_active_mask,
)


# Reconstruct the next RNG draw using coordinates only.
next_pseudo_weights_after_restart = (
    generate_coupled_pseudo_weights(
        replicate_id=(
            restored_checkpoint[
                "next_replicate_id"
            ]
        ),
        namespace=RNG_NAMESPACES[
            "outcome_blind_benchmark"
        ],
    )
)

np.testing.assert_array_equal(
    next_pseudo_weights_after_restart,
    next_pseudo_weights_before_restart,
)


# -------------------------------------------------------------------------
# Cleanup test checkpoint
# -------------------------------------------------------------------------

checkpoint_size_bytes = (
    CHECKPOINT_TEST_PATH.stat().st_size
)

CHECKPOINT_TEST_PATH.unlink()

if CHECKPOINT_TEST_TEMP_PATH.exists():
    CHECKPOINT_TEST_TEMP_PATH.unlink()


print(
    "ATOMIC CHECKPOINT / RESTART VALIDATION: PASS"
)
print(
    "Checkpoint boundary:",
    CHECKPOINT_TEST_NEXT_REPLICATE_ID,
)
print(
    "Hypothesis states:",
    f"{n_checkpoint_hypotheses:,}",
)
print(
    "Active hypotheses:",
    f"{checkpoint_test_active_mask.sum():,}",
)
print(
    "RNG reconstruction after restart:",
    "PASS",
)
print(
    "Checkpoint write:",
    f"{checkpoint_write_seconds:.4f} seconds",
)
print(
    "Checkpoint read:",
    f"{checkpoint_read_seconds:.4f} seconds",
)
print(
    "Checkpoint size:",
    f"{checkpoint_size_bytes / 1024:.1f} KiB",
)
print(
    "Temporary validation checkpoint removed:",
    not CHECKPOINT_TEST_PATH.exists(),
)

ATOMIC CHECKPOINT / RESTART VALIDATION: PASS
Checkpoint boundary: 1000
Hypothesis states: 23,742
Active hypotheses: 2,375
RNG reconstruction after restart: PASS
Checkpoint write: 0.0198 seconds
Checkpoint read: 0.0262 seconds
Checkpoint size: 210.0 KiB
Temporary validation checkpoint removed: True


In [47]:
# =============================================================================
# Benchmark active-set-aware maximum 100-percent case
# =============================================================================

MAX_CASE_LABEL = (
    "maximum_100_percent"
)

MAX_BENCHMARK_REPLICATE_ID = 0
MEMORY_SAMPLE_INTERVAL_SECONDS = 0.05

max_memory_samples = []
max_memory_stop_event = threading.Event()


def sample_max_benchmark_memory():
    while not max_memory_stop_event.is_set():
        max_memory_samples.append(
            current_working_set_bytes()
        )

        max_memory_stop_event.wait(
            MEMORY_SAMPLE_INTERVAL_SECONDS
        )


max_baseline_memory_bytes = (
    current_working_set_bytes()
)

max_memory_samples.append(
    max_baseline_memory_bytes
)

max_memory_thread = threading.Thread(
    target=sample_max_benchmark_memory,
    daemon=True,
)

max_benchmark_start = time.perf_counter()

max_memory_thread.start()

try:
    max_active_benchmark_result = (
        score_active_benchmark_replicate(
            replicate_id=(
                MAX_BENCHMARK_REPLICATE_ID
            ),
            case_label=MAX_CASE_LABEL,
        )
    )
finally:
    max_memory_stop_event.set()
    max_memory_thread.join()

max_benchmark_elapsed_seconds = (
    time.perf_counter()
    - max_benchmark_start
)

max_final_memory_bytes = (
    current_working_set_bytes()
)

max_memory_samples.append(
    max_final_memory_bytes
)

max_peak_working_set_bytes = max(
    max_memory_samples
)

max_peak_increment_bytes = max(
    0,
    max_peak_working_set_bytes
    - max_baseline_memory_bytes,
)


# -------------------------------------------------------------------------
# Exact validation against the original full replicate 0
# -------------------------------------------------------------------------

max_active_scores = (
    max_active_benchmark_result[
        "resistance_like"
    ]["perturbagen"]
)

max_active_hypothesis_mask = (
    max_active_benchmark_result[
        "active_hypothesis_mask"
    ]
)

if not max_active_hypothesis_mask.all():
    raise RuntimeError(
        "Maximum-active benchmark does not contain "
        "all frozen hypotheses."
    )

np.testing.assert_allclose(
    max_active_scores,
    full_reference_scores,
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)

if (
    max_peak_working_set_bytes
    / 1024**3
    >
    benchmark_resource_envelope[
        "resource_limits"
    ]["maximum_ram_gib"]
):
    raise RuntimeError(
        "Maximum-active benchmark exceeded "
        "the frozen RAM envelope."
    )


print(
    "MAXIMUM 100% ACTIVE-SET BENCHMARK: PASS"
)
print(
    "Case:",
    MAX_CASE_LABEL,
)
print(
    "Namespace:",
    RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
)
print(
    "Replicate:",
    MAX_BENCHMARK_REPLICATE_ID,
)
print(
    "Active hypotheses:",
    f"{max_active_hypothesis_mask.sum():,}",
)
print(
    "Signatures scored:",
    f"{max_active_benchmark_result['active_signature_count']:,}",
)
print(
    "Active-set scores match original full replicate:",
    "PASS",
)
print(
    "Elapsed:",
    f"{max_benchmark_elapsed_seconds:.2f} seconds",
)
print(
    "Baseline working set:",
    f"{max_baseline_memory_bytes / 1024**3:.2f} GiB",
)
print(
    "Peak working set:",
    f"{max_peak_working_set_bytes / 1024**3:.2f} GiB",
)
print(
    "Peak increment:",
    f"{max_peak_increment_bytes / 1024**3:.2f} GiB",
)

MAXIMUM 100% ACTIVE-SET BENCHMARK: PASS
Case: maximum_100_percent
Namespace: 3
Replicate: 0
Active hypotheses: 23,742
Signatures scored: 362,036
Active-set scores match original full replicate: PASS
Elapsed: 17.42 seconds
Baseline working set: 10.71 GiB
Peak working set: 10.81 GiB
Peak increment: 0.10 GiB


In [48]:
# =============================================================================
# Freeze and benchmark sparse-tail active sets
# =============================================================================

TAIL_CASE_SPECS = {
    "tail_1_percent": 238,
    "tail_0_1_percent": 24,
    "tail_0_01_percent": 3,
}

TAIL_MANIFEST_PATH = (
    Paths.perturbational_hypotheses
    / "700_benchmark_sparse_tail_manifest.csv"
)

# -------------------------------------------------------------------------
# Freeze deterministic burden-heavy sparse-tail cases BEFORE timing
# -------------------------------------------------------------------------

tail_case_blocks = []

for case_label, active_count in (
    TAIL_CASE_SPECS.items()
):
    case_block = (
        benchmark_hypothesis_catalog
        .head(active_count)
        .copy()
    )

    case_block["case_label"] = case_label
    case_block["active_hypothesis_rank"] = (
        np.arange(
            len(case_block),
            dtype=np.int64,
        )
    )

    tail_case_blocks.append(
        case_block
    )

benchmark_sparse_tail_manifest = (
    pd.concat(
        tail_case_blocks,
        ignore_index=True,
    )[
        [
            "case_label",
            "active_hypothesis_rank",
            "program_id",
            "program_index",
            "perturbagen_id",
            "pert_id",
            "signature_count",
            "computational_burden_rank",
        ]
    ]
)

if TAIL_MANIFEST_PATH.exists():
    persisted_tail_manifest = pd.read_csv(
        TAIL_MANIFEST_PATH
    )

    pd.testing.assert_frame_equal(
        persisted_tail_manifest,
        benchmark_sparse_tail_manifest,
        check_dtype=False,
        check_exact=True,
    )

    tail_manifest_action = "REUSED_IDENTICAL"

else:
    benchmark_sparse_tail_manifest.to_csv(
        TAIL_MANIFEST_PATH,
        index=False,
    )

    tail_manifest_action = "CREATED"


# -------------------------------------------------------------------------
# Build execution plans
# -------------------------------------------------------------------------

for case_label in TAIL_CASE_SPECS:
    case_manifest = (
        benchmark_sparse_tail_manifest.loc[
            benchmark_sparse_tail_manifest[
                "case_label"
            ].eq(case_label)
        ]
    )

    active_hypothesis_mask = np.zeros(
        (
            hierarchy_counts["perturbagens"],
            len(PROGRAM_IDS),
        ),
        dtype=bool,
    )

    active_hypothesis_mask[
        case_manifest[
            "perturbagen_id"
        ].to_numpy(dtype=np.int64),
        case_manifest[
            "program_index"
        ].to_numpy(dtype=np.int64),
    ] = True

    active_perturbagen_union = (
        active_hypothesis_mask.any(axis=1)
    )

    active_signature_indices = (
        np.flatnonzero(
            active_perturbagen_union[
                perturbagen_id
            ]
        )
        .astype(np.int64)
    )

    benchmark_case_execution_plans[
        case_label
    ] = {
        "active_hypothesis_mask": (
            active_hypothesis_mask
        ),
        "active_perturbagen_union": (
            active_perturbagen_union
        ),
        "active_signature_indices": (
            active_signature_indices
        ),
    }


print(
    "Sparse-tail benchmark cases: FROZEN"
)
print(
    "Manifest action:",
    tail_manifest_action,
)
print(
    "Selection rule:",
    "descending signature burden; "
    "pert_id ascending; program order ascending",
)


# -------------------------------------------------------------------------
# Benchmark each frozen sparse-tail case
# -------------------------------------------------------------------------

tail_benchmark_records = []

for case_label in TAIL_CASE_SPECS:
    case_plan = (
        benchmark_case_execution_plans[
            case_label
        ]
    )

    memory_samples_tail = []
    stop_event_tail = threading.Event()

    def sample_tail_memory():
        while not stop_event_tail.is_set():
            memory_samples_tail.append(
                current_working_set_bytes()
            )
            stop_event_tail.wait(0.05)

    baseline_bytes = (
        current_working_set_bytes()
    )
    memory_samples_tail.append(
        baseline_bytes
    )

    memory_thread_tail = threading.Thread(
        target=sample_tail_memory,
        daemon=True,
    )

    start_time = time.perf_counter()
    memory_thread_tail.start()

    try:
        tail_result = (
            score_active_benchmark_replicate(
                replicate_id=0,
                case_label=case_label,
            )
        )
    finally:
        stop_event_tail.set()
        memory_thread_tail.join()

    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )

    final_bytes = (
        current_working_set_bytes()
    )
    memory_samples_tail.append(
        final_bytes
    )

    peak_bytes = max(
        memory_samples_tail
    )

    active_mask = (
        tail_result[
            "active_hypothesis_mask"
        ]
    )

    tail_scores = (
        tail_result[
            "resistance_like"
        ]["perturbagen"]
    )

    np.testing.assert_allclose(
        tail_scores[
            active_mask
        ],
        full_reference_scores[
            active_mask
        ],
        rtol=NUMERICAL_RTOL,
        atol=NUMERICAL_ATOL,
    )

    if not np.isnan(
        tail_scores[
            ~active_mask
        ]
    ).all():
        raise RuntimeError(
            f"{case_label}: inactive hypotheses "
            "were not retained as NaN."
        )

    tail_benchmark_records.append(
        {
            "case_label": case_label,
            "active_hypotheses": int(
                active_mask.sum()
            ),
            "active_perturbagens_union": int(
                case_plan[
                    "active_perturbagen_union"
                ].sum()
            ),
            "signatures_scored": int(
                tail_result[
                    "active_signature_count"
                ]
            ),
            "elapsed_seconds": float(
                elapsed_seconds
            ),
            "baseline_gib": (
                baseline_bytes / 1024**3
            ),
            "peak_gib": (
                peak_bytes / 1024**3
            ),
            "peak_increment_gib": (
                max(
                    0,
                    peak_bytes - baseline_bytes,
                )
                / 1024**3
            ),
        }
    )

tail_benchmark_summary = pd.DataFrame(
    tail_benchmark_records
)

if (
    tail_benchmark_summary[
        "peak_gib"
    ].max()
    >
    benchmark_resource_envelope[
        "resource_limits"
    ]["maximum_ram_gib"]
):
    raise RuntimeError(
        "Sparse-tail benchmark exceeded "
        "the frozen RAM envelope."
    )

print(
    "SPARSE-TAIL ACTIVE-SET BENCHMARKS: PASS"
)
print(
    "All active scores match full replicate: PASS"
)

display(
    tail_benchmark_summary
)

Sparse-tail benchmark cases: FROZEN
Manifest action: REUSED_IDENTICAL
Selection rule: descending signature burden; pert_id ascending; program order ascending
SPARSE-TAIL ACTIVE-SET BENCHMARKS: PASS
All active scores match full replicate: PASS


,case_label,active_hypotheses,active_perturbagens_union,signatures_scored,elapsed_seconds,baseline_gib,peak_gib,peak_increment_gib
0,tail_1_percent,238,80,46668,4.719462,10.341713,10.418907,0.077194
1,tail_0_1_percent,24,8,11490,1.099315,9.967823,10.081917,0.114094
2,tail_0_01_percent,3,1,2361,0.347273,9.967964,10.003525,0.035561


In [49]:
# =============================================================================
# Project candidate Bmax under the frozen outcome-blind benchmark
# =============================================================================

SEQUENTIAL_EXCEEDANCE_TARGET = 20
PRIMARY_HYPOTHESIS_COUNT = 23742
PRIMARY_BY_Q = 0.05

# Measured active-set timings, all using namespace 3 and replicate 0.
measured_case_timings = {
    1.0: float(
        max_benchmark_elapsed_seconds
    ),
    0.5: float(
        reduced_50_benchmark_elapsed_seconds
    ),
    0.1: float(
        reduced_benchmark_elapsed_seconds
    ),
    0.01: float(
        tail_benchmark_summary.loc[
            tail_benchmark_summary[
                "case_label"
            ].eq("tail_1_percent"),
            "elapsed_seconds",
        ].iloc[0]
    ),
    0.001: float(
        tail_benchmark_summary.loc[
            tail_benchmark_summary[
                "case_label"
            ].eq("tail_0_1_percent"),
            "elapsed_seconds",
        ].iloc[0]
    ),
    0.0001: float(
        tail_benchmark_summary.loc[
            tail_benchmark_summary[
                "case_label"
            ].eq("tail_0_01_percent"),
            "elapsed_seconds",
        ].iloc[0]
    ),
}

# -------------------------------------------------------------------------
# Outcome-blind active-set projection
#
# Under null calibration, the expected fraction of hypotheses still below
# h exceedances after L draws scales approximately as h / (L + 1).
#
# We deliberately use an upper step-envelope:
#
#   draws     active-set timing used
#   0-39      100%
#   40-199     50%
#   200-1999   10%
#   2k-19999    1%
#   20k-199999  0.1%
#   >=200k      0.01%
#
# The measured cases themselves were burden-heavy with respect to signature
# count. No observed connectivity is used.
# -------------------------------------------------------------------------

projection_segments = [
    (0, 40, 1.0),
    (40, 200, 0.5),
    (200, 2000, 0.1),
    (2000, 20000, 0.01),
    (20000, 200000, 0.001),
    (200000, None, 0.0001),
]


def project_candidate_runtime_seconds(
    bmax,
):
    scoring_seconds = 0.0
    segment_records = []

    for (
        segment_start,
        segment_stop,
        active_fraction,
    ) in projection_segments:

        if segment_start >= bmax:
            continue

        effective_stop = (
            bmax
            if segment_stop is None
            else min(
                segment_stop,
                bmax,
            )
        )

        if effective_stop <= segment_start:
            continue

        replicate_count = (
            effective_stop
            - segment_start
        )

        seconds_per_replicate = (
            measured_case_timings[
                active_fraction
            ]
        )

        segment_seconds = (
            replicate_count
            * seconds_per_replicate
        )

        scoring_seconds += (
            segment_seconds
        )

        segment_records.append(
            {
                "start_replicate": (
                    segment_start
                ),
                "stop_replicate": (
                    effective_stop
                ),
                "replicates": (
                    replicate_count
                ),
                "active_fraction": (
                    active_fraction
                ),
                "seconds_per_replicate": (
                    seconds_per_replicate
                ),
                "projected_hours": (
                    segment_seconds
                    / 3600
                ),
            }
        )

    checkpoint_interval = (
        benchmark_resource_envelope[
            "checkpoint_restart_policy"
        ][
            "checkpoint_interval_replicates"
        ]
    )

    checkpoint_count = (
        bmax
        // checkpoint_interval
    )

    checkpoint_seconds = (
        checkpoint_count
        * checkpoint_write_seconds
    )

    return (
        scoring_seconds,
        checkpoint_seconds,
        segment_records,
    )


harmonic_m = np.sum(
    1.0
    / np.arange(
        1,
        PRIMARY_HYPOTHESIS_COUNT + 1,
        dtype=np.float64,
    )
)

measured_peak_gib = max(
    max_peak_working_set_bytes / 1024**3,
    reduced_50_peak_working_set_bytes / 1024**3,
    reduced_peak_working_set_bytes / 1024**3,
    tail_benchmark_summary[
        "peak_gib"
    ].max(),
)

runtime_limit_hours = (
    benchmark_resource_envelope[
        "resource_limits"
    ][
        "maximum_projected_primary_wall_clock_hours"
    ]
)

ram_limit_gib = (
    benchmark_resource_envelope[
        "resource_limits"
    ][
        "maximum_ram_gib"
    ]
)

candidate_records = []
projection_detail_records = []

for bmax in (
    benchmark_resource_envelope[
        "benchmark_design_freeze"
    ]["candidate_bmax_ladder"]
):
    (
        scoring_seconds,
        checkpoint_seconds,
        segment_records,
    ) = project_candidate_runtime_seconds(
        int(bmax)
    )

    projected_seconds = (
        scoring_seconds
        + checkpoint_seconds
    )

    minimum_mc_p = (
        1.0
        / (int(bmax) + 1)
    )

    minimum_by_rank_for_resolution = int(
        np.ceil(
            (
                minimum_mc_p
                * PRIMARY_HYPOTHESIS_COUNT
                * harmonic_m
            )
            / PRIMARY_BY_Q
        )
    )

    fits_runtime = (
        projected_seconds / 3600
        <= runtime_limit_hours
    )

    fits_ram = (
        measured_peak_gib
        <= ram_limit_gib
    )

    candidate_records.append(
        {
            "candidate_bmax": int(bmax),
            "projected_scoring_hours": (
                scoring_seconds / 3600
            ),
            "projected_checkpoint_hours": (
                checkpoint_seconds / 3600
            ),
            "projected_hours_before_other_overhead": (
                projected_seconds / 3600
            ),
            "runtime_limit_hours": (
                runtime_limit_hours
            ),
            "measured_peak_gib": (
                measured_peak_gib
            ),
            "ram_limit_gib": (
                ram_limit_gib
            ),
            "minimum_attainable_mc_p": (
                minimum_mc_p
            ),
            "minimum_BY_rank_at_mc_floor": (
                minimum_by_rank_for_resolution
            ),
            "fits_runtime": fits_runtime,
            "fits_ram": fits_ram,
            "fits_frozen_envelope": (
                fits_runtime
                and fits_ram
            ),
        }
    )

    for record in segment_records:
        projection_detail_records.append(
            {
                "candidate_bmax": int(bmax),
                **record,
            }
        )


bmax_projection_summary = pd.DataFrame(
    candidate_records
)

bmax_projection_detail = pd.DataFrame(
    projection_detail_records
)

eligible_bmax = (
    bmax_projection_summary.loc[
        bmax_projection_summary[
            "fits_frozen_envelope"
        ],
        "candidate_bmax",
    ]
    .tolist()
)

FINAL_BMAX = (
    max(eligible_bmax)
    if eligible_bmax
    else None
)

print(
    "OUTCOME-BLIND BMAX PROJECTION: COMPLETE"
)
print(
    "Projection model:",
    "upper step-envelope of null-calibrated "
    "h/(L+1) active-set decay",
)
print(
    "Exceedance target h:",
    SEQUENTIAL_EXCEEDANCE_TARGET,
)
print(
    "Measured peak RAM:",
    f"{measured_peak_gib:.2f} GiB",
)
print(
    "Frozen runtime limit:",
    f"{runtime_limit_hours:.2f} hours",
)
print(
    "Selected Bmax:",
    (
        f"{FINAL_BMAX:,}"
        if FINAL_BMAX is not None
        else "NONE — current implementation does not "
             "satisfy the frozen resource envelope"
    ),
)

display(
    bmax_projection_summary
)

display(
    bmax_projection_detail
)

OUTCOME-BLIND BMAX PROJECTION: COMPLETE
Projection model: upper step-envelope of null-calibrated h/(L+1) active-set decay
Exceedance target h: 20
Measured peak RAM: 10.82 GiB
Frozen runtime limit: 24.00 hours
Selected Bmax: NONE — current implementation does not satisfy the frozen resource envelope


,candidate_bmax,projected_scoring_hours,projected_checkpoint_hours,projected_hours_before_other_overhead,runtime_limit_hours,measured_peak_gib,ram_limit_gib,minimum_attainable_mc_p,minimum_BY_rank_at_mc_floor,fits_runtime,fits_ram,fits_frozen_envelope
0,500000,111.720621,0.002754,111.723375,24.0,10.823158,12.0,0.000002,11,False,True,False
1,250000,87.604454,0.001377,87.605831,24.0,10.823158,12.0,0.000004,21,False,True,False
2,100000,52.244682,0.000551,52.245233,24.0,10.823158,12.0,0.000010,51,False,True,False


,candidate_bmax,start_replicate,stop_replicate,replicates,active_fraction,seconds_per_replicate,projected_hours
0,500000,0,40,40,1.0000,17.422726,0.193586
1,500000,40,200,160,0.5000,13.627324,0.605659
2,500000,200,2000,1800,0.1000,6.837794,3.418897
3,500000,2000,20000,18000,0.0100,4.719462,23.597310
4,500000,20000,200000,180000,0.0010,1.099315,54.965770
5,500000,200000,500000,300000,0.0001,0.347273,28.939400
6,250000,0,40,40,1.0000,17.422726,0.193586
7,250000,40,200,160,0.5000,13.627324,0.605659
8,250000,200,2000,1800,0.1000,6.837794,3.418897
9,250000,2000,20000,18000,0.0100,4.719462,23.597310


In [50]:
# =============================================================================
# Optimize and benchmark sparse terminal-only null aggregation
# =============================================================================

def compact_group_median(
    values,
    parent_ids,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    parent_ids = np.asarray(
        parent_ids,
        dtype=np.int64,
    )

    if values.ndim != 2:
        raise ValueError(
            "Compact median reduction expects a 2D array."
        )

    if len(values) != len(parent_ids):
        raise ValueError(
            "Values and parent IDs have incompatible lengths."
        )

    order = np.argsort(
        parent_ids,
        kind="stable",
    )

    sorted_parent_ids = parent_ids[
        order
    ]

    unique_parents, starts, counts = np.unique(
        sorted_parent_ids,
        return_index=True,
        return_counts=True,
    )

    reduced = np.empty(
        (
            len(unique_parents),
            values.shape[1],
        ),
        dtype=np.float64,
    )

    for group_size in np.unique(
        counts
    ):
        parent_positions = np.flatnonzero(
            counts == group_size
        )

        child_positions = order[
            starts[
                parent_positions
            ][:, np.newaxis]
            + np.arange(
                group_size,
                dtype=np.int64,
            )
        ]

        reduced[
            parent_positions,
            :,
        ] = np.median(
            values[
                child_positions,
                :
            ],
            axis=1,
        )

    return (
        reduced,
        unique_parents.astype(
            np.int64,
            copy=False,
        ),
    )


def aggregate_sparse_to_perturbagen(
    signature_scores,
    signature_indices,
    active_hypothesis_mask,
):
    signature_scores = np.asarray(
        signature_scores,
        dtype=np.float64,
    )

    signature_indices = np.asarray(
        signature_indices,
        dtype=np.int64,
    )

    if signature_scores.shape != (
        len(signature_indices),
        len(PROGRAM_IDS),
    ):
        raise ValueError(
            "Sparse signature scores and indices "
            "have incompatible shapes."
        )

    perturbagen_scores = np.full(
        (
            hierarchy_counts[
                "perturbagens"
            ],
            len(PROGRAM_IDS),
        ),
        np.nan,
        dtype=np.float64,
    )

    signature_perturbagens = (
        perturbagen_id[
            signature_indices
        ]
    )

    active_perturbagens = np.flatnonzero(
        active_hypothesis_mask.any(
            axis=1
        )
    )

    for active_perturbagen in (
        active_perturbagens
    ):
        local_rows = np.flatnonzero(
            signature_perturbagens
            == active_perturbagen
        )

        if len(local_rows) == 0:
            raise RuntimeError(
                "Active perturbagen has no "
                "selected signatures."
            )

        local_signature_indices = (
            signature_indices[
                local_rows
            ]
        )

        local_scores = (
            signature_scores[
                local_rows,
                :
            ]
        )

        condition_scores, condition_ids_local = (
            compact_group_median(
                local_scores,
                condition_id[
                    local_signature_indices
                ],
            )
        )

        timepoint_scores, timepoint_ids_local = (
            compact_group_median(
                condition_scores,
                condition_to_timepoint[
                    condition_ids_local
                ],
            )
        )

        cell_scores, cell_ids_local = (
            compact_group_median(
                timepoint_scores,
                timepoint_to_cell_line[
                    timepoint_ids_local
                ],
            )
        )

        lineage_scores, lineage_ids_local = (
            compact_group_median(
                cell_scores,
                cell_line_to_lineage[
                    cell_ids_local
                ],
            )
        )

        (
            final_perturbagen_scores,
            final_perturbagen_ids,
        ) = compact_group_median(
            lineage_scores,
            lineage_to_perturbagen[
                lineage_ids_local
            ],
        )

        if (
            len(final_perturbagen_ids) != 1
            or int(
                final_perturbagen_ids[0]
            )
            != int(active_perturbagen)
        ):
            raise RuntimeError(
                "Sparse hierarchy crossed "
                "perturbagen boundaries."
            )

        active_programs = (
            active_hypothesis_mask[
                active_perturbagen,
                :
            ]
        )

        perturbagen_scores[
            active_perturbagen,
            active_programs,
        ] = final_perturbagen_scores[
            0,
            active_programs,
        ]

    return perturbagen_scores


def score_sparse_benchmark_replicate(
    replicate_id,
    case_label,
):
    case_plan = (
        benchmark_case_execution_plans[
            case_label
        ]
    )

    active_hypothesis_mask = (
        case_plan[
            "active_hypothesis_mask"
        ]
    )

    active_signature_indices = (
        case_plan[
            "active_signature_indices"
        ]
    )

    pseudo_weights = (
        generate_coupled_pseudo_weights(
            replicate_id=replicate_id,
            namespace=RNG_NAMESPACES[
                "outcome_blind_benchmark"
            ],
        )
    )

    active_signature_scores = np.empty(
        (
            len(active_signature_indices),
            len(PROGRAM_IDS),
        ),
        dtype=np.float64,
    )

    for local_start in range(
        0,
        len(active_signature_indices),
        BENCHMARK_SIGNATURE_BLOCK_SIZE,
    ):
        local_stop = min(
            local_start
            + BENCHMARK_SIGNATURE_BLOCK_SIZE,
            len(active_signature_indices),
        )

        global_indices = (
            active_signature_indices[
                local_start:local_stop
            ]
        )

        signature_block = np.asarray(
            primary_bing_matrix[
                np.ix_(
                    query_bing_row_indices,
                    global_indices,
                )
            ],
            dtype=np.float64,
        )

        block_scores, block_evaluable = (
            score_pseudo_program_block(
                signature_block,
                pseudo_weights,
            )
        )

        if not block_evaluable.all():
            raise RuntimeError(
                "Sparse benchmark encountered "
                "non-evaluable signatures."
            )

        active_signature_scores[
            local_start:local_stop,
            :
        ] = block_scores

    raw_perturbagen_scores = (
        aggregate_sparse_to_perturbagen(
            active_signature_scores,
            active_signature_indices,
            active_hypothesis_mask,
        )
    )

    resistance_like_scores = (
        raw_perturbagen_scores
        * resistance_like_multiplier_vector[
            np.newaxis,
            :
        ]
    )

    return {
        "case_label": case_label,
        "active_hypothesis_mask": (
            active_hypothesis_mask
        ),
        "active_signature_count": int(
            len(active_signature_indices)
        ),
        "resistance_like_perturbagen_scores": (
            resistance_like_scores
        ),
    }


# -------------------------------------------------------------------------
# Benchmark all three frozen sparse-tail cases
# -------------------------------------------------------------------------

sparse_optimized_records = []

for case_label in (
    "tail_1_percent",
    "tail_0_1_percent",
    "tail_0_01_percent",
):
    memory_samples_sparse = []
    stop_event_sparse = threading.Event()

    def sample_sparse_memory():
        while not stop_event_sparse.is_set():
            memory_samples_sparse.append(
                current_working_set_bytes()
            )
            stop_event_sparse.wait(0.05)

    baseline_bytes = (
        current_working_set_bytes()
    )

    memory_samples_sparse.append(
        baseline_bytes
    )

    memory_thread_sparse = threading.Thread(
        target=sample_sparse_memory,
        daemon=True,
    )

    start_time = time.perf_counter()

    memory_thread_sparse.start()

    try:
        sparse_result = (
            score_sparse_benchmark_replicate(
                replicate_id=0,
                case_label=case_label,
            )
        )
    finally:
        stop_event_sparse.set()
        memory_thread_sparse.join()

    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )

    final_bytes = (
        current_working_set_bytes()
    )

    memory_samples_sparse.append(
        final_bytes
    )

    peak_bytes = max(
        memory_samples_sparse
    )

    active_mask = (
        sparse_result[
            "active_hypothesis_mask"
        ]
    )

    sparse_scores = (
        sparse_result[
            "resistance_like_perturbagen_scores"
        ]
    )

    np.testing.assert_allclose(
        sparse_scores[
            active_mask
        ],
        full_reference_scores[
            active_mask
        ],
        rtol=NUMERICAL_RTOL,
        atol=NUMERICAL_ATOL,
    )

    if not np.isnan(
        sparse_scores[
            ~active_mask
        ]
    ).all():
        raise RuntimeError(
            f"{case_label}: inactive hypotheses "
            "were not retained as NaN."
        )

    sparse_optimized_records.append(
        {
            "case_label": case_label,
            "active_hypotheses": int(
                active_mask.sum()
            ),
            "active_perturbagens": int(
                active_mask.any(
                    axis=1
                ).sum()
            ),
            "signatures_scored": (
                sparse_result[
                    "active_signature_count"
                ]
            ),
            "elapsed_seconds": float(
                elapsed_seconds
            ),
            "peak_gib": (
                peak_bytes / 1024**3
            ),
        }
    )


sparse_optimized_summary = pd.DataFrame(
    sparse_optimized_records
)

previous_tail_timings = (
    tail_benchmark_summary[
        [
            "case_label",
            "elapsed_seconds",
        ]
    ]
    .rename(
        columns={
            "elapsed_seconds": (
                "previous_seconds"
            )
        }
    )
)

sparse_optimized_summary = (
    sparse_optimized_summary
    .merge(
        previous_tail_timings,
        on="case_label",
        how="left",
        validate="one_to_one",
    )
)

sparse_optimized_summary[
    "speedup"
] = (
    sparse_optimized_summary[
        "previous_seconds"
    ]
    / sparse_optimized_summary[
        "elapsed_seconds"
    ]
)

print(
    "SPARSE TERMINAL-ONLY NULL AGGREGATION: PASS"
)
print(
    "All active scores match full replicate: PASS"
)

display(
    sparse_optimized_summary
)

SPARSE TERMINAL-ONLY NULL AGGREGATION: PASS
All active scores match full replicate: PASS


,case_label,active_hypotheses,active_perturbagens,signatures_scored,elapsed_seconds,peak_gib,previous_seconds,speedup
0,tail_1_percent,238,80,46668,4.084168,10.062450,4.719462,1.155550
1,tail_0_1_percent,24,8,11490,0.725559,10.040165,1.099315,1.515130
2,tail_0_01_percent,3,1,2361,0.162722,10.005165,0.347273,2.134144


In [51]:
# =============================================================================
# Benchmark batched null scoring
# =============================================================================

BENCHMARK_NULL_BATCH_SIZE = 20
BATCH_BENCHMARK_CASE = "tail_0_1_percent"

batch_case_plan = (
    benchmark_case_execution_plans[
        BATCH_BENCHMARK_CASE
    ]
)

batch_active_mask = (
    batch_case_plan[
        "active_hypothesis_mask"
    ]
)

batch_signature_indices = (
    batch_case_plan[
        "active_signature_indices"
    ]
)

batch_replicate_ids = np.arange(
    BENCHMARK_NULL_BATCH_SIZE,
    dtype=np.int64,
)


def generate_pseudo_weight_batch(
    replicate_ids,
):
    weight_tensor = np.stack(
        [
            generate_coupled_pseudo_weights(
                replicate_id=int(replicate_id),
                namespace=RNG_NAMESPACES[
                    "outcome_blind_benchmark"
                ],
            )
            for replicate_id in replicate_ids
        ],
        axis=0,
    )

    # replicate × gene × program
    return weight_tensor


def score_signature_block_batch(
    signature_block,
    weight_tensor,
):
    signature_block = np.asarray(
        signature_block,
        dtype=np.float64,
    )

    weight_tensor = np.asarray(
        weight_tensor,
        dtype=np.float64,
    )

    n_replicates = (
        weight_tensor.shape[0]
    )

    # gene × (replicate × program)
    flat_weights = (
        weight_tensor
        .transpose(1, 0, 2)
        .reshape(
            weight_tensor.shape[1],
            n_replicates
            * len(PROGRAM_IDS),
        )
    )

    signature_norms = np.linalg.norm(
        signature_block,
        axis=0,
    )

    if not np.all(
        signature_norms > 0
    ):
        raise RuntimeError(
            "Batched benchmark encountered "
            "a zero-norm signature."
        )

    numerators = (
        flat_weights.T
        @ signature_block
    )

    repeated_weight_norms = np.tile(
        frozen_weight_norms,
        n_replicates,
    )

    scores = (
        numerators
        /
        (
            repeated_weight_norms[
                :,
                np.newaxis,
            ]
            * signature_norms[
                np.newaxis,
                :
            ]
        )
    )

    # signature × replicate × program
    return (
        scores.T
        .reshape(
            signature_block.shape[1],
            n_replicates,
            len(PROGRAM_IDS),
        )
    )


batch_weight_tensor = (
    generate_pseudo_weight_batch(
        batch_replicate_ids
    )
)

batched_signature_scores = np.empty(
    (
        len(batch_signature_indices),
        BENCHMARK_NULL_BATCH_SIZE,
        len(PROGRAM_IDS),
    ),
    dtype=np.float64,
)

batch_memory_samples = []
batch_stop_event = threading.Event()


def sample_batch_memory():
    while not batch_stop_event.is_set():
        batch_memory_samples.append(
            current_working_set_bytes()
        )
        batch_stop_event.wait(0.05)


batch_baseline_bytes = (
    current_working_set_bytes()
)

batch_memory_samples.append(
    batch_baseline_bytes
)

batch_memory_thread = threading.Thread(
    target=sample_batch_memory,
    daemon=True,
)

batch_start = time.perf_counter()
batch_memory_thread.start()

try:
    for local_start in range(
        0,
        len(batch_signature_indices),
        BENCHMARK_SIGNATURE_BLOCK_SIZE,
    ):
        local_stop = min(
            local_start
            + BENCHMARK_SIGNATURE_BLOCK_SIZE,
            len(batch_signature_indices),
        )

        global_indices = (
            batch_signature_indices[
                local_start:local_stop
            ]
        )

        signature_block = np.asarray(
            primary_bing_matrix[
                np.ix_(
                    query_bing_row_indices,
                    global_indices,
                )
            ],
            dtype=np.float64,
        )

        batched_signature_scores[
            local_start:local_stop,
            :,
            :
        ] = score_signature_block_batch(
            signature_block,
            batch_weight_tensor,
        )

finally:
    batch_stop_event.set()
    batch_memory_thread.join()

batch_elapsed_seconds = (
    time.perf_counter()
    - batch_start
)

batch_peak_bytes = max(
    batch_memory_samples
)


# Validate replicate 0 against the already validated scalar kernel.
replicate_zero_aggregation = (
    aggregate_active_signature_scores(
        batched_signature_scores[
            :,
            0,
            :
        ],
        batch_signature_indices,
        batch_active_mask,
    )
)

replicate_zero_scores = (
    replicate_zero_aggregation[
        "perturbagen"
    ]
    * resistance_like_multiplier_vector[
        np.newaxis,
        :
    ]
)

np.testing.assert_allclose(
    replicate_zero_scores[
        batch_active_mask
    ],
    full_reference_scores[
        batch_active_mask
    ],
    rtol=NUMERICAL_RTOL,
    atol=NUMERICAL_ATOL,
)


batch_seconds_per_replicate = (
    batch_elapsed_seconds
    / BENCHMARK_NULL_BATCH_SIZE
)

scalar_seconds_per_replicate = float(
    tail_benchmark_summary.loc[
        tail_benchmark_summary[
            "case_label"
        ].eq(BATCH_BENCHMARK_CASE),
        "elapsed_seconds",
    ].iloc[0]
)

batch_speedup = (
    scalar_seconds_per_replicate
    / batch_seconds_per_replicate
)

print(
    "BATCHED NULL SCORING BENCHMARK: PASS"
)
print(
    "Case:",
    BATCH_BENCHMARK_CASE,
)
print(
    "Batch size:",
    BENCHMARK_NULL_BATCH_SIZE,
)
print(
    "Replicates scored:",
    len(batch_replicate_ids),
)
print(
    "Signatures per replicate:",
    f"{len(batch_signature_indices):,}",
)
print(
    "Replicate 0 matches scalar reference:",
    "PASS",
)
print(
    "Total batch elapsed:",
    f"{batch_elapsed_seconds:.3f} seconds",
)
print(
    "Seconds per replicate:",
    f"{batch_seconds_per_replicate:.4f}",
)
print(
    "Scalar seconds per replicate:",
    f"{scalar_seconds_per_replicate:.4f}",
)
print(
    "Effective scoring speedup:",
    f"{batch_speedup:.2f}x",
)
print(
    "Peak working set:",
    f"{batch_peak_bytes / 1024**3:.2f} GiB",
)

BATCHED NULL SCORING BENCHMARK: PASS
Case: tail_0_1_percent
Batch size: 20
Replicates scored: 20
Signatures per replicate: 11,490
Replicate 0 matches scalar reference: PASS
Total batch elapsed: 0.830 seconds
Seconds per replicate: 0.0415
Scalar seconds per replicate: 1.0993
Effective scoring speedup: 26.50x
Peak working set: 10.02 GiB


In [52]:
# =============================================================================
# Validate exact sequential stopping under batched execution
# =============================================================================

TEST_H = 20
TEST_BMAX = 100
TEST_BATCH_SIZE = 20

# Deterministic synthetic exceedance matrix:
# rows = replicates, columns = hypotheses.
#
# Includes:
# - early stopping,
# - stopping inside a batch,
# - 20th exceedance exactly at Bmax,
# - Bmax reached with g < 20.
test_exceedances = np.zeros(
    (TEST_BMAX, 6),
    dtype=bool,
)

# H0: 20 exceedances in first 20 draws -> stop L=20
test_exceedances[:20, 0] = True

# H1: 20th exceedance at L=37 -> inside second batch
test_exceedances[
    np.r_[
        np.arange(19),
        36,
    ],
    1,
] = True

# H2: 20th exceedance at L=61 -> inside fourth batch
test_exceedances[
    np.r_[
        np.arange(19),
        60,
    ],
    2,
] = True

# H3: exactly 19 exceedances by Bmax -> Bmax branch
test_exceedances[:19, 3] = True

# H4: zero exceedances -> Bmax branch
# remains all False

# H5: 20th exceedance exactly at Bmax
test_exceedances[
    np.r_[
        np.arange(19),
        TEST_BMAX - 1,
    ],
    5,
] = True


def run_scalar_sequential_test(
    exceedances,
    h,
    bmax,
):
    n_hypotheses = exceedances.shape[1]

    g = np.zeros(
        n_hypotheses,
        dtype=np.int64,
    )

    L = np.zeros(
        n_hypotheses,
        dtype=np.int64,
    )

    active = np.ones(
        n_hypotheses,
        dtype=bool,
    )

    stop_reason = np.full(
        n_hypotheses,
        "",
        dtype=object,
    )

    for replicate_id in range(bmax):
        active_before = active.copy()

        if not active_before.any():
            break

        g[active_before] += exceedances[
            replicate_id,
            active_before,
        ]

        L[active_before] = (
            replicate_id + 1
        )

        newly_stopped = (
            active_before
            & (g >= h)
        )

        stop_reason[
            newly_stopped
        ] = "h_exceedances"

        active[
            newly_stopped
        ] = False

    stop_reason[
        active
    ] = "bmax"

    return g, L, stop_reason


def run_batched_sequential_test(
    exceedances,
    h,
    bmax,
    batch_size,
):
    n_hypotheses = exceedances.shape[1]

    g = np.zeros(
        n_hypotheses,
        dtype=np.int64,
    )

    L = np.zeros(
        n_hypotheses,
        dtype=np.int64,
    )

    active = np.ones(
        n_hypotheses,
        dtype=bool,
    )

    stop_reason = np.full(
        n_hypotheses,
        "",
        dtype=object,
    )

    for batch_start in range(
        0,
        bmax,
        batch_size,
    ):
        batch_stop = min(
            batch_start + batch_size,
            bmax,
        )

        # Scores may be computed in one batch, but stopping
        # bookkeeping remains replicate-ordered and exact.
        for replicate_id in range(
            batch_start,
            batch_stop,
        ):
            active_before = active.copy()

            if not active_before.any():
                break

            g[active_before] += exceedances[
                replicate_id,
                active_before,
            ]

            L[active_before] = (
                replicate_id + 1
            )

            newly_stopped = (
                active_before
                & (g >= h)
            )

            stop_reason[
                newly_stopped
            ] = "h_exceedances"

            active[
                newly_stopped
            ] = False

        if not active.any():
            break

    stop_reason[
        active
    ] = "bmax"

    return g, L, stop_reason


scalar_g, scalar_L, scalar_reason = (
    run_scalar_sequential_test(
        test_exceedances,
        h=TEST_H,
        bmax=TEST_BMAX,
    )
)

batched_g, batched_L, batched_reason = (
    run_batched_sequential_test(
        test_exceedances,
        h=TEST_H,
        bmax=TEST_BMAX,
        batch_size=TEST_BATCH_SIZE,
    )
)

np.testing.assert_array_equal(
    batched_g,
    scalar_g,
)

np.testing.assert_array_equal(
    batched_L,
    scalar_L,
)

np.testing.assert_array_equal(
    batched_reason,
    scalar_reason,
)

expected_L = np.array(
    [20, 37, 61, 100, 100, 100],
    dtype=np.int64,
)

expected_g = np.array(
    [20, 20, 20, 19, 0, 20],
    dtype=np.int64,
)

expected_reason = np.array(
    [
        "h_exceedances",
        "h_exceedances",
        "h_exceedances",
        "bmax",
        "bmax",
        "h_exceedances",
    ],
    dtype=object,
)

np.testing.assert_array_equal(
    batched_L,
    expected_L,
)

np.testing.assert_array_equal(
    batched_g,
    expected_g,
)

np.testing.assert_array_equal(
    batched_reason,
    expected_reason,
)

test_p = np.where(
    batched_g >= TEST_H,
    TEST_H / batched_L,
    (batched_g + 1)
    / (TEST_BMAX + 1),
)

if test_p[5] != (
    TEST_H / TEST_BMAX
):
    raise RuntimeError(
        "20th exceedance exactly at Bmax "
        "did not use p = h / L."
    )

print(
    "BATCHED SEQUENTIAL STOPPING VALIDATION: PASS"
)
print(
    "Scalar and batched g: EXACT MATCH"
)
print(
    "Scalar and batched L: EXACT MATCH"
)
print(
    "Scalar and batched stop reason: EXACT MATCH"
)
print(
    "20th exceedance at Bmax boundary: PASS"
)

display(
    pd.DataFrame(
        {
            "hypothesis": np.arange(6),
            "g": batched_g,
            "L": batched_L,
            "stop_reason": batched_reason,
            "p": test_p,
        }
    )
)

BATCHED SEQUENTIAL STOPPING VALIDATION: PASS
Scalar and batched g: EXACT MATCH
Scalar and batched L: EXACT MATCH
Scalar and batched stop reason: EXACT MATCH
20th exceedance at Bmax boundary: PASS


,hypothesis,g,L,stop_reason,p
0,0,20,20,h_exceedances,1.000000
1,1,20,37,h_exceedances,0.540541
2,2,20,61,h_exceedances,0.327869
3,3,19,100,bmax,0.198020
4,4,0,100,bmax,0.009901
5,5,20,100,h_exceedances,0.200000


In [53]:
# =============================================================================
# Freeze statistically adequate final Bmax
# =============================================================================

PRIMARY_MULTIPLICITY_FAMILY_SIZE = 23742
PRIMARY_BY_Q = 0.05
SEQUENTIAL_EXCEEDANCE_TARGET = 20

by_harmonic_factor = np.sum(
    1.0
    / np.arange(
        1,
        PRIMARY_MULTIPLICITY_FAMILY_SIZE + 1,
        dtype=np.float64,
    )
)

by_rank1_threshold = (
    PRIMARY_BY_Q
    / (
        PRIMARY_MULTIPLICITY_FAMILY_SIZE
        * by_harmonic_factor
    )
)

# Smallest integer Bmax for which the minimum attainable
# terminal Monte Carlo p-value can reach the global BY
# rank-1 critical value.
FINAL_BMAX = int(
    np.ceil(
        1.0 / by_rank1_threshold
        - 1.0
    )
)

minimum_attainable_mc_p = (
    1.0
    / (FINAL_BMAX + 1)
)

previous_bmax_floor = (
    1.0
    / FINAL_BMAX
)

if not (
    minimum_attainable_mc_p
    <= by_rank1_threshold
):
    raise RuntimeError(
        "Selected Bmax does not reach the "
        "global BY rank-1 threshold."
    )

if not (
    previous_bmax_floor
    > by_rank1_threshold
):
    raise RuntimeError(
        "Selected Bmax is not the minimal integer "
        "satisfying rank-1 BY reachability."
    )


BMAX_AMENDMENT_PATH = (
    Paths.perturbational_hypotheses
    / "700_bmax_statistical_amendment.json"
)

bmax_statistical_amendment = {
    "status": "FROZEN_BEFORE_OBSERVED_CONNECTIVITY",
    "scientific_results_inspected": False,
    "benchmark_namespace": RNG_NAMESPACES[
        "outcome_blind_benchmark"
    ],
    "primary_multiplicity": {
        "method": "Benjamini-Yekutieli",
        "family_size": (
            PRIMARY_MULTIPLICITY_FAMILY_SIZE
        ),
        "q": PRIMARY_BY_Q,
        "harmonic_factor": float(
            by_harmonic_factor
        ),
        "rank1_critical_p": float(
            by_rank1_threshold
        ),
    },
    "sequential_monte_carlo": {
        "exceedance_target_h": (
            SEQUENTIAL_EXCEEDANCE_TARGET
        ),
        "final_bmax": FINAL_BMAX,
        "minimum_attainable_mc_p": float(
            minimum_attainable_mc_p
        ),
    },
    "selection_rule": (
        "Select the minimal integer Bmax such that "
        "the minimum attainable terminal Monte Carlo "
        "p-value 1/(Bmax+1) is no greater than the "
        "global Benjamini-Yekutieli rank-1 critical "
        "threshold q/(m*H_m)."
    ),
    "operational_amendment": {
        "original_24h_runtime_limit_role": (
            "operational planning target only"
        ),
        "original_candidate_ladder": [
            500000,
            250000,
            100000,
        ],
        "original_candidate_ladder_status": (
            "superseded for final Bmax selection"
        ),
        "reason": (
            "The original runtime ceiling and candidate "
            "ladder were operational constraints without "
            "inferential justification. They are superseded "
            "before any observed program-perturbagen "
            "connectivity is inspected. The null, h=20, "
            "RNG, aggregation hierarchy, multiplicity "
            "procedure, and scientific estimands remain "
            "unchanged."
        ),
    },
    "implementation": {
        "batched_null_scoring_allowed": True,
        "batched_stopping_exactly_validated": True,
        "checkpoint_interval_replicates": (
            benchmark_resource_envelope[
                "checkpoint_restart_policy"
            ][
                "checkpoint_interval_replicates"
            ]
        ),
    },
}


if BMAX_AMENDMENT_PATH.exists():
    with BMAX_AMENDMENT_PATH.open(
        "r",
        encoding="utf-8",
    ) as handle:
        persisted_bmax_amendment = json.load(
            handle
        )

    if (
        persisted_bmax_amendment
        != bmax_statistical_amendment
    ):
        raise RuntimeError(
            "A different Bmax amendment already exists. "
            "Refusing to overwrite a prospectively "
            "frozen Bmax decision."
        )

    bmax_amendment_action = "REUSED_IDENTICAL"

else:
    with BMAX_AMENDMENT_PATH.open(
        "x",
        encoding="utf-8",
    ) as handle:
        json.dump(
            bmax_statistical_amendment,
            handle,
            indent=2,
            sort_keys=True,
        )
        handle.write("\n")

    bmax_amendment_action = "CREATED"


print(
    "FINAL BMAX STATISTICAL FREEZE: PASS"
)
print(
    "Amendment action:",
    bmax_amendment_action,
)
print(
    "Global BY family:",
    f"{PRIMARY_MULTIPLICITY_FAMILY_SIZE:,}",
)
print(
    "BY q:",
    PRIMARY_BY_Q,
)
print(
    "Harmonic factor:",
    f"{by_harmonic_factor:.12f}",
)
print(
    "BY rank-1 critical p:",
    f"{by_rank1_threshold:.12e}",
)
print(
    "Final Bmax:",
    f"{FINAL_BMAX:,}",
)
print(
    "Minimum attainable MC p:",
    f"{minimum_attainable_mc_p:.12e}",
)
print(
    "Original 24-hour limit:",
    "RECLASSIFIED AS OPERATIONAL ONLY",
)
print(
    "Observed scientific connectivity inspected:",
    "NO",
)

FINAL BMAX STATISTICAL FREEZE: PASS
Amendment action: REUSED_IDENTICAL
Global BY family: 23,742
BY q: 0.05
Harmonic factor: 10.652237635093
BY rank-1 critical p: 1.977023617254e-07
Final Bmax: 5,058,108
Minimum attainable MC p: 1.977023429112e-07
Original 24-hour limit: RECLASSIFIED AS OPERATIONAL ONLY
Observed scientific connectivity inspected: NO


In [60]:
# =============================================================================
# Define Notebook 700 stable output paths
# =============================================================================

PHASE7_700_OUTPUT_DIR = (
    Paths.perturbational_hypotheses
)

PHASE7_700_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

program_query_definitions_path = (
    PHASE7_700_OUTPUT_DIR
    / "700_program_query_definitions.csv"
)

null_stratification_manifest_path = (
    PHASE7_700_OUTPUT_DIR
    / "700_null_stratification_manifest.csv"
)

benchmark_report_path = (
    PHASE7_700_OUTPUT_DIR
    / "700_benchmark_report.json"
)

analysis_metadata_path = (
    PHASE7_700_OUTPUT_DIR
    / "700_analysis_metadata.json"
)

print(
    "Notebook 700 output directory:",
    project_relative_path(
        PHASE7_700_OUTPUT_DIR
    ),
)


Notebook 700 output directory: data/processed/perturbational_hypotheses


In [61]:
# =============================================================================
# Prepare stable Notebook 700 tabular handoffs
# =============================================================================

program_query_definitions_final = (
    program_query_definitions
    .sort_values(
        [
            "program_id",
            "gene_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

null_stratification_manifest_final = (
    null_stratification_manifest
    .sort_values(
        [
            "stratum_id",
            "perturbational_mad",
            "gene_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

print(
    "Program-query rows:",
    f"{len(program_query_definitions_final):,}",
)
print(
    "Null-manifest genes:",
    f"{len(null_stratification_manifest_final):,}",
)


Program-query rows: 4,485
Null-manifest genes: 10,174


In [62]:
# =============================================================================
# Assemble Notebook 700 benchmark report
# =============================================================================

benchmark_report = {
    "schema_version": 1,
    "notebook": "700_program_signature_construction",
    "benchmark_role": (
        "outcome_blind_phase7_computational_calibration"
    ),
    "observed_program_perturbagen_connectivity_inspected": False,
    "rng": {
        "entropy": int(RNG_ENTROPY),
        "namespace": int(
            RNG_NAMESPACES[
                "outcome_blind_benchmark"
            ]
        ),
        "generator": "PCG64DXSM",
        "permutation_mapping": (
            "NumPy Generator.permutation within frozen "
            "feature-space × MAD-decile strata using "
            "independent namespace × replicate × stratum seeds"
        ),
    },
    "resource_envelope": (
        persisted_resource_envelope
    ),
    "measured_cases": {
        "maximum_100_percent": {
            "active_hypotheses": 23742,
            "signatures_scored": int(
                max_active_benchmark_result[
                    "active_signature_count"
                ]
            ),
            "elapsed_seconds": float(
                max_benchmark_elapsed_seconds
            ),
            "peak_working_set_gib": float(
                max_peak_working_set_bytes
                / 1024**3
            ),
        },
        "reduced_50_percent": {
            "active_hypotheses": 11871,
            "signatures_scored": int(
                reduced_50_benchmark_result[
                    "active_signature_count"
                ]
            ),
            "elapsed_seconds": float(
                reduced_50_benchmark_elapsed_seconds
            ),
            "peak_working_set_gib": float(
                reduced_50_peak_working_set_bytes
                / 1024**3
            ),
        },
        "reduced_10_percent_ceiling": {
            "active_hypotheses": 2375,
            "signatures_scored": int(
                reduced_benchmark_result[
                    "active_signature_count"
                ]
            ),
            "elapsed_seconds": float(
                reduced_benchmark_elapsed_seconds
            ),
            "peak_working_set_gib": float(
                reduced_peak_working_set_bytes
                / 1024**3
            ),
        },
        "sparse_tail": json.loads(
            tail_benchmark_summary.to_json(
                orient="records"
            )
        ),
    },
    "batched_null_scoring": {
        "selected_for_phase7_execution": True,
        "benchmark_case": (
            BATCH_BENCHMARK_CASE
        ),
        "batch_size": int(
            BENCHMARK_NULL_BATCH_SIZE
        ),
        "replicates_measured": int(
            len(batch_replicate_ids)
        ),
        "total_elapsed_seconds": float(
            batch_elapsed_seconds
        ),
        "seconds_per_replicate": float(
            batch_seconds_per_replicate
        ),
        "scalar_seconds_per_replicate": float(
            scalar_seconds_per_replicate
        ),
        "effective_speedup": float(
            batch_speedup
        ),
        "peak_working_set_gib": float(
            batch_peak_bytes
            / 1024**3
        ),
        "replicate_zero_matches_scalar_reference": True,
        "exact_sequential_g_validated": True,
        "exact_sequential_L_validated": True,
        "exact_stop_reason_validated": True,
        "bmax_boundary_validated": True,
    },
    "checkpoint_restart": {
        "validated": True,
        "checkpoint_boundary": int(
            CHECKPOINT_TEST_NEXT_REPLICATE_ID
        ),
        "write_seconds": float(
            checkpoint_write_seconds
        ),
        "read_seconds": float(
            checkpoint_read_seconds
        ),
        "checkpoint_size_bytes": int(
            checkpoint_size_bytes
        ),
        "rng_reconstruction_after_restart": True,
    },
    "legacy_operational_projection": {
        "status": (
            "superseded_for_final_bmax_selection"
        ),
        "candidate_results": json.loads(
            bmax_projection_summary.to_json(
                orient="records"
            )
        ),
    },
    "final_bmax_decision": (
        bmax_statistical_amendment
    ),
}


In [63]:
# =============================================================================
# Assemble Notebook 700 analysis metadata
# =============================================================================

analysis_metadata = {
    "schema_version": 1,
    "notebook": (
        "700_program_signature_construction"
    ),
    "analysis_role": (
        "outcome_blind_phase7_query_null_and_"
        "simulation_budget_preparation"
    ),
    "specification_date": "2026-09-29",
    "phase7_observed_connectivity_started": False,
    "program_ids": list(
        PROGRAM_IDS
    ),
    "primary_query": {
        "representation": (
            "continuous signed weighted BING"
        ),
        "shared_support_genes": int(
            len(query_gene_ids)
        ),
        "weight_norms": {
            program_id: float(norm)
            for program_id, norm in zip(
                PROGRAM_IDS,
                frozen_weight_norms,
            )
        },
        "resistance_like_multipliers": {
            program_id: int(multiplier)
            for program_id, multiplier in zip(
                PROGRAM_IDS,
                resistance_like_multiplier_vector,
            )
        },
    },
    "null_stratification": {
        "total_bing_genes": int(
            len(
                null_stratification_manifest_final
            )
        ),
        "strata": int(
            null_stratification_manifest_final[
                "stratum_id"
            ].nunique()
        ),
        "query_genes": int(
            null_stratification_manifest_final[
                "primary_query_member"
            ].sum()
        ),
        "degenerate_one_gene_strata": int(
            manifest_program_summary[
                "degenerate_one_gene_stratum"
            ].sum()
        ),
        "mad_definition": (
            "exact unscaled median absolute deviation "
            "across all 362036 frozen primary signatures"
        ),
    },
    "numerical_policy": {
        "source_gctx_dtype": "float32",
        "score_and_aggregation_dtype": "float64",
        "rtol": float(NUMERICAL_RTOL),
        "atol": float(NUMERICAL_ATOL),
    },
    "rng": {
        "entropy": int(RNG_ENTROPY),
        "namespaces": {
            key: int(value)
            for key, value in (
                RNG_NAMESPACES.items()
            )
        },
        "bit_generator": "PCG64DXSM",
        "seed_coordinates": (
            "namespace × replicate_id × stratum_id"
        ),
        "within_stratum_permutation_mapping": (
            "NumPy Generator.permutation"
        ),
        "draws_with_replacement_across_replicates": True,
    },
    "sequential_monte_carlo": {
        "exceedance_target_h": int(
            SEQUENTIAL_EXCEEDANCE_TARGET
        ),
        "final_bmax": int(
            FINAL_BMAX
        ),
        "minimum_attainable_mc_p": float(
            minimum_attainable_mc_p
        ),
        "batch_size": int(
            BENCHMARK_NULL_BATCH_SIZE
        ),
        "checkpoint_interval_replicates": int(
            benchmark_resource_envelope[
                "checkpoint_restart_policy"
            ][
                "checkpoint_interval_replicates"
            ]
        ),
        "selection_rule": (
            bmax_statistical_amendment[
                "selection_rule"
            ]
        ),
        "original_24h_runtime_limit_role": (
            "operational planning target only"
        ),
        "historical_candidate_ladder_status": (
            "superseded for final Bmax selection"
        ),
    },
    "multiplicity": {
        "primary_method": (
            "Benjamini-Yekutieli"
        ),
        "q": float(
            PRIMARY_BY_Q
        ),
        "family_size": int(
            PRIMARY_MULTIPLICITY_FAMILY_SIZE
        ),
        "harmonic_factor": float(
            by_harmonic_factor
        ),
        "rank1_critical_p": float(
            by_rank1_threshold
        ),
    },
    "upstream_artifacts": list(
        INPUT_ARTIFACT_IDS
    ),
    "stable_outputs": [
        "phase7.700.program_query_definitions",
        "phase7.700.null_stratification_manifest",
        "phase7.700.benchmark_report",
        "phase7.700.analysis_metadata",
    ],
    "interpretation_limitations": [
        (
            "Notebook 700 is outcome-blind and does not "
            "contain observed program-perturbagen connectivity."
        ),
        (
            "The conditional randomization procedure "
            "supports computational calibration under its "
            "exchangeability assumptions; it does not "
            "establish a universal biological null."
        ),
        (
            "Bmax controls Monte Carlo resolution and does "
            "not itself determine biological or therapeutic "
            "support."
        ),
        (
            "Association and perturbational reversal remain "
            "computational hypotheses rather than causal or "
            "clinical validation."
        ),
    ],
}


In [64]:
# =============================================================================
# Persist stable Notebook 700 handoffs
# =============================================================================

program_query_definitions_final.to_csv(
    program_query_definitions_path,
    index=False,
)

null_stratification_manifest_final.to_csv(
    null_stratification_manifest_path,
    index=False,
)

with benchmark_report_path.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        benchmark_report,
        handle,
        indent=2,
        sort_keys=True,
    )
    handle.write("\n")

with analysis_metadata_path.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        analysis_metadata,
        handle,
        indent=2,
        sort_keys=True,
    )
    handle.write("\n")

print(
    "NOTEBOOK 700 STABLE HANDOFFS: PERSISTED"
)


NOTEBOOK 700 STABLE HANDOFFS: PERSISTED


In [65]:
# =============================================================================
# Verify Notebook 700 artifact persistence
# =============================================================================

stable_output_paths = {
    "phase7.700.program_query_definitions": (
        program_query_definitions_path
    ),
    "phase7.700.null_stratification_manifest": (
        null_stratification_manifest_path
    ),
    "phase7.700.benchmark_report": (
        benchmark_report_path
    ),
    "phase7.700.analysis_metadata": (
        analysis_metadata_path
    ),
}

persistence_checks = {
    artifact_id: path.is_file()
    for artifact_id, path in (
        stable_output_paths.items()
    )
}

display(
    pd.Series(
        persistence_checks,
        name="exists",
    )
)

if not all(
    persistence_checks.values()
):
    missing = [
        artifact_id
        for artifact_id, exists in (
            persistence_checks.items()
        )
        if not exists
    ]

    raise RuntimeError(
        "Notebook 700 stable output missing: "
        + ", ".join(missing)
    )

for artifact_id, path in (
    stable_output_paths.items()
):
    print(
        artifact_id,
        "->",
        project_relative_path(path),
    )


phase7.700.program_query_definitions       True
phase7.700.null_stratification_manifest    True
phase7.700.benchmark_report                True
phase7.700.analysis_metadata               True
Name: exists, dtype: bool

phase7.700.program_query_definitions -> data/processed/perturbational_hypotheses/700_program_query_definitions.csv
phase7.700.null_stratification_manifest -> data/processed/perturbational_hypotheses/700_null_stratification_manifest.csv
phase7.700.benchmark_report -> data/processed/perturbational_hypotheses/700_benchmark_report.json
phase7.700.analysis_metadata -> data/processed/perturbational_hypotheses/700_analysis_metadata.json


In [66]:
# =============================================================================
# Define Notebook 700 stable artifact outputs
# =============================================================================

artifact_outputs = {
    "phase7.700.program_query_definitions": {
        "path": program_query_definitions_path,
        "expected": program_query_definitions_final,
        "artifact_role": "handoff",
    },
    "phase7.700.null_stratification_manifest": {
        "path": null_stratification_manifest_path,
        "expected": null_stratification_manifest_final,
        "artifact_role": "handoff",
    },
    "phase7.700.benchmark_report": {
        "path": benchmark_report_path,
        "expected": benchmark_report,
        "artifact_role": "metadata",
    },
    "phase7.700.analysis_metadata": {
        "path": analysis_metadata_path,
        "expected": analysis_metadata,
        "artifact_role": "metadata",
    },
}

print(
    "Stable Notebook 700 outputs:",
    len(artifact_outputs),
)


Stable Notebook 700 outputs: 4


In [67]:
# =============================================================================
# Build Notebook 700 artifact identities
# =============================================================================

from pancancer_epigenetics.utils.file_checks import (
    calculate_sha256,
)

artifact_identity_rows = []

for artifact_id, spec in artifact_outputs.items():
    artifact_path = spec["path"]
    expected = spec["expected"]

    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Notebook 700 output is missing: {artifact_path}"
        )

    if artifact_path.suffix == ".csv":
        persisted = pd.read_csv(
            artifact_path,
            low_memory=False,
        )

        if persisted.shape != expected.shape:
            raise RuntimeError(
                f"Shape mismatch for {artifact_id}: "
                f"{persisted.shape} != {expected.shape}"
            )

        if persisted.columns.tolist() != expected.columns.tolist():
            raise RuntimeError(
                f"Column mismatch for {artifact_id}."
            )

        shape = [
            int(persisted.shape[0]),
            int(persisted.shape[1]),
        ]

    elif artifact_path.suffix == ".json":
        with artifact_path.open(
            "r",
            encoding="utf-8",
        ) as handle:
            persisted = json.load(handle)

        if persisted != expected:
            raise RuntimeError(
                f"JSON content mismatch for {artifact_id}."
            )

        shape = None

    else:
        raise ValueError(
            f"Unsupported Notebook 700 output format: "
            f"{artifact_path.suffix}"
        )

    artifact_identity_rows.append(
        {
            "artifact_id": artifact_id,
            "path": project_relative_path(
                artifact_path
            ),
            "artifact_role": spec[
                "artifact_role"
            ],
            "shape": shape,
            "size_bytes": int(
                artifact_path.stat().st_size
            ),
            "sha256": calculate_sha256(
                artifact_path
            ),
        }
    )

artifact_identities = pd.DataFrame(
    artifact_identity_rows
)

artifact_identity_index = (
    artifact_identities
    .set_index("artifact_id")
)

display(artifact_identities)


,artifact_id,path,artifact_role,shape,size_bytes,sha256
0,phase7.700.program_query_definitions,data/processed/perturbational_hypotheses/700_p...,handoff,"[4485, 16]",714179,9ac087b48e0a6c62597671c1cccff5f72e5c21c13cb3b7...
1,phase7.700.null_stratification_manifest,data/processed/perturbational_hypotheses/700_n...,handoff,"[10174, 11]",882767,355fb778cd6b24f7ce012d5ae59480fcd2b98b51d46ae7...
2,phase7.700.benchmark_report,data/processed/perturbational_hypotheses/700_b...,metadata,None,13077,be7930d5cc2f116babab12d60abf2d467655a05c0707c6...
3,phase7.700.analysis_metadata,data/processed/perturbational_hypotheses/700_a...,metadata,None,3616,b2e1b59d56f495f97370a2dd769448cdf9bf525846b079...


In [68]:
# =============================================================================
# Define Notebook 700 artifact provenance
# =============================================================================

NOTEBOOK_700_PATH = (
    "notebooks/phase7_perturbational_hypotheses/"
    "700_program_signature_construction.ipynb"
)

artifact_inputs = {
    "phase7.700.program_query_definitions": [
        "phase4.401.consensus_transcriptomic_program_catalog",
        "phase4.401.consensus_transcriptomic_gene_weights",
        "phase3.311.program_robustness_summary",
        "phase1.110.bing_gene_handoff",
    ],
    "phase7.700.null_stratification_manifest": [
        "phase7.700.program_query_definitions",
        "phase1.110.primary_signature_handoff",
        "phase1.110.raw_input_inventory",
    ],
    "phase7.700.benchmark_report": [
        "phase7.700.program_query_definitions",
        "phase7.700.null_stratification_manifest",
        "phase1.110.primary_signature_handoff",
        "phase1.110.primary_perturbagen_coverage",
    ],
    "phase7.700.analysis_metadata": [
        "phase7.700.program_query_definitions",
        "phase7.700.null_stratification_manifest",
        "phase7.700.benchmark_report",
    ],
}

print(
    "Artifact provenance definitions:",
    len(artifact_inputs),
)


Artifact provenance definitions: 4


In [69]:
# =============================================================================
# Build Notebook 700 artifact registry entries
# =============================================================================

artifact_identity_index = (
    artifact_identities
    .set_index("artifact_id")
)

registry_entries = {}

for artifact_id in artifact_outputs:
    identity = artifact_identity_index.loc[
        artifact_id
    ]

    registry_entries[artifact_id] = {
        "path": identity["path"],
        "phase": 7,
        "status": "frozen",
        "artifact_role": identity[
            "artifact_role"
        ],
        "producer": {
            "type": "notebook",
            "path": NOTEBOOK_700_PATH,
        },
        "shape": (
            identity["shape"]
            if isinstance(
                identity["shape"],
                list,
            )
            else None
        ),
        "size_bytes": int(
            identity["size_bytes"]
        ),
        "sha256": identity["sha256"],
        "inputs": [
            {
                "type": "artifact",
                "artifact_id": input_artifact_id,
            }
            for input_artifact_id in (
                artifact_inputs[
                    artifact_id
                ]
            )
        ],
    }


In [70]:
# =============================================================================
# Register frozen Notebook 700 artifacts
# =============================================================================

from pancancer_epigenetics.utils.artifact_registry import (
    validate_artifact_registry,
)

updated_registry = load_artifact_registry()

for artifact_id, entry in (
    registry_entries.items()
):
    existing = updated_registry[
        "artifacts"
    ].get(artifact_id)

    if (
        existing is not None
        and existing != entry
    ):
        raise RuntimeError(
            "A non-identical frozen Notebook 700 "
            f"registry entry already exists: {artifact_id}"
        )

    updated_registry[
        "artifacts"
    ][artifact_id] = entry

validate_artifact_registry(
    updated_registry
)

Paths.artifact_registry.write_text(
    json.dumps(
        updated_registry,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

print(
    "NOTEBOOK 700 ARTIFACT REGISTRATION: PASS"
)
print(
    "Registered artifacts:",
    len(registry_entries),
)
display(artifact_identities)


NOTEBOOK 700 ARTIFACT REGISTRATION: PASS
Registered artifacts: 4


,artifact_id,path,artifact_role,shape,size_bytes,sha256
0,phase7.700.program_query_definitions,data/processed/perturbational_hypotheses/700_p...,handoff,"[4485, 16]",714179,9ac087b48e0a6c62597671c1cccff5f72e5c21c13cb3b7...
1,phase7.700.null_stratification_manifest,data/processed/perturbational_hypotheses/700_n...,handoff,"[10174, 11]",882767,355fb778cd6b24f7ce012d5ae59480fcd2b98b51d46ae7...
2,phase7.700.benchmark_report,data/processed/perturbational_hypotheses/700_b...,metadata,None,13077,be7930d5cc2f116babab12d60abf2d467655a05c0707c6...
3,phase7.700.analysis_metadata,data/processed/perturbational_hypotheses/700_a...,metadata,None,3616,b2e1b59d56f495f97370a2dd769448cdf9bf525846b079...


In [71]:
# =============================================================================
# Validate final Phase 7 contract handoff
# =============================================================================

contract_path = (
    Paths.root
    / "docs"
    / "contracts"
    / "phase7"
    / "PHASE7_ANALYSIS_CONTRACT.md"
)

contract_text = contract_path.read_text(
    encoding="utf-8"
)

normalized_contract_text = " ".join(
    contract_text.lower().split()
)

contract_handoff_checks = {
    "final_bmax_recorded": (
        "bmax = 5,058,108"
        in normalized_contract_text
        or "bmax = 5058108"
        in normalized_contract_text
    ),
    "runtime_selection_rule_superseded": (
        "superseded"
        in normalized_contract_text
        and "runtime-based selection rule"
        in normalized_contract_text
    ),
    "runtime_ceiling_is_noninferential": (
        "24-hour"
        in normalized_contract_text
        and "operational planning"
        in normalized_contract_text
        and "inferential eligibility gate"
        in normalized_contract_text
    ),
    "by_rank1_resolution_recorded": (
        "rank-1"
        in normalized_contract_text
        and str(
            PRIMARY_MULTIPLICITY_FAMILY_SIZE
        )
        in normalized_contract_text.replace(
            ",",
            "",
        )
    ),
}

display(
    pd.Series(
        contract_handoff_checks,
        name="PASS",
    )
)

if not all(
    contract_handoff_checks.values()
):
    failed = [
        key
        for key, passed in (
            contract_handoff_checks.items()
        )
        if not passed
    ]

    raise RuntimeError(
        "Phase 7 contract handoff validation failed: "
        + ", ".join(failed)
    )


final_bmax_recorded                  True
runtime_selection_rule_superseded    True
runtime_ceiling_is_noninferential    True
by_rank1_resolution_recorded         True
Name: PASS, dtype: bool

In [72]:
# =============================================================================
# Validate Notebook 700 closure invariants
# =============================================================================

final_registry = load_artifact_registry()

registered_700 = {
    artifact_id: artifact
    for artifact_id, artifact in (
        final_registry["artifacts"].items()
    )
    if artifact_id.startswith(
        "phase7.700."
    )
}

required_query_columns = {
    "program_id",
    "source_cell_line_program",
    "phase4_orientation_multiplier",
    "phase3_lineage_adjusted_rho",
    "phase3_robustness_category",
    "resistance_like_orientation_multiplier",
    "gene_id",
    "frozen_weight",
    "cmap_style_arm",
    "cmap_style_member",
}

closure_qa_results = {
    "four_artifacts_registered": (
        set(registered_700)
        == set(stable_output_paths)
    ),
    "query_rows_4485": (
        len(
            program_query_definitions_final
        )
        == 4485
    ),
    "query_required_columns_present": (
        required_query_columns.issubset(
            program_query_definitions_final.columns
        )
    ),
    "query_1495_per_program": (
        program_query_definitions_final
        .groupby("program_id")
        .size()
        .eq(1495)
        .all()
    ),
    "null_manifest_10174_genes": (
        len(
            null_stratification_manifest_final
        )
        == 10174
    ),
    "null_manifest_20_strata": (
        null_stratification_manifest_final[
            "stratum_id"
        ].nunique()
        == 20
    ),
    "final_bmax_exact": (
        FINAL_BMAX == 5058108
    ),
    "mc_floor_reaches_by_rank1": (
        minimum_attainable_mc_p
        <= by_rank1_threshold
    ),
    "benchmark_outcome_blind": (
        benchmark_report[
            "observed_program_perturbagen_connectivity_inspected"
        ]
        is False
    ),
    "analysis_metadata_outcome_blind": (
        analysis_metadata[
            "phase7_observed_connectivity_started"
        ]
        is False
    ),
    "batched_stopping_validated": (
        benchmark_report[
            "batched_null_scoring"
        ][
            "exact_sequential_g_validated"
        ]
        and benchmark_report[
            "batched_null_scoring"
        ][
            "exact_sequential_L_validated"
        ]
        and benchmark_report[
            "batched_null_scoring"
        ][
            "exact_stop_reason_validated"
        ]
        and benchmark_report[
            "batched_null_scoring"
        ][
            "bmax_boundary_validated"
        ]
    ),
    "contract_handoff_valid": (
        all(
            contract_handoff_checks.values()
        )
    ),
}

for artifact_id, path in (
    stable_output_paths.items()
):
    registered_path = (
        Paths.root
        / registered_700[
            artifact_id
        ]["path"]
    )

    closure_qa_results[
        f"{artifact_id}:path"
    ] = (
        registered_path.resolve()
        == path.resolve()
    )

    closure_qa_results[
        f"{artifact_id}:sha256"
    ] = (
        calculate_sha256(path)
        == registered_700[
            artifact_id
        ]["sha256"]
    )

validate_artifact_registry(
    final_registry
)


{'schema_version': 1,
 'artifacts': {'phase2.203.final_rna_counts': {'path': 'data/interim/expression/tcga_primary_tumor_rnaseq_final_case_level_unstranded_raw_counts.npy',
   'phase': 2,
   'status': 'frozen',
   'artifact_role': 'data',
   'producer': {'type': 'notebook',
    'path': 'notebooks/phase2_tumor_discovery_layer/203_tcga_multiomic_integration.ipynb'},
   'shape': [60660, 9965],
   'size_bytes': 2417907728,
   'sha256': 'e23efdd601e370beeb30a58a7e592c6f03ff59c4e308912d141588d1e70a51b3',
   'inputs': [{'type': 'artifact',
     'artifact_id': 'phase2.201.candidate_raw_rna_counts'},
    {'type': 'artifact', 'artifact_id': 'phase2.201.rna_file_qc_inventory'},
    {'type': 'artifact',
     'artifact_id': 'phase2.202.qc_annotated_candidate_pair_inventory'}]},
  'phase2.203.final_rna_feature_index': {'path': 'data/interim/expression/tcga_primary_tumor_rnaseq_final_case_level_gene_feature_index.csv',
   'phase': 2,
   'status': 'frozen',
   'artifact_role': 'metadata',
   'producer

In [73]:
# =============================================================================
# Final Notebook 700 local QA
# =============================================================================

qa_series = pd.Series(
    closure_qa_results,
    name="PASS",
)

display(qa_series)

if not qa_series.all():
    failed = qa_series.loc[
        ~qa_series
    ].index.tolist()

    raise RuntimeError(
        "Notebook 700 closure QA failed: "
        + ", ".join(failed)
    )

print(
    "NOTEBOOK 700 FINAL QA: PASS"
)
print(
    "Stable outputs registered: 4/4"
)
print(
    "Final Bmax:",
    f"{FINAL_BMAX:,}",
)
print(
    "Observed connectivity inspected:",
    "NO",
)


four_artifacts_registered                         True
query_rows_4485                                   True
query_required_columns_present                    True
query_1495_per_program                            True
null_manifest_10174_genes                         True
null_manifest_20_strata                           True
final_bmax_exact                                  True
mc_floor_reaches_by_rank1                         True
benchmark_outcome_blind                           True
analysis_metadata_outcome_blind                   True
batched_stopping_validated                        True
contract_handoff_valid                            True
phase7.700.program_query_definitions:path         True
phase7.700.program_query_definitions:sha256       True
phase7.700.null_stratification_manifest:path      True
phase7.700.null_stratification_manifest:sha256    True
phase7.700.benchmark_report:path                  True
phase7.700.benchmark_report:sha256                True
phase7.700

NOTEBOOK 700 FINAL QA: PASS
Stable outputs registered: 4/4
Final Bmax: 5,058,108
Observed connectivity inspected: NO


## Conclusion

Notebook 700 froze the four outcome-blind Phase 7 preparation artifacts required for downstream perturbational connectivity analysis: continuous BING program-query definitions, the exact feature-space × MAD-decile null-stratification manifest, the computational benchmark report, and the analysis-metadata record.

The primary sequential Monte Carlo ceiling is prospectively fixed at `Bmax = 5,058,108` under the final statistical-resolution criterion. The original 24-hour runtime ceiling is retained only as operational benchmark context and does not act as an inferential eligibility gate.

The batched implementation preserves the validated sequential stopping logic, deterministic RNG mapping, and checkpoint/restart contract. Notebook 700 contains no observed program–perturbagen connectivity, perturbagen ranking, conditional-randomization p-value, multiplicity result, compound prioritization, or mechanism-of-action result.

The resulting frozen handoffs define the computational inputs and provenance boundary for Notebook 701.
